In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23A - CELL 1
# 初始化 malignant-state analysis
# ============================================================

from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import sparse
import json
import sys
import platform
import datetime
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# 路径
# ============================================================

STEP22_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

STEP23_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = STEP23_ROOT / "Data"
RESULTS_DIR = STEP23_ROOT / "Results"
FIGURES_DIR = STEP23_ROOT / "Figures"
LOGS_DIR = STEP23_ROOT / "Logs"
NOTEBOOK_DIR = STEP23_ROOT / "Notebook"

for folder in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    NOTEBOOK_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 输入
# ============================================================

RAW_INPUT = (
    STEP22_ROOT /
    "Data" /
    "GSE149614_STEP22_EPI_RAW_COUNTS.h5ad"
)

LABEL_INPUT = (
    STEP22_ROOT /
    "Data" /
    "GSE149614_STEP22_FINAL_MALIGNANCY_ANNOTATED.h5ad"
)


assert RAW_INPUT.exists(), (
    f"缺少RAW输入：\n{RAW_INPUT}"
)

assert LABEL_INPUT.exists(), (
    f"缺少Step22最终标签：\n{LABEL_INPUT}"
)


print("=" * 100)
print("STEP23A MALIGNANT-ONLY DATASET")
print("=" * 100)

print("Python:", sys.version.split()[0])
print("Scanpy:", sc.__version__)
print("Platform:", platform.platform())

print("\nRAW:")
print(RAW_INPUT)

print("\nLABEL:")
print(LABEL_INPUT)

In [ ]:
# ============================================================
# STEP23A - CELL 2
# 读取raw counts和Step22最终标签
# ============================================================

print("Loading raw counts...")

adata_raw = sc.read_h5ad(
    RAW_INPUT
)

print("Loading Step22 final labels...")

adata_label = sc.read_h5ad(
    LABEL_INPUT
)


print("\n" + "=" * 100)
print("INPUT AUDIT")
print("=" * 100)

print(
    "RAW shape:",
    adata_raw.shape
)

print(
    "Label shape:",
    adata_label.shape
)

print(
    "RAW unique cells:",
    adata_raw.obs_names.nunique()
)

print(
    "Label unique cells:",
    adata_label.obs_names.nunique()
)

print(
    "RAW unique genes:",
    adata_raw.var_names.nunique()
)

print(
    "Label unique genes:",
    adata_label.var_names.nunique()
)


assert adata_raw.obs_names.is_unique
assert adata_label.obs_names.is_unique
assert adata_raw.var_names.is_unique
assert adata_label.var_names.is_unique

In [ ]:
# ============================================================
# STEP23A - CELL 3
# Cell ID匹配审计
# ============================================================

raw_cells = set(
    adata_raw.obs_names
)

label_cells = set(
    adata_label.obs_names
)


common_cells = (
    raw_cells
    &
    label_cells
)

raw_only = (
    raw_cells
    -
    label_cells
)

label_only = (
    label_cells
    -
    raw_cells
)


cell_match_audit = pd.DataFrame({

    "Metric": [
        "RAW_cells",
        "LABEL_cells",
        "Common_cells",
        "RAW_only",
        "LABEL_only"
    ],

    "N": [
        len(raw_cells),
        len(label_cells),
        len(common_cells),
        len(raw_only),
        len(label_only)
    ]
})


display(
    cell_match_audit
)


cell_match_audit.to_csv(
    RESULTS_DIR /
    "Step23A_Cell_ID_Matching_Audit.csv",
    index=False,
    encoding="utf-8-sig"
)


assert len(label_only) == 0, (
    "Step22最终标签中存在RAW对象没有的细胞，"
    "请先停止检查。"
)


print("\nCell ID matching = PASS")

In [ ]:
# ============================================================
# STEP23A - CELL 4
# Step22恶性标签审计
# ============================================================

assert (
    "malignant_status"
    in adata_label.obs.columns
)


print("=" * 100)
print("STEP22 MALIGNANCY LABELS")
print("=" * 100)


malignancy_counts = (
    adata_label.obs[
        "malignant_status"
    ]
    .astype(str)
    .value_counts()
)


display(
    malignancy_counts
    .to_frame(
        "N_cells"
    )
)


malignant_cells = (
    adata_label.obs_names[
        adata_label.obs[
            "malignant_status"
        ]
        .astype(str)
        ==
        "Malignant"
    ]
    .tolist()
)


print(
    "\nMalignant cells:",
    f"{len(malignant_cells):,}"
)


assert len(
    malignant_cells
) == 12406, (
    "Malignant cell数量与Step22最终结果不一致，"
    "请停止检查。"
)

In [ ]:
# ============================================================
# STEP23A - CELL 5
# 从原始counts提取高可信Malignant cells
# ============================================================

# 按Step22标签中的顺序提取
adata_mal_raw = (
    adata_raw[
        malignant_cells,
        :
    ]
    .copy()
)


print("=" * 100)
print("MALIGNANT RAW DATASET")
print("=" * 100)

print(
    "Shape:",
    adata_mal_raw.shape
)

print(
    "Cells:",
    f"{adata_mal_raw.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_mal_raw.n_vars:,}"
)


assert (
    adata_mal_raw.n_obs
    ==
    len(malignant_cells)
)

In [ ]:
# ============================================================
# STEP23A - CELL 6
# 同步Step22最终metadata
# ============================================================

metadata_columns = [

    # 基础信息
    "sample",
    "patient",
    "site",

    # Step21
    "major_celltype",

    # Step22 CNV
    "cnv_leiden_r0_4",
    "cnv_reference_class",

    "cnv_burden_primary",
    "cnv_burden_balanced",
    "cnv_burden_heponly",

    # 表达程序
    "mature_hepatocyte_score",
    "ductal_progenitor_score",
    "tumor_associated_expression_score",

    # 最终恶性判定
    "malignant_status",
    "malignancy_confidence",
    "malignancy_basis",

    # 原QC变量，如存在
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "total_counts_mt",

    "total_counts_ribo",
    "pct_counts_ribo"
]


metadata_columns = [
    x
    for x in metadata_columns
    if x in adata_label.obs.columns
]


label_metadata = (
    adata_label.obs.loc[
        adata_mal_raw.obs_names,
        metadata_columns
    ]
    .copy()
)


# 避免重复字段冲突
for col in metadata_columns:

    adata_mal_raw.obs[
        col
    ] = (
        label_metadata[
            col
        ]
        .values
    )


print(
    "Transferred metadata columns:"
)

print(
    metadata_columns
)


print(
    "\nmalignant_status:"
)

display(
    adata_mal_raw.obs[
        "malignant_status"
    ]
    .astype(str)
    .value_counts()
)


assert (
    adata_mal_raw.obs[
        "malignant_status"
    ]
    .astype(str)
    ==
    "Malignant"
).all()

In [ ]:
# ============================================================
# STEP23A - CELL 7
# RAW COUNT审计
# ============================================================

X = adata_mal_raw.X


if sparse.issparse(X):

    data_values = (
        X.data
    )

else:

    data_values = (
        np.asarray(X)
        .ravel()
    )


# 只检查非零值
nonzero_values = (
    data_values[
        data_values != 0
    ]
)


if len(nonzero_values) > 200000:

    rng = np.random.default_rng(
        SEED
    )

    test_values = rng.choice(
        nonzero_values,
        size=200000,
        replace=False
    )

else:

    test_values = (
        nonzero_values
    )


integer_fraction = float(
    np.mean(
        np.isclose(
            test_values,
            np.round(
                test_values
            ),
            atol=1e-6
        )
    )
)


negative_fraction = float(
    np.mean(
        test_values < 0
    )
)


print("=" * 100)
print("RAW COUNT AUDIT")
print("=" * 100)

print(
    "X type:",
    type(X)
)

print(
    "Non-zero values checked:",
    len(test_values)
)

print(
    "Integer-like fraction:",
    round(
        integer_fraction,
        6
    )
)

print(
    "Negative fraction:",
    round(
        negative_fraction,
        6
    )
)

print(
    "Min:",
    float(
        np.min(
            test_values
        )
    )
)

print(
    "Median:",
    float(
        np.median(
            test_values
        )
    )
)

print(
    "Max:",
    float(
        np.max(
            test_values
        )
    )
)


assert integer_fraction > 0.999, (
    "X不像原始整数counts，请停止。"
)

assert negative_fraction == 0, (
    "RAW counts出现负值，请停止。"
)


print("\nRAW counts = PASS")

In [ ]:
# ============================================================
# STEP23A - CELL 8
# malignant-only QC重新计算
# 不进行任何二次过滤
# ============================================================

# ------------------------------------------------------------
# mitochondrial genes
# ------------------------------------------------------------

adata_mal_raw.var[
    "mt"
] = (
    adata_mal_raw.var_names
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


# ------------------------------------------------------------
# ribosomal genes
# ------------------------------------------------------------

adata_mal_raw.var[
    "ribo"
] = (
    adata_mal_raw.var_names
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


sc.pp.calculate_qc_metrics(

    adata_mal_raw,

    qc_vars=[
        "mt",
        "ribo"
    ],

    percent_top=None,

    log1p=False,

    inplace=True
)


print("=" * 100)
print("MALIGNANT-ONLY QC")
print("=" * 100)


qc_columns = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_ribo"
]


display(
    adata_mal_raw.obs[
        qc_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
    .round(3)
)

In [ ]:
# ============================================================
# STEP23A - CELL 9
# Malignant cells患者组成
# ============================================================

patient_counts = (
    adata_mal_raw.obs[
        "patient"
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "patient"
    )
    .reset_index(
        name="N_cells"
    )
)


patient_counts[
    "Pct_cells"
] = (

    patient_counts[
        "N_cells"
    ]

    /
    adata_mal_raw.n_obs

    *
    100
)


patient_counts[
    "Rank"
] = np.arange(
    1,
    len(patient_counts) + 1
)


print("=" * 100)
print("MALIGNANT CELLS BY PATIENT")
print("=" * 100)


display(
    patient_counts
    .round(3)
)


patient_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Malignant_Cells_ByPatient.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 10
# Sample / Site composition
# ============================================================

sample_counts = (
    adata_mal_raw.obs[
        "sample"
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "sample"
    )
    .reset_index(
        name="N_cells"
    )
)


sample_counts[
    "Pct_cells"
] = (

    sample_counts[
        "N_cells"
    ]

    /
    adata_mal_raw.n_obs

    *
    100
)


site_counts = (
    adata_mal_raw.obs[
        "site"
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "site"
    )
    .reset_index(
        name="N_cells"
    )
)


site_counts[
    "Pct_cells"
] = (

    site_counts[
        "N_cells"
    ]

    /
    adata_mal_raw.n_obs

    *
    100
)


print("=" * 100)
print("BY SAMPLE")
print("=" * 100)

display(
    sample_counts
    .round(3)
)


print("\n" + "=" * 100)
print("BY SITE")
print("=" * 100)

display(
    site_counts
    .round(3)
)


sample_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Malignant_Cells_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)


site_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Malignant_Cells_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 11
# Patient × Site
# ============================================================

patient_site_counts = pd.crosstab(

    adata_mal_raw.obs[
        "patient"
    ].astype(str),

    adata_mal_raw.obs[
        "site"
    ].astype(str)

)


patient_site_pct = (
    patient_site_counts
    .div(
        patient_site_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


print("=" * 100)
print("PATIENT × SITE COUNTS")
print("=" * 100)

display(
    patient_site_counts
)


print("\nPATIENT × SITE %")

display(
    patient_site_pct
    .round(2)
)


patient_site_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Patient_x_Site_Counts.csv",
    encoding="utf-8-sig"
)


patient_site_pct.to_csv(
    RESULTS_DIR /
    "Step23A_Patient_x_Site_Percent.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 12
# Malignant cells CNV clone composition
# ============================================================

CNV_KEY = (
    "cnv_leiden_r0_4"
)


assert (
    CNV_KEY
    in adata_mal_raw.obs.columns
)


cnv_cluster_counts = (
    adata_mal_raw.obs[
        CNV_KEY
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "CNV_cluster"
    )
    .reset_index(
        name="N_cells"
    )
)


cnv_cluster_counts[
    "Pct_cells"
] = (

    cnv_cluster_counts[
        "N_cells"
    ]

    /
    adata_mal_raw.n_obs

    *
    100
)


print("=" * 100)
print("MALIGNANT CNV CLONES")
print("=" * 100)

display(
    cnv_cluster_counts
    .round(3)
)


cnv_cluster_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Malignant_CNV_Clone_Composition.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 13
# Patient × CNV clone
# ============================================================

patient_cnv_counts = pd.crosstab(

    adata_mal_raw.obs[
        "patient"
    ].astype(str),

    adata_mal_raw.obs[
        CNV_KEY
    ].astype(str)

)


patient_cnv_pct = (
    patient_cnv_counts
    .div(
        patient_cnv_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


print("=" * 100)
print("PATIENT × CNV CLONE")
print("=" * 100)

display(
    patient_cnv_counts
)


patient_cnv_counts.to_csv(
    RESULTS_DIR /
    "Step23A_Patient_x_CNVClone_Counts.csv",
    encoding="utf-8-sig"
)


patient_cnv_pct.to_csv(
    RESULTS_DIR /
    "Step23A_Patient_x_CNVClone_Percent.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 14
# 给每个细胞添加患者malignant-cell数量
# ============================================================

patient_cell_number_map = (

    adata_mal_raw.obs[
        "patient"
    ]
    .astype(str)
    .value_counts()
    .to_dict()
)


adata_mal_raw.obs[
    "patient_malignant_n"
] = (

    adata_mal_raw.obs[
        "patient"
    ]
    .astype(str)
    .map(
        patient_cell_number_map
    )
    .astype(int)
)


print("=" * 100)
print("PATIENT MALIGNANT CELL SUPPORT")
print("=" * 100)


display(

    adata_mal_raw.obs[
        [
            "patient",
            "patient_malignant_n"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "patient_malignant_n",
        ascending=False
    )

)

In [ ]:
# ============================================================
# STEP23A - CELL 15
# QC by patient
# ============================================================

qc_by_patient = (

    adata_mal_raw.obs
    .groupby(
        "patient",
        observed=True
    )
    .agg(

        N_cells=(
            "total_counts",
            "size"
        ),

        Median_counts=(
            "total_counts",
            "median"
        ),

        Median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        Median_pct_mt=(
            "pct_counts_mt",
            "median"
        ),

        P95_pct_mt=(
            "pct_counts_mt",
            lambda x:
                x.quantile(
                    0.95
                )
        )
    )
    .reset_index()
    .sort_values(
        "N_cells",
        ascending=False
    )

)


print("=" * 100)
print("QC BY PATIENT")
print("=" * 100)


display(
    qc_by_patient
    .round(3)
)


qc_by_patient.to_csv(
    RESULTS_DIR /
    "Step23A_QC_ByPatient.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23A - CELL 16
# Publication plotting settings
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Arial"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi":
        120,

    "savefig.dpi":
        600,

    "figure.facecolor":
        "white",

    "axes.facecolor":
        "white",

    "font.size":
        9,

    "axes.labelsize":
        10,

    "axes.titlesize":
        11,

    "xtick.labelsize":
        9,

    "ytick.labelsize":
        9,

    "legend.fontsize":
        8,

    "axes.linewidth":
        0.8,

    "axes.grid":
        False,

    "pdf.fonttype":
        42,

    "ps.fonttype":
        42
})


NPG = [

    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85",
    "#BDBDBD"
]


def save_publication_figure(
    fig,
    filename
):

    png_path = (
        FIGURES_DIR /
        f"{filename}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{filename}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23A - CELL 17
# Malignant cells by patient
# ============================================================

plot_df = (
    patient_counts
    .sort_values(
        "N_cells",
        ascending=True
    )
)


fig, ax = plt.subplots(
    figsize=(7.2, 5.2)
)


bars = ax.barh(

    plot_df[
        "patient"
    ],

    plot_df[
        "N_cells"
    ]
)


ax.set_xlabel(
    "恶性细胞数量"
)

ax.set_ylabel(
    "患者"
)

ax.set_title(
    "Step23 malignant-only 队列的患者组成"
)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


for bar, n in zip(
    bars,
    plot_df[
        "N_cells"
    ]
):

    ax.text(

        bar.get_width(),

        bar.get_y()
        +
        bar.get_height()
        / 2,

        f" {n:,}",

        va="center",

        fontsize=8
    )


fig.tight_layout()


save_publication_figure(
    fig,
    "Step23A_01_Malignant_Cells_ByPatient"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23A - CELL 18
# Malignant cells by site
# ============================================================

site_order = [
    x
    for x in [
        "Tumor",
        "PVTT",
        "Lymph",
        "Normal"
    ]
    if x in site_counts[
        "site"
    ].tolist()
]


site_plot = (
    site_counts
    .set_index(
        "site"
    )
    .loc[
        site_order
    ]
    .reset_index()
)


SITE_COLORS = {

    "Normal":
        "#4DBBD5",

    "Tumor":
        "#E64B35",

    "PVTT":
        "#00A087",

    "Lymph":
        "#3C5488"
}


fig, ax = plt.subplots(
    figsize=(6.0, 5.0)
)


bars = ax.bar(

    site_plot[
        "site"
    ],

    site_plot[
        "N_cells"
    ],

    color=[
        SITE_COLORS[x]
        for x in site_plot[
            "site"
        ]
    ]
)


ax.set_ylabel(
    "恶性细胞数量"
)

ax.set_xlabel(
    "取样部位"
)

ax.set_title(
    "Malignant cells 的组织来源"
)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


for bar, n in zip(
    bars,
    site_plot[
        "N_cells"
    ]
):

    ax.text(

        bar.get_x()
        +
        bar.get_width()
        / 2,

        bar.get_height(),

        f"{n:,}",

        ha="center",
        va="bottom",
        fontsize=8
    )


fig.tight_layout()


save_publication_figure(
    fig,
    "Step23A_02_Malignant_Cells_BySite"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23A - CELL 19
# malignant-only QC distributions
# ============================================================

qc_plot_vars = [

    (
        "total_counts",
        "UMI / counts"
    ),

    (
        "n_genes_by_counts",
        "检测基因数"
    ),

    (
        "pct_counts_mt",
        "线粒体比例 (%)"
    )
]


for i, (
    col,
    label
) in enumerate(
    qc_plot_vars,
    start=1
):

    values = (
        adata_mal_raw.obs[
            col
        ]
        .dropna()
        .values
    )


    fig, ax = plt.subplots(
        figsize=(6.2, 4.8)
    )


    ax.hist(
        values,
        bins=60,
        alpha=0.85
    )


    median_value = float(
        np.median(
            values
        )
    )


    ax.axvline(
        median_value,
        linestyle="--",
        linewidth=1
    )


    ax.set_xlabel(
        label
    )

    ax.set_ylabel(
        "细胞数"
    )

    ax.set_title(
        f"Malignant cells：{label}"
    )


    ax.text(
        0.98,
        0.95,
        f"Median = {median_value:.2f}",
        transform=ax.transAxes,
        ha="right",
        va="top"
    )


    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)


    fig.tight_layout()


    save_publication_figure(

        fig,

        (
            f"Step23A_0{i+2}_QC_"
            +
            col
        )
    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23A - CELL 20
# 保存malignant-only raw counts
# ============================================================

MALIGNANT_RAW_OUTPUT = (
    DATA_DIR /
    "GSE149614_STEP23A_MALIGNANT_RAW_COUNTS.h5ad"
)


adata_mal_raw.write_h5ad(
    MALIGNANT_RAW_OUTPUT,
    compression="gzip"
)


print("=" * 100)
print("STEP23A RAW MALIGNANT OBJECT SAVED")
print("=" * 100)

print(
    MALIGNANT_RAW_OUTPUT
)

print(
    "\nShape:",
    adata_mal_raw.shape
)

In [ ]:
# ============================================================
# STEP23A - CELL 21
# 保存cell-level metadata
# ============================================================

metadata_output = (
    adata_mal_raw.obs
    .copy()
)


metadata_output.index.name = (
    "Cell"
)


MALIGNANT_METADATA_OUTPUT = (
    RESULTS_DIR /
    "Step23A_Malignant_Cell_Metadata.csv"
)


metadata_output.to_csv(
    MALIGNANT_METADATA_OUTPUT,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MALIGNANT_METADATA_OUTPUT
)

print(
    "Rows:",
    len(
        metadata_output
    )
)

In [ ]:
# ============================================================
# STEP23A - CELL 22
# Step23A summary / decision log
# ============================================================

step23a_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23A_Malignant_Only_Data_Audit",

    "Status":
        "COMPLETE",

    "Input_raw_counts":
        str(
            RAW_INPUT
        ),

    "Input_step22_labels":
        str(
            LABEL_INPUT
        ),

    "Malignant_selection_rule":
        (
            "Step22 malignant_status == Malignant"
        ),

    "N_malignant_cells":
        int(
            adata_mal_raw.n_obs
        ),

    "N_genes":
        int(
            adata_mal_raw.n_vars
        ),

    "Raw_counts_verified":
        bool(
            integer_fraction > 0.999
            and
            negative_fraction == 0
        ),

    "Secondary_QC_filtering":
        False,

    "Rationale":
        (
            "Step23A preserves all Step22 high-confidence malignant "
            "cells. QC is re-audited but no second hard filtering is "
            "performed."
        ),

    "Important_design":
        (
            "Patient imbalance and CNV clone composition are retained "
            "and explicitly tracked because malignant-state clustering "
            "must not be interpreted as biological state without "
            "cross-patient support."
        ),

    "Output_raw_h5ad":
        str(
            MALIGNANT_RAW_OUTPUT
        ),

    "Next_step":
        (
            "Step23B malignant-only normalization, HVG, PCA and "
            "patient-effect audit"
        )
}


SUMMARY_PATH = (
    LOGS_DIR /
    "Step23A_Final_Summary.json"
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step23a_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    json.dumps(
        step23a_summary,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP23A - CELL 23
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    (
        "Data",
        DATA_DIR
    ),

    (
        "Results",
        RESULTS_DIR
    ),

    (
        "Figures",
        FIGURES_DIR
    ),

    (
        "Logs",
        LOGS_DIR
    )

]:

    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (
    RESULTS_DIR /
    "Step23A_Output_Manifest.csv"
)


manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    manifest_df
)


print(
    "\nSaved:",
    MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP23A - CELL 24
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 115)
print("STEP23A FINAL REVIEW")
print("=" * 115)


print("\n[1] Malignant-only dataset")
print("-" * 115)

print(
    "Cells:",
    f"{adata_mal_raw.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_mal_raw.n_vars:,}"
)

print(
    "RAW integer-like fraction:",
    round(
        integer_fraction,
        6
    )
)


print("\n[2] Malignant cells by patient")
print("-" * 115)

display(
    patient_counts
    .round(3)
)


print("\n[3] Malignant cells by site")
print("-" * 115)

display(
    site_counts
    .round(3)
)


print("\n[4] Patient × Site")
print("-" * 115)

display(
    patient_site_counts
)


print("\n[5] Malignant CNV clone composition")
print("-" * 115)

display(
    cnv_cluster_counts
    .round(3)
)


print("\n[6] Patient × CNV clone")
print("-" * 115)

display(
    patient_cnv_counts
)


print("\n[7] QC by patient")
print("-" * 115)

display(
    qc_by_patient
    .round(3)
)


print("\n[8] Files")
print("-" * 115)

for path in [

    MALIGNANT_RAW_OUTPUT,
    MALIGNANT_METADATA_OUTPUT,
    SUMMARY_PATH,
    MANIFEST_PATH

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 115)

print(
    "STEP23A STATUS = COMPLETE"
)

print(
    "NEXT = Step23B malignant-only normalization / HVG / PCA"
)

print("=" * 115)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23B - CELL 25
# 初始化 malignant-only transcriptional space
# ============================================================

from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.stats import spearmanr

from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

import json
import warnings
import gc

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# 路径
# ============================================================

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

for folder in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


INPUT_RAW = (
    DATA_DIR /
    "GSE149614_STEP23A_MALIGNANT_RAW_COUNTS.h5ad"
)

assert INPUT_RAW.exists()


adata = sc.read_h5ad(
    INPUT_RAW
)


print("=" * 105)
print("STEP23B MALIGNANT-ONLY TRANSCRIPTIONAL SPACE")
print("=" * 105)

print(
    "Input:",
    INPUT_RAW
)

print(
    "Shape:",
    adata.shape
)

print(
    "Patients:",
    adata.obs["patient"].nunique()
)

print(
    "Samples:",
    adata.obs["sample"].nunique()
)

print(
    "Sites:",
    sorted(
        adata.obs["site"]
        .astype(str)
        .unique()
    )
)

assert adata.n_obs == 12406

print("\nSTEP23B INPUT = PASS")

In [ ]:
# ============================================================
# STEP23B - CELL 26
# Publication plotting settings
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Arial"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


NPG_COLORS = [

    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85",
    "#BDBDBD"
]


SITE_COLORS = {

    "Normal": "#4DBBD5",
    "Tumor": "#E64B35",
    "PVTT": "#00A087",
    "Lymph": "#3C5488"
}


def save_publication_figure(
    fig,
    filename
):

    png_path = (
        FIGURES_DIR /
        f"{filename}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{filename}.pdf"
    )


    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23B - CELL 27
# RAW count safety check
# ============================================================

X = adata.X


if sparse.issparse(X):

    values = X.data

else:

    values = np.asarray(
        X
    ).ravel()


values = values[
    values != 0
]


rng = np.random.default_rng(
    SEED
)


if len(values) > 200000:

    test_values = rng.choice(
        values,
        size=200000,
        replace=False
    )

else:

    test_values = values


integer_fraction = np.mean(

    np.isclose(

        test_values,

        np.round(
            test_values
        ),

        atol=1e-6
    )
)


negative_fraction = np.mean(
    test_values < 0
)


print(
    "Integer-like fraction:",
    round(
        float(integer_fraction),
        6
    )
)

print(
    "Negative fraction:",
    round(
        float(negative_fraction),
        6
    )
)


assert integer_fraction > 0.999
assert negative_fraction == 0


print("\nRAW COUNTS = PASS")

In [ ]:
# ============================================================
# STEP23B - CELL 28
# Normalization + log1p
# ============================================================

adata_log = (
    adata.copy()
)


sc.pp.normalize_total(

    adata_log,

    target_sum=1e4,

    inplace=True
)


sc.pp.log1p(
    adata_log
)


print("=" * 100)
print("NORMALIZATION COMPLETE")
print("=" * 100)

print(
    "Shape:",
    adata_log.shape
)

print(
    "Normalization:",
    "library size = 1e4 + log1p"
)

In [ ]:
# ============================================================
# STEP23B - CELL 29
# 保存全基因log-normalized checkpoint
# ============================================================

LOGNORM_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23B_MALIGNANT_LOGNORM_ALL_GENES.h5ad"

)


adata_log.write_h5ad(
    LOGNORM_OUTPUT,
    compression="gzip"
)


print(
    "Saved:"
)

print(
    LOGNORM_OUTPUT
)

In [ ]:
# ============================================================
# STEP23B - CELL 30
# 定义用于HVG估计的患者
# ============================================================

MIN_PATIENT_CELLS_FOR_HVG = 20


patient_n = (

    adata_log.obs[
        "patient"
    ]
    .astype(str)
    .value_counts()
)


hvg_patients = (

    patient_n[
        patient_n
        >=
        MIN_PATIENT_CELLS_FOR_HVG
    ]
    .index
    .tolist()
)


excluded_hvg_patients = (

    patient_n[
        patient_n
        <
        MIN_PATIENT_CELLS_FOR_HVG
    ]
    .index
    .tolist()
)


hvg_fit_mask = (

    adata_log.obs[
        "patient"
    ]
    .astype(str)
    .isin(
        hvg_patients
    )
)


print("=" * 100)
print("HVG DISCOVERY COHORT")
print("=" * 100)

print(
    "Included patients:"
)

print(
    hvg_patients
)

print(
    "\nExcluded from HVG variance estimation:"
)

print(
    excluded_hvg_patients
)

print(
    "\nCells used to estimate HVGs:",
    int(
        hvg_fit_mask.sum()
    )
)

print(
    "Cells retained in final analysis:",
    adata_log.n_obs
)

In [ ]:
# ============================================================
# STEP23B - CELL 31
# patient-aware HVG selection
# ============================================================

N_TOP_HVG = 3000


adata_hvg_fit = (

    adata_log[
        hvg_fit_mask,
        :
    ]
    .copy()
)


sc.pp.highly_variable_genes(

    adata_hvg_fit,

    flavor="seurat",

    n_top_genes=N_TOP_HVG,

    batch_key="patient",

    inplace=True
)


# ============================================================
# 将HVG信息同步回完整12,406-cell对象
# ============================================================

adata_log.var[
    "highly_variable"
] = False


selected_hvg_genes = (

    adata_hvg_fit.var_names[
        adata_hvg_fit.var[
            "highly_variable"
        ]
    ]
    .tolist()
)


adata_log.var.loc[
    selected_hvg_genes,
    "highly_variable"
] = True


# 同步HVG相关统计
hvg_stat_cols = [

    "means",
    "dispersions",
    "dispersions_norm",
    "highly_variable_nbatches",
    "highly_variable_intersection"

]


for col in hvg_stat_cols:

    if col in adata_hvg_fit.var.columns:

        adata_log.var[
            col
        ] = (
            adata_hvg_fit.var[
                col
            ]
            .reindex(
                adata_log.var_names
            )
            .values
        )


print("=" * 100)
print("HVG SELECTION")
print("=" * 100)

print(
    "Selected HVGs:",
    int(
        adata_log.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "HVG patients:",
    len(
        hvg_patients
    )
)

In [ ]:
# ============================================================
# STEP23B - CELL 32
# HVG跨患者支持审计
# ============================================================

hvg_columns = [

    "highly_variable"

]


for col in [

    "means",
    "dispersions",
    "dispersions_norm",
    "highly_variable_nbatches",
    "highly_variable_intersection"

]:

    if col in adata_log.var.columns:

        hvg_columns.append(
            col
        )


hvg_table = (

    adata_log.var[
        hvg_columns
    ]
    .copy()
)


hvg_table.index.name = (
    "Gene"
)


hvg_table[
    "Gene"
] = hvg_table.index


hvg_table[
    "Is_MT"
] = (

    hvg_table[
        "Gene"
    ]
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


hvg_table[
    "Is_Ribosomal"
] = (

    hvg_table[
        "Gene"
    ]
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


HVG_TABLE_PATH = (

    RESULTS_DIR /
    "Step23B_HVG_Full_Table.csv"

)


hvg_table.to_csv(
    HVG_TABLE_PATH,
    encoding="utf-8-sig"
)


# ============================================================
# selected HVG support
# ============================================================

selected_hvg_table = (

    hvg_table[
        hvg_table[
            "highly_variable"
        ]
        ==
        True
    ]
    .copy()
)


if (
    "highly_variable_nbatches"
    in selected_hvg_table.columns
):

    hvg_support = (

        selected_hvg_table[
            "highly_variable_nbatches"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "N_patients_HVG"
        )
        .reset_index(
            name="N_genes"
        )
    )

else:

    hvg_support = pd.DataFrame()


print("=" * 100)
print("HVG CROSS-PATIENT SUPPORT")
print("=" * 100)

if len(
    hvg_support
) > 0:

    display(
        hvg_support
    )


    hvg_support.to_csv(

        RESULTS_DIR /
        "Step23B_HVG_CrossPatient_Support.csv",

        index=False,
        encoding="utf-8-sig"
    )


print(
    "\nMitochondrial genes among HVGs:",
    int(
        selected_hvg_table[
            "Is_MT"
        ].sum()
    )
)

print(
    "Ribosomal genes among HVGs:",
    int(
        selected_hvg_table[
            "Is_Ribosomal"
        ].sum()
    )
)

In [ ]:
# ============================================================
# STEP23B - CELL 33
# 定义PCA使用HVG
# ============================================================

is_mt_gene = (

    adata_log.var_names
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


adata_log.var[
    "highly_variable_for_pca"
] = (

    adata_log.var[
        "highly_variable"
    ]
    .astype(bool)

    &

    (~is_mt_gene)
)


pca_genes = (

    adata_log.var_names[
        adata_log.var[
            "highly_variable_for_pca"
        ]
    ]
    .tolist()
)


print(
    "Original HVGs:",
    int(
        adata_log.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "HVGs used for PCA:",
    len(
        pca_genes
    )
)

print(
    "MT genes excluded from PCA:",
    int(
        np.sum(
            adata_log.var[
                "highly_variable"
            ].astype(bool).values
            &
            np.asarray(
                is_mt_gene
            )
        )
    )
)

In [ ]:
# ============================================================
# STEP23B - CELL 34
# HVG diagnostic plot
# ============================================================

plot_hvg = (
    hvg_table
    .copy()
)


fig, ax = plt.subplots(
    figsize=(6.5, 5.4)
)


non_hvg = (

    plot_hvg[
        "highly_variable"
    ]
    ==
    False
)


hvg_mask = (

    plot_hvg[
        "highly_variable"
    ]
    ==
    True
)


ax.scatter(

    plot_hvg.loc[
        non_hvg,
        "means"
    ],

    plot_hvg.loc[
        non_hvg,
        "dispersions_norm"
    ],

    s=4,
    alpha=0.25,
    color="#BDBDBD",
    edgecolors="none"
)


ax.scatter(

    plot_hvg.loc[
        hvg_mask,
        "means"
    ],

    plot_hvg.loc[
        hvg_mask,
        "dispersions_norm"
    ],

    s=6,
    alpha=0.65,
    color="#E64B35",
    edgecolors="none",
    label=f"HVG (n={hvg_mask.sum():,})"
)


ax.set_xlabel(
    "平均表达"
)

ax.set_ylabel(
    "标准化离散度"
)

ax.set_title(
    "Malignant-only patient-aware HVG"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23B_01_PatientAware_HVG"

)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23B - CELL 35
# Scale HVGs + PCA
# ============================================================

adata_pca = (

    adata_log[
        :,
        pca_genes
    ]
    .copy()
)


print(
    "PCA input:",
    adata_pca.shape
)


sc.pp.scale(

    adata_pca,

    zero_center=True,

    max_value=10
)


sc.tl.pca(

    adata_pca,

    n_comps=50,

    svd_solver="arpack",

    random_state=42
)


# ============================================================
# transfer PCA back to full-gene object
# ============================================================

adata_log.obsm[
    "X_pca_malignant_unintegrated"
] = (

    adata_pca.obsm[
        "X_pca"
    ]
    .copy()
)


print(
    "PCA shape:",
    adata_log.obsm[
        "X_pca_malignant_unintegrated"
    ].shape
)

In [ ]:
# ============================================================
# STEP23B - CELL 36
# PCA variance
# ============================================================

variance_ratio = (

    adata_pca.uns[
        "pca"
    ][
        "variance_ratio"
    ]
)


pca_variance = pd.DataFrame({

    "PC":
        np.arange(
            1,
            len(
                variance_ratio
            )
            +
            1
        ),

    "VarianceRatio":
        variance_ratio
})


pca_variance[
    "CumulativeVariance"
] = (

    pca_variance[
        "VarianceRatio"
    ]
    .cumsum()
)


display(
    pca_variance
    .head(30)
    .round(5)
)


pca_variance.to_csv(

    RESULTS_DIR /
    "Step23B_PCA_Variance.csv",

    index=False,
    encoding="utf-8-sig"
)


fig, ax = plt.subplots(
    figsize=(7.0, 5.0)
)


ax.plot(

    pca_variance[
        "PC"
    ],

    pca_variance[
        "CumulativeVariance"
    ],

    marker="o",
    markersize=3,
    linewidth=1
)


ax.set_xlabel(
    "主成分"
)

ax.set_ylabel(
    "累计解释方差"
)

ax.set_title(
    "Malignant-only PCA累计解释方差"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


save_publication_figure(
    fig,
    "Step23B_02_PCA_CumulativeVariance"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23B - CELL 37
# PCA loading审计
# ============================================================

loadings = (

    adata_pca.varm[
        "PCs"
    ]
)


loading_rows = []


N_TOP_LOADING = 20
N_PC_AUDIT = 20


for pc_idx in range(
    min(
        N_PC_AUDIT,
        loadings.shape[1]
    )
):

    values = (
        loadings[
            :,
            pc_idx
        ]
    )


    top_idx = (

        np.argsort(
            np.abs(
                values
            )
        )[
            ::-1
        ][
            :N_TOP_LOADING
        ]
    )


    for rank, idx in enumerate(
        top_idx,
        start=1
    ):

        gene = (
            adata_pca.var_names[
                idx
            ]
        )


        loading_rows.append({

            "PC":
                pc_idx + 1,

            "Rank":
                rank,

            "Gene":
                gene,

            "Loading":
                float(
                    values[
                        idx
                    ]
                ),

            "AbsLoading":
                float(
                    abs(
                        values[
                            idx
                        ]
                    )
                ),

            "Is_Ribosomal":
                bool(
                    gene.upper().startswith(
                        (
                            "RPL",
                            "RPS"
                        )
                    )
                ),

            "Is_MT":
                bool(
                    gene.upper().startswith(
                        "MT-"
                    )
                )
        })


pca_loadings = pd.DataFrame(
    loading_rows
)


pca_loadings.to_csv(

    RESULTS_DIR /
    "Step23B_PCA_TopLoadings_PC1_20.csv",

    index=False,
    encoding="utf-8-sig"
)


print("=" * 100)
print("TOP PCA LOADINGS")
print("=" * 100)


display(
    pca_loadings[
        pca_loadings[
            "PC"
        ]
        <=
        5
    ]
)

In [ ]:
# ============================================================
# STEP23B - CELL 38
# Unintegrated malignant-only neighbors + UMAP
# ============================================================

N_PCS_PRIMARY = 30
N_NEIGHBORS = 15


sc.pp.neighbors(

    adata_log,

    use_rep=
        "X_pca_malignant_unintegrated",

    n_pcs=
        N_PCS_PRIMARY,

    n_neighbors=
        N_NEIGHBORS,

    key_added=
        "neighbors_malignant_unintegrated",

    random_state=42
)


sc.tl.umap(

    adata_log,

    neighbors_key=
        "neighbors_malignant_unintegrated",

    random_state=42,

    min_dist=0.5
)


adata_log.obsm[
    "X_umap_malignant_unintegrated"
] = (

    adata_log.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "UMAP shape:",
    adata_log.obsm[
        "X_umap_malignant_unintegrated"
    ].shape
)

In [ ]:
# ============================================================
# STEP23B - CELL 39
# 保存embedding coordinates
# ============================================================

umap_coords = pd.DataFrame(

    adata_log.obsm[
        "X_umap_malignant_unintegrated"
    ],

    index=
        adata_log.obs_names,

    columns=[
        "UMAP1",
        "UMAP2"
    ]
)


umap_meta_cols = [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",

    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",

    "mature_hepatocyte_score",
    "ductal_progenitor_score",
    "tumor_associated_expression_score"
]


umap_meta_cols = [

    x
    for x in umap_meta_cols
    if x in adata_log.obs.columns

]


for col in umap_meta_cols:

    umap_coords[
        col
    ] = (

        adata_log.obs.loc[
            umap_coords.index,
            col
        ]
        .values
    )


umap_coords.index.name = "Cell"


umap_coords.to_csv(

    RESULTS_DIR /
    "Step23B_Unintegrated_UMAP_Coordinates.csv",

    encoding="utf-8-sig"
)


# PCA前30维
pca_coords = pd.DataFrame(

    adata_log.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :30
    ],

    index=
        adata_log.obs_names,

    columns=[
        f"PC{i}"
        for i in range(
            1,
            31
        )
    ]
)


pca_coords.index.name = (
    "Cell"
)


pca_coords.to_csv(

    RESULTS_DIR /
    "Step23B_Unintegrated_PCA_Coordinates_PC1_30.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23B - FIX
# 清理malignant-only对象中的unused categorical levels
# ============================================================

categorical_columns = [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"
]


print("=" * 100)
print("CLEAN UNUSED CATEGORICAL LEVELS")
print("=" * 100)


for col in categorical_columns:

    if col not in adata_log.obs.columns:
        continue

    # --------------------------------------------------------
    # 看清理前类别
    # --------------------------------------------------------

    if isinstance(
        adata_log.obs[col].dtype,
        pd.CategoricalDtype
    ):

        before_categories = (
            adata_log.obs[col]
            .cat
            .categories
            .astype(str)
            .tolist()
        )

    else:

        before_categories = sorted(
            adata_log.obs[col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


    # --------------------------------------------------------
    # 重新根据实际存在值创建category
    # 这样会彻底删除 HCC03/HCC05/HCC06 等unused levels
    # --------------------------------------------------------

    adata_log.obs[col] = (
        adata_log.obs[col]
        .astype(str)
        .astype("category")
    )


    after_categories = (
        adata_log.obs[col]
        .cat
        .categories
        .astype(str)
        .tolist()
    )


    print(f"\n{col}")

    print(
        "Before:",
        before_categories
    )

    print(
        "After :",
        after_categories
    )


print("\nUnused categorical levels cleaned.")

In [ ]:
# ============================================================
# STEP23B - CELL 40 修正版
# UMAP categorical diagnostics
# Patient / CNV clone / Site
# ============================================================

categorical_plots = [

    (
        "patient",
        "患者",
        "Step23B_03_UMAP_ByPatient"
    ),

    (
        "cnv_leiden_r0_4",
        "CNV clone",
        "Step23B_04_UMAP_ByCNVClone"
    ),

    (
        "site",
        "取样部位",
        "Step23B_05_UMAP_BySite"
    )
]


for color_col, title_text, filename in categorical_plots:

    # ========================================================
    # 再保险：只保留实际存在category
    # ========================================================

    adata_log.obs[color_col] = (
        adata_log.obs[color_col]
        .astype(str)
        .astype("category")
    )


    categories = (
        adata_log.obs[color_col]
        .cat.categories
        .astype(str)
        .tolist()
    )


    print("\n" + "=" * 80)

    print(
        color_col,
        ":",
        categories
    )


    # ========================================================
    # 设置palette
    # ========================================================

    if color_col == "site":

        palette = {

            category:
                SITE_COLORS.get(
                    category,
                    "#BDBDBD"
                )

            for category in categories
        }


    else:

        palette = {

            category:
                NPG_COLORS[
                    i
                    %
                    len(
                        NPG_COLORS
                    )
                ]

            for i, category in enumerate(
                categories
            )
        }


    print(
        "Palette keys:",
        list(
            palette.keys()
        )
    )


    # ========================================================
    # Plot
    # ========================================================

    ax = sc.pl.embedding(

        adata_log,

        basis=
            "umap_malignant_unintegrated",

        color=
            color_col,

        palette=
            palette,

        size=8,

        alpha=0.78,

        frameon=False,

        legend_loc=
            "right margin",

        title=
            f"Malignant-only 未整合空间：{title_text}",

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        8.3,
        6.5
    )


    fig.tight_layout()


    # ========================================================
    # 先保存，再显示
    # ========================================================

    save_publication_figure(
        fig,
        filename
    )


    plt.show()
    plt.close(fig)


print("\n" + "=" * 100)
print("CELL 40 COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# STEP23B - CELL 41
# UMAP QC diagnostics
# ============================================================

qc_umap_vars = [

    (
        "total_counts",
        "总Counts"
    ),

    (
        "n_genes_by_counts",
        "检测基因数"
    ),

    (
        "pct_counts_mt",
        "线粒体比例"
    )
]


for i, (
    col,
    title_text
) in enumerate(
    qc_umap_vars,
    start=1
):

    ax = sc.pl.embedding(

        adata_log,

        basis=
            "umap_malignant_unintegrated",

        color=
            col,

        cmap=
            "viridis",

        size=8,

        frameon=False,

        title=
            f"Malignant-only：{title_text}",

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        7.5,
        6.3
    )


    fig.tight_layout()


    save_publication_figure(

        fig,

        f"Step23B_0{i+5}_UMAP_{col}"

    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23B - CELL 42
# Expression program UMAP audit
# ============================================================

program_vars = [

    "mature_hepatocyte_score",
    "ductal_progenitor_score",
    "tumor_associated_expression_score"

]


program_vars = [

    x
    for x in program_vars
    if x in adata_log.obs.columns

]


for i, col in enumerate(
    program_vars,
    start=1
):

    ax = sc.pl.embedding(

        adata_log,

        basis=
            "umap_malignant_unintegrated",

        color=
            col,

        cmap=
            "viridis",

        size=8,

        frameon=False,

        title=
            col,

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        7.5,
        6.3
    )


    fig.tight_layout()


    save_publication_figure(

        fig,

        f"Step23B_{i+8:02d}_UMAP_{col}"

    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23B - CELL 43
# Global categorical silhouette
# ============================================================

X_pca = (

    adata_log.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :30
    ]
)


silhouette_rows = []


for variable in [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"

]:

    if variable not in adata_log.obs.columns:
        continue


    labels = (

        adata_log.obs[
            variable
        ]
        .astype(str)
        .values
    )


    if len(
        np.unique(
            labels
        )
    ) < 2:

        continue


    score = silhouette_score(

        X_pca,

        labels,

        metric="euclidean",

        sample_size=
            min(
                3000,
                len(
                    labels
                )
            ),

        random_state=42
    )


    silhouette_rows.append({

        "Variable":
            variable,

        "Silhouette":
            score,

        "N_categories":
            len(
                np.unique(
                    labels
                )
            )
    })


global_silhouette = pd.DataFrame(
    silhouette_rows
)


print("=" * 100)
print("GLOBAL PCA SILHOUETTE")
print("=" * 100)


display(
    global_silhouette
    .round(4)
)


global_silhouette.to_csv(

    RESULTS_DIR /
    "Step23B_Global_PCA_Silhouette.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23B - CELL 44
# Within-patient site silhouette
# ============================================================

MIN_CELLS_PER_SITE = 20


within_patient_site_rows = []


for patient in sorted(

    adata_log.obs[
        "patient"
    ]
    .astype(str)
    .unique()

):

    mask_patient = (

        adata_log.obs[
            "patient"
        ]
        .astype(str)
        ==
        patient
    )


    temp_obs = (

        adata_log.obs.loc[
            mask_patient
        ]
    )


    site_n = (

        temp_obs[
            "site"
        ]
        .astype(str)
        .value_counts()
    )


    eligible_sites = (

        site_n[
            site_n
            >=
            MIN_CELLS_PER_SITE
        ]
        .index
        .tolist()
    )


    if len(
        eligible_sites
    ) < 2:

        continue


    final_mask = (

        mask_patient

        &

        adata_log.obs[
            "site"
        ]
        .astype(str)
        .isin(
            eligible_sites
        )
    )


    row_idx = np.where(
        final_mask.values
    )[0]


    labels = (

        adata_log.obs.loc[
            final_mask,
            "site"
        ]
        .astype(str)
        .values
    )


    X_temp = (
        X_pca[
            row_idx,
            :
        ]
    )


    score = silhouette_score(

        X_temp,

        labels,

        metric="euclidean",

        sample_size=
            min(
                2000,
                len(
                    labels
                )
            ),

        random_state=42
    )


    within_patient_site_rows.append({

        "Patient":
            patient,

        "Sites":
            "|".join(
                eligible_sites
            ),

        "N_cells":
            len(
                labels
            ),

        "Silhouette_site":
            score
    })


within_patient_site = pd.DataFrame(
    within_patient_site_rows
)


print("=" * 100)
print("WITHIN-PATIENT SITE SEPARATION")
print("=" * 100)


display(
    within_patient_site
    .round(4)
)


within_patient_site.to_csv(

    RESULTS_DIR /
    "Step23B_WithinPatient_Site_Silhouette.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23B - CELL 45
# PCA × QC correlation
# ============================================================

qc_vars = [

    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt"

]


qc_vars = [

    x
    for x in qc_vars
    if x in adata_log.obs.columns

]


pc_qc_rows = []


for pc_idx in range(
    20
):

    pc_values = (
        X_pca[
            :,
            pc_idx
        ]
    )


    for var in qc_vars:

        values = (

            adata_log.obs[
                var
            ]
            .astype(float)
            .values
        )


        rho, p = spearmanr(
            pc_values,
            values
        )


        pc_qc_rows.append({

            "PC":
                pc_idx + 1,

            "QC_variable":
                var,

            "Spearman_rho":
                rho,

            "Abs_rho":
                abs(
                    rho
                ),

            "P_value":
                p
        })


pc_qc_correlations = pd.DataFrame(
    pc_qc_rows
)


pc_qc_correlations.to_csv(

    RESULTS_DIR /
    "Step23B_PC_QC_Correlations.csv",

    index=False,
    encoding="utf-8-sig"
)


qc_pc_summary = (

    pc_qc_correlations

    .sort_values(
        "Abs_rho",
        ascending=False
    )

    .groupby(
        "QC_variable",
        as_index=False
    )

    .first()
)


print("=" * 100)
print("MAX QC ASSOCIATION WITH FIRST 20 PCs")
print("=" * 100)


display(
    qc_pc_summary
    .round(4)
)

In [ ]:
# ============================================================
# STEP23B - CELL 46
# Categorical eta-squared for PCs
# ============================================================

def eta_squared(values, groups):

    values = np.asarray(
        values,
        dtype=float
    )

    groups = np.asarray(
        groups
    )


    grand_mean = np.mean(
        values
    )


    ss_total = np.sum(

        (
            values
            -
            grand_mean
        )
        ** 2

    )


    if ss_total == 0:

        return np.nan


    ss_between = 0.0


    for group in np.unique(groups):

        mask = (
            groups
            ==
            group
        )

        group_values = (
            values[
                mask
            ]
        )


        ss_between += (

            len(group_values)

            *

            (
                np.mean(group_values)
                -
                grand_mean
            )
            ** 2

        )


    return (
        ss_between
        /
        ss_total
    )


eta_rows = []


categorical_vars = [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"

]


for variable in categorical_vars:

    if variable not in adata_log.obs.columns:
        continue


    labels = (

        adata_log.obs[
            variable
        ]
        .astype(str)
        .values

    )


    for pc_idx in range(20):

        eta = eta_squared(

            X_pca[
                :,
                pc_idx
            ],

            labels

        )


        eta_rows.append({

            "Variable":
                variable,

            "PC":
                pc_idx + 1,

            "EtaSquared":
                eta

        })


pc_eta = pd.DataFrame(
    eta_rows
)


pc_eta.to_csv(

    RESULTS_DIR /
    "Step23B_PC_Categorical_EtaSquared.csv",

    index=False,
    encoding="utf-8-sig"

)


# ============================================================
# 每个变量最强的PC
# ============================================================

eta_summary = (

    pc_eta

    .sort_values(
        "EtaSquared",
        ascending=False
    )

    .groupby(
        "Variable",
        as_index=False
    )

    .first()

)


print("=" * 105)
print("MAX CATEGORICAL EFFECT ON FIRST 20 PCs")
print("=" * 105)


display(
    eta_summary
    .round(4)
)


print("\nTop categorical effects:")

display(

    pc_eta

    .sort_values(
        "EtaSquared",
        ascending=False
    )

    .head(20)

    .round(4)

)

In [ ]:
# ============================================================
# STEP23B - CELL 47
# kNN neighborhood purity
# ============================================================

from sklearn.neighbors import NearestNeighbors


K = 15


nn_model = NearestNeighbors(

    n_neighbors=
        K + 1,

    metric=
        "euclidean",

    n_jobs=-1

)


nn_model.fit(
    X_pca
)


_, neighbor_indices = (
    nn_model.kneighbors(
        X_pca
    )
)


# 第一个邻居通常是自己
neighbor_indices = (

    neighbor_indices[
        :,
        1:
    ]

)


purity_rows = []


for variable in [

    "patient",
    "cnv_leiden_r0_4",
    "site"

]:

    labels = (

        adata_log.obs[
            variable
        ]
        .astype(str)
        .values

    )


    neighbor_labels = (

        labels[
            neighbor_indices
        ]

    )


    same_fraction = np.mean(

        neighbor_labels

        ==

        labels[
            :,
            None
        ],

        axis=1

    )


    purity_col = (
        f"knn15_{variable}_purity"
    )


    adata_log.obs[
        purity_col
    ] = same_fraction


    purity_rows.append({

        "Variable":
            variable,

        "Mean_purity":
            float(
                np.mean(
                    same_fraction
                )
            ),

        "Median_purity":
            float(
                np.median(
                    same_fraction
                )
            ),

        "P25":
            float(
                np.quantile(
                    same_fraction,
                    0.25
                )
            ),

        "P75":
            float(
                np.quantile(
                    same_fraction,
                    0.75
                )
            ),

        "P90":
            float(
                np.quantile(
                    same_fraction,
                    0.90
                )
            )

    })


knn_purity_summary = pd.DataFrame(
    purity_rows
)


print("=" * 105)
print("LOCAL NEIGHBORHOOD PURITY")
print("=" * 105)


display(
    knn_purity_summary
    .round(4)
)


knn_purity_summary.to_csv(

    RESULTS_DIR /
    "Step23B_kNN15_Purity_Summary.csv",

    index=False,
    encoding="utf-8-sig"

)

In [ ]:
# ============================================================
# STEP23B - CELL 48
# kNN purity by patient
# ============================================================

patient_purity_rows = []


for patient in sorted(

    adata_log.obs[
        "patient"
    ]
    .astype(str)
    .unique()

):

    mask = (

        adata_log.obs[
            "patient"
        ]
        .astype(str)
        ==
        patient

    )


    patient_purity_rows.append({

        "Patient":
            patient,

        "N_cells":
            int(
                mask.sum()
            ),

        "Median_patient_purity":
            float(

                adata_log.obs.loc[
                    mask,
                    "knn15_patient_purity"
                ]
                .median()

            ),

        "Mean_patient_purity":
            float(

                adata_log.obs.loc[
                    mask,
                    "knn15_patient_purity"
                ]
                .mean()

            ),

        "Median_CNVclone_purity":
            float(

                adata_log.obs.loc[
                    mask,
                    "knn15_cnv_leiden_r0_4_purity"
                ]
                .median()

            ),

        "Median_site_purity":
            float(

                adata_log.obs.loc[
                    mask,
                    "knn15_site_purity"
                ]
                .median()

            )

    })


patient_purity = pd.DataFrame(
    patient_purity_rows
)


print("=" * 105)
print("LOCAL PURITY BY PATIENT")
print("=" * 105)


display(
    patient_purity
    .round(4)
)


patient_purity.to_csv(

    RESULTS_DIR /
    "Step23B_kNN15_Purity_ByPatient.csv",

    index=False,
    encoding="utf-8-sig"

)

In [ ]:
# ============================================================
# STEP23B - CELL 49
# PC categorical effect figure
# ============================================================

variables_to_plot = [

    x
    for x in [

        "patient",
        "cnv_leiden_r0_4",
        "site"

    ]

    if x in pc_eta[
        "Variable"
    ].unique()

]


fig, ax = plt.subplots(
    figsize=(8.0, 5.2)
)


for i, variable in enumerate(
    variables_to_plot
):

    temp = (

        pc_eta[
            pc_eta[
                "Variable"
            ]
            ==
            variable
        ]

        .sort_values(
            "PC"
        )

    )


    ax.plot(

        temp[
            "PC"
        ],

        temp[
            "EtaSquared"
        ],

        marker="o",

        markersize=3,

        linewidth=1.2,

        color=
            NPG_COLORS[
                i
                %
                len(
                    NPG_COLORS
                )
            ],

        label=
            variable

    )


ax.set_xlabel("Principal Component")
ax.set_ylabel(r"$\eta^2$")
ax.set_title(
    "Effects of Patient, CNV Clone, and Sampling Site on PCA Structure"
)

ax.set_xticks(
    np.arange(
        1,
        21
    )
)


ax.legend(
    frameon=False
)


ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23B_12_PC_Categorical_EtaSquared"

)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23B - CELL 50
# 保存Step23B primary unintegrated object
# ============================================================

STEP23B_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23B_MALIGNANT_UNINTEGRATED_PRIMARY.h5ad"

)


adata_log.write_h5ad(

    STEP23B_OUTPUT,

    compression="gzip"

)


print("=" * 105)
print("STEP23B PRIMARY OBJECT SAVED")
print("=" * 105)

print(
    STEP23B_OUTPUT
)

print(
    "\nShape:",
    adata_log.shape
)

print(
    "PCA:",
    adata_log.obsm[
        "X_pca_malignant_unintegrated"
    ].shape
)

print(
    "UMAP:",
    adata_log.obsm[
        "X_umap_malignant_unintegrated"
    ].shape
)

In [ ]:
# ============================================================
# STEP23B - CELL 51
# Step23B decision log
# ============================================================

step23b_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23B_Malignant_Transcriptional_Space",

    "Status":
        "PRIMARY_UNINTEGRATED_SPACE_COMPLETE",

    "N_cells":
        int(
            adata_log.n_obs
        ),

    "N_genes":
        int(
            adata_log.n_vars
        ),

    "Normalization":
        (
            "normalize_total(target_sum=1e4) + log1p"
        ),

    "HVG_method":
        "Seurat dispersion-based",

    "N_top_HVG":
        int(
            N_TOP_HVG
        ),

    "N_HVG_selected":
        int(
            adata_log.var[
                "highly_variable"
            ].sum()
        ),

    "N_HVG_used_for_PCA":
        int(
            adata_log.var[
                "highly_variable_for_pca"
            ].sum()
        ),

    "HVG_batch_key":
        "patient",

    "HVG_patient_min_cells":
        int(
            MIN_PATIENT_CELLS_FOR_HVG
        ),

    "HVG_patients":
        hvg_patients,

    "Patients_excluded_from_HVG_estimation":
        excluded_hvg_patients,

    "All_12406_cells_retained":
        True,

    "Mitochondrial_HVG_removed_from_PCA":
        True,

    "Ribosomal_HVG_removed_from_PCA":
        False,

    "Regression":
        "None",

    "Integration":
        "None",

    "PCA_components_calculated":
        50,

    "Neighbors_PCs":
        int(
            N_PCS_PRIMARY
        ),

    "Neighbors_k":
        int(
            N_NEIGHBORS
        ),

    "Primary_representation":
        "X_pca_malignant_unintegrated",

    "Primary_UMAP":
        "X_umap_malignant_unintegrated",

    "Integration_decision":
        "PENDING_REVIEW",

    "Reason_integration_not_preapplied":
        (
            "Patient identity is strongly confounded with CNV clone "
            "and partially with tissue site. These differences may "
            "represent genuine tumor biology, so patient effects must "
            "be quantified before considering integration."
        ),

    "Important_interpretation":
        (
            "The unintegrated malignant transcriptional space is a "
            "diagnostic primary representation. It will not be used "
            "to define universal malignant states until patient, "
            "CNV-clone, site and technical effects have been reviewed."
        ),

    "Output":
        str(
            STEP23B_OUTPUT
        ),

    "Next_step":
        (
            "Review patient/CNV/site/QC structure and determine "
            "whether patient-aware integration and/or balanced "
            "sensitivity analysis is required before malignant-state "
            "clustering."
        )

}


SUMMARY_PATH = (

    LOGS_DIR /
    "Step23B_Final_Summary.json"

)


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23b_summary,

        f,

        ensure_ascii=False,

        indent=4

    )


print("=" * 105)
print("STEP23B SUMMARY SAVED")
print("=" * 105)

print(
    SUMMARY_PATH
)

In [ ]:
# ============================================================
# STEP23B - CELL 52
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    (
        "Data",
        DATA_DIR
    ),

    (
        "Results",
        RESULTS_DIR
    ),

    (
        "Figures",
        FIGURES_DIR
    ),

    (
        "Logs",
        LOGS_DIR
    )

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(

                        path.stat().st_size

                        /

                        1024**2,

                        3

                    )

            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (

    RESULTS_DIR /
    "Step23B_Output_Manifest.csv"

)


manifest_df.to_csv(

    MANIFEST_PATH,

    index=False,

    encoding="utf-8-sig"

)


print("=" * 105)
print("STEP23B MANIFEST")
print("=" * 105)

print(
    "Files:",
    len(
        manifest_df
    )
)


display(
    manifest_df.tail(30)
)


print(
    "\nSaved:",
    MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP23B - CELL 53
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 115)
print("STEP23B FINAL REVIEW")
print("=" * 115)


# ============================================================
# 1. Dataset / HVG
# ============================================================

print("\n[1] Dataset / HVG")
print("-" * 115)

print(
    "Cells:",
    f"{adata_log.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_log.n_vars:,}"
)

print(
    "HVG:",
    int(
        adata_log.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "PCA genes:",
    int(
        adata_log.var[
            "highly_variable_for_pca"
        ].sum()
    )
)

print(
    "HVG patients:",
    hvg_patients
)

print(
    "Excluded from HVG fitting:",
    excluded_hvg_patients
)


# ============================================================
# 2. HVG cross-patient support
# ============================================================

print("\n[2] HVG跨患者支持")
print("-" * 115)


if (
    "hvg_support"
    in globals()
    and
    len(
        hvg_support
    ) > 0
):

    display(
        hvg_support
    )

else:

    print(
        "hvg_support unavailable or empty"
    )


# ============================================================
# 3. PCA variance
# ============================================================

print("\n[3] PCA解释方差")
print("-" * 115)


display(

    pca_variance
    .head(30)
    .round(5)

)


print(
    "\nCumulative variance:"
)


for n_pc in [

    10,
    20,
    30,
    50

]:

    if n_pc <= len(
        pca_variance
    ):

        print(

            f"PC1-{n_pc}:",

            round(

                float(

                    pca_variance.iloc[
                        n_pc - 1
                    ][
                        "CumulativeVariance"
                    ]

                ),

                4

            )

        )


# ============================================================
# 4. Silhouette
# ============================================================

print("\n[4] Global PCA silhouette")
print("-" * 115)


display(
    global_silhouette
    .round(4)
)


# ============================================================
# 5. Within-patient site
# ============================================================

print("\n[5] Within-patient Site silhouette")
print("-" * 115)


if len(
    within_patient_site
) > 0:

    display(
        within_patient_site
        .round(4)
    )

else:

    print(
        "No patient had >=2 eligible sites."
    )


# ============================================================
# 6. categorical eta squared
# ============================================================

print("\n[6] 最大类别效应 η²")
print("-" * 115)


display(
    eta_summary
    .round(4)
)


# ============================================================
# 7. QC effect
# ============================================================

print("\n[7] QC与前20PC最大相关")
print("-" * 115)


display(
    qc_pc_summary
    .round(4)
)


# ============================================================
# 8. local purity
# ============================================================

print("\n[8] kNN局部纯度")
print("-" * 115)


display(
    knn_purity_summary
    .round(4)
)


print("\n[9] kNN局部纯度 by patient")
print("-" * 115)


display(
    patient_purity
    .round(4)
)


# ============================================================
# 10. PCA top loading genes
# ============================================================

print("\n[10] PC1-PC5 Top loading genes")
print("-" * 115)


display(

    pca_loadings[
        pca_loadings[
            "PC"
        ]
        <=
        5
    ][
        [
            "PC",
            "Rank",
            "Gene",
            "Loading",
            "Is_Ribosomal",
            "Is_MT"
        ]
    ]

)


# ============================================================
# 11. Output files
# ============================================================

print("\n[11] Files")
print("-" * 115)


important_files = [

    LOGNORM_OUTPUT,

    STEP23B_OUTPUT,

    RESULTS_DIR /
    "Step23B_HVG_Full_Table.csv",

    RESULTS_DIR /
    "Step23B_PCA_Variance.csv",

    RESULTS_DIR /
    "Step23B_PCA_TopLoadings_PC1_20.csv",

    RESULTS_DIR /
    "Step23B_Global_PCA_Silhouette.csv",

    RESULTS_DIR /
    "Step23B_WithinPatient_Site_Silhouette.csv",

    RESULTS_DIR /
    "Step23B_PC_QC_Correlations.csv",

    RESULTS_DIR /
    "Step23B_PC_Categorical_EtaSquared.csv",

    RESULTS_DIR /
    "Step23B_kNN15_Purity_Summary.csv",

    RESULTS_DIR /
    "Step23B_kNN15_Purity_ByPatient.csv",

    SUMMARY_PATH,

    MANIFEST_PATH

]


for path in important_files:

    print(

        "[OK]"
        if path.exists()
        else "[MISSING]",

        path.name

    )


print("\n" + "=" * 115)

print(
    "STEP23B STATUS = PRIMARY UNINTEGRATED SPACE COMPLETE"
)

print(
    "INTEGRATION DECISION = PENDING REVIEW"
)

print(
    "DO NOT START MALIGNANT-STATE CLUSTERING BEFORE REVIEW."
)

print("=" * 115)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23C - CELL 54
# Load Step23B object and define discovery cohort
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import spearmanr
from scipy.spatial.distance import jensenshannon

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

from sklearn.neighbors import NearestNeighbors

import json
import sys
import gc
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


INPUT = (
    DATA_DIR /
    "GSE149614_STEP23B_MALIGNANT_UNINTEGRATED_PRIMARY.h5ad"
)

assert INPUT.exists()


adata_all = sc.read_h5ad(
    INPUT
)


# ============================================================
# Clean unused categorical levels
# ============================================================

for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"
]:

    if col in adata_all.obs.columns:

        adata_all.obs[col] = (
            adata_all.obs[col]
            .astype(str)
            .astype("category")
        )


# ============================================================
# Discovery patients
# ============================================================

DISCOVERY_PATIENTS = [
    "HCC04",
    "HCC07",
    "HCC08",
    "HCC09",
    "HCC10"
]


discovery_mask = (
    adata_all.obs["patient"]
    .astype(str)
    .isin(DISCOVERY_PATIENTS)
)


adata_disc = (
    adata_all[
        discovery_mask,
        :
    ]
    .copy()
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"
]:

    if col in adata_disc.obs.columns:

        adata_disc.obs[col] = (
            adata_disc.obs[col]
            .astype(str)
            .astype("category")
        )


excluded_cells = (
    adata_all.obs.loc[
        ~discovery_mask,
        [
            "patient",
            "sample",
            "site"
        ]
    ]
    .copy()
)


print("=" * 105)
print("STEP23C SHARED MALIGNANT-STATE DISCOVERY")
print("=" * 105)

print(
    "All malignant cells:",
    f"{adata_all.n_obs:,}"
)

print(
    "Discovery cells:",
    f"{adata_disc.n_obs:,}"
)

print(
    "Discovery patients:",
    DISCOVERY_PATIENTS
)

print(
    "Excluded from state discovery:",
    f"{len(excluded_cells):,}"
)

display(
    adata_disc.obs[
        "patient"
    ]
    .value_counts()
    .to_frame("N_cells")
)


excluded_cells.to_csv(
    RESULTS_DIR /
    "Step23C_Cells_Excluded_From_State_Discovery.csv",
    encoding="utf-8-sig"
)


assert adata_disc.n_obs == 12403

In [ ]:
# ============================================================
# STEP23C - CELL 55
# Publication plotting settings
# All figure text = English
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


NPG_COLORS = [
    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85",
    "#BDBDBD"
]


SITE_COLORS = {
    "Normal": "#4DBBD5",
    "Tumor": "#E64B35",
    "PVTT": "#00A087",
    "Lymph": "#3C5488"
}


def save_publication_figure(
    fig,
    filename
):

    png = (
        FIGURES_DIR /
        f"{filename}.png"
    )

    pdf = (
        FIGURES_DIR /
        f"{filename}.pdf"
    )

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23C - CELL 56
# Harmony environment check
# ============================================================

try:

    import harmonypy

    print(
        "harmonypy version:",
        getattr(
            harmonypy,
            "__version__",
            "unknown"
        )
    )

    print(
        "Harmony environment = PASS"
    )

except ImportError:

    print(
        "harmonypy is not installed."
    )

    print(
        "Run this command in a new cell:"
    )

    print(
        f'"{sys.executable}" -m pip install harmonypy'
    )

    raise

In [ ]:
# ============================================================
# STEP23C - CELL 57
# Prepare PCA representation for Harmony
# ============================================================

assert (
    "X_pca_malignant_unintegrated"
    in adata_disc.obsm
)


N_PCS_HARMONY = 30


adata_disc.obsm[
    "X_pca_for_harmony"
] = (

    adata_disc.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :N_PCS_HARMONY
    ]
    .copy()
)


print(
    "Harmony input:",
    adata_disc.obsm[
        "X_pca_for_harmony"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C - CELL 58
# Harmony integration
# ONLY correct patient
# ============================================================

print("=" * 105)
print("RUNNING PATIENT-AWARE HARMONY")
print("=" * 105)

np.random.seed(
    SEED
)


sc.external.pp.harmony_integrate(

    adata_disc,

    key=
        "patient",

    basis=
        "X_pca_for_harmony",

    adjusted_basis=
        "X_pca_harmony_patient"
)


print(
    "Harmony representation:",
    adata_disc.obsm[
        "X_pca_harmony_patient"
    ].shape
)


assert (
    adata_disc.obsm[
        "X_pca_harmony_patient"
    ].shape[0]
    ==
    adata_disc.n_obs
)

In [ ]:
# ============================================================
# STEP23C - CELL 59
# Save Harmony checkpoint immediately
# ============================================================

HARMONY_CHECKPOINT = (
    DATA_DIR /
    "GSE149614_STEP23C_HARMONY_PATIENT_CHECKPOINT.h5ad"
)


adata_disc.write_h5ad(
    HARMONY_CHECKPOINT,
    compression="gzip"
)


print(
    "Saved:"
)

print(
    HARMONY_CHECKPOINT
)

In [ ]:
# ============================================================
# STEP23C - CELL 60
# Harmony neighbor graph + UMAP
# ============================================================

N_NEIGHBORS = 15


sc.pp.neighbors(

    adata_disc,

    use_rep=
        "X_pca_harmony_patient",

    n_neighbors=
        N_NEIGHBORS,

    n_pcs=
        N_PCS_HARMONY,

    key_added=
        "neighbors_harmony_patient",

    random_state=
        SEED
)


sc.tl.umap(

    adata_disc,

    neighbors_key=
        "neighbors_harmony_patient",

    min_dist=
        0.5,

    random_state=
        SEED
)


adata_disc.obsm[
    "X_umap_harmony_patient"
] = (
    adata_disc.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "Harmony UMAP:",
    adata_disc.obsm[
        "X_umap_harmony_patient"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C - CELL 61
# Export Harmony UMAP coordinates
# ============================================================

harmony_umap = pd.DataFrame(

    adata_disc.obsm[
        "X_umap_harmony_patient"
    ],

    index=
        adata_disc.obs_names,

    columns=[
        "UMAP1",
        "UMAP2"
    ]
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt"
]:

    if col in adata_disc.obs.columns:

        harmony_umap[col] = (
            adata_disc.obs.loc[
                harmony_umap.index,
                col
            ]
            .values
        )


harmony_umap.index.name = "Cell"


harmony_umap.to_csv(

    RESULTS_DIR /
    "Step23C_Harmony_UMAP_Coordinates.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 62
# Harmony UMAP categorical audit
# ============================================================

plot_specs = [

    (
        "patient",
        "Malignant-cell Harmony Space by Patient",
        "Step23C_01_Harmony_UMAP_ByPatient"
    ),

    (
        "cnv_leiden_r0_4",
        "Malignant-cell Harmony Space by CNV Clone",
        "Step23C_02_Harmony_UMAP_ByCNVClone"
    ),

    (
        "site",
        "Malignant-cell Harmony Space by Sampling Site",
        "Step23C_03_Harmony_UMAP_BySite"
    )

]


for color_col, title_text, filename in plot_specs:

    # 保证没有unused categories
    adata_disc.obs[color_col] = (
        adata_disc.obs[color_col]
        .astype(str)
        .astype("category")
    )


    categories = (
        adata_disc.obs[color_col]
        .cat.categories
        .astype(str)
        .tolist()
    )


    if color_col == "site":

        palette = {
            x: SITE_COLORS.get(
                x,
                "#BDBDBD"
            )
            for x in categories
        }

    else:

        palette = {
            x:
                NPG_COLORS[
                    i %
                    len(NPG_COLORS)
                ]

            for i, x in enumerate(
                categories
            )
        }


    ax = sc.pl.embedding(

        adata_disc,

        basis=
            "umap_harmony_patient",

        color=
            color_col,

        palette=
            palette,

        size=
            8,

        alpha=
            0.78,

        frameon=
            False,

        legend_loc=
            "right margin",

        title=
            title_text,

        show=
            False
    )


    fig = ax.figure

    fig.set_size_inches(
        8.3,
        6.5
    )

    fig.tight_layout()


    save_publication_figure(
        fig,
        filename
    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23C - CELL 63
# Biological program audit in Harmony space
# ============================================================

program_specs = [

    (
        "mature_hepatocyte_score",
        "Mature Hepatocyte Program"
    ),

    (
        "ductal_progenitor_score",
        "Ductal/Progenitor Program"
    ),

    (
        "tumor_associated_expression_score",
        "Tumor-associated Expression Program"
    )

]


for i, (
    score,
    title_text
) in enumerate(
    program_specs,
    start=1
):

    if score not in adata_disc.obs.columns:
        continue


    ax = sc.pl.embedding(

        adata_disc,

        basis=
            "umap_harmony_patient",

        color=
            score,

        cmap=
            "viridis",

        size=
            8,

        frameon=
            False,

        title=
            title_text,

        show=
            False
    )


    fig = ax.figure

    fig.set_size_inches(
        7.5,
        6.3
    )

    fig.tight_layout()


    save_publication_figure(

        fig,

        f"Step23C_{i+3:02d}_Harmony_UMAP_"
        f"{score}"

    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23C - CELL 64
# Pre-vs-post Harmony silhouette audit
# ============================================================

X_pre = (
    adata_disc.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :30
    ]
)


X_post = (
    adata_disc.obsm[
        "X_pca_harmony_patient"
    ][
        :,
        :30
    ]
)


silhouette_rows = []


for variable in [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"

]:

    labels = (
        adata_disc.obs[
            variable
        ]
        .astype(str)
        .values
    )


    for representation, X in [

        (
            "Unintegrated",
            X_pre
        ),

        (
            "Harmony_patient",
            X_post
        )

    ]:

        score = silhouette_score(

            X,

            labels,

            metric=
                "euclidean",

            sample_size=
                min(
                    3000,
                    len(labels)
                ),

            random_state=
                SEED
        )


        silhouette_rows.append({

            "Representation":
                representation,

            "Variable":
                variable,

            "Silhouette":
                score,

            "N_categories":
                len(
                    np.unique(
                        labels
                    )
                )
        })


integration_silhouette = pd.DataFrame(
    silhouette_rows
)


print("=" * 105)
print("PRE / POST HARMONY SILHOUETTE")
print("=" * 105)


display(

    integration_silhouette
    .pivot(
        index="Variable",
        columns="Representation",
        values="Silhouette"
    )
    .round(4)

)


integration_silhouette.to_csv(

    RESULTS_DIR /
    "Step23C_Integration_Silhouette.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 65
# Pre-vs-post local neighborhood purity
# ============================================================

K = 15


def calculate_knn_purity(
    X,
    labels,
    k=15
):

    model = NearestNeighbors(

        n_neighbors=
            k + 1,

        metric=
            "euclidean",

        n_jobs=
            -1
    )


    model.fit(
        X
    )


    _, idx = (
        model.kneighbors(
            X
        )
    )


    idx = (
        idx[
            :,
            1:
        ]
    )


    labels = np.asarray(
        labels
    )


    neighbor_labels = (
        labels[
            idx
        ]
    )


    purity = np.mean(

        neighbor_labels

        ==

        labels[
            :,
            None
        ],

        axis=1
    )


    return (
        purity,
        idx
    )


purity_rows = []

neighbor_indices_store = {}


for representation, X in [

    (
        "Unintegrated",
        X_pre
    ),

    (
        "Harmony_patient",
        X_post
    )

]:

    for variable in [

        "patient",
        "cnv_leiden_r0_4",
        "site"

    ]:

        labels = (
            adata_disc.obs[
                variable
            ]
            .astype(str)
            .values
        )


        purity, idx = (
            calculate_knn_purity(
                X,
                labels,
                K
            )
        )


        neighbor_indices_store[
            (
                representation,
                variable
            )
        ] = idx


        # random-mixing expectation
        proportions = (

            pd.Series(labels)
            .value_counts(
                normalize=True
            )
            .values
        )


        expected_random_purity = float(
            np.sum(
                proportions ** 2
            )
        )


        purity_rows.append({

            "Representation":
                representation,

            "Variable":
                variable,

            "Mean_purity":
                float(
                    np.mean(
                        purity
                    )
                ),

            "Median_purity":
                float(
                    np.median(
                        purity
                    )
                ),

            "P25":
                float(
                    np.quantile(
                        purity,
                        0.25
                    )
                ),

            "P75":
                float(
                    np.quantile(
                        purity,
                        0.75
                    )
                ),

            "Expected_random_purity":
                expected_random_purity,

            "Observed_to_random_ratio":
                float(
                    np.mean(
                        purity
                    )
                    /
                    expected_random_purity
                )
        })


integration_purity = pd.DataFrame(
    purity_rows
)


print("=" * 105)
print("PRE / POST HARMONY kNN PURITY")
print("=" * 105)


display(
    integration_purity
    .round(4)
)


integration_purity.to_csv(

    RESULTS_DIR /
    "Step23C_Integration_kNN15_Purity.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 66
# Within-patient site preservation
# ============================================================

MIN_SITE_CELLS = 20


site_preservation_rows = []


for patient in DISCOVERY_PATIENTS:

    patient_mask = (
        adata_disc.obs[
            "patient"
        ]
        .astype(str)
        ==
        patient
    )


    site_counts = (

        adata_disc.obs.loc[
            patient_mask,
            "site"
        ]
        .astype(str)
        .value_counts()
    )


    eligible_sites = (

        site_counts[
            site_counts
            >=
            MIN_SITE_CELLS
        ]
        .index
        .tolist()
    )


    if len(
        eligible_sites
    ) < 2:

        continue


    mask = (

        patient_mask

        &

        adata_disc.obs[
            "site"
        ]
        .astype(str)
        .isin(
            eligible_sites
        )
    )


    idx = np.where(
        mask.values
    )[0]


    labels = (
        adata_disc.obs.loc[
            mask,
            "site"
        ]
        .astype(str)
        .values
    )


    for representation, X in [

        (
            "Unintegrated",
            X_pre
        ),

        (
            "Harmony_patient",
            X_post
        )

    ]:

        score = silhouette_score(

            X[
                idx,
                :
            ],

            labels,

            metric=
                "euclidean",

            sample_size=
                min(
                    2000,
                    len(labels)
                ),

            random_state=
                SEED
        )


        site_preservation_rows.append({

            "Patient":
                patient,

            "Sites":
                "|".join(
                    eligible_sites
                ),

            "Representation":
                representation,

            "N_cells":
                len(labels),

            "Site_silhouette":
                score
        })


site_preservation = pd.DataFrame(
    site_preservation_rows
)


print("=" * 105)
print("WITHIN-PATIENT SITE PRESERVATION")
print("=" * 105)


display(

    site_preservation
    .pivot_table(

        index=[
            "Patient",
            "Sites"
        ],

        columns=
            "Representation",

        values=
            "Site_silhouette"
    )
    .round(4)

)


site_preservation.to_csv(

    RESULTS_DIR /
    "Step23C_WithinPatient_Site_Preservation.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 67
# Neighborhood overlap pre/post Harmony
# ============================================================

def knn_indices(
    X,
    k=15
):

    model = NearestNeighbors(

        n_neighbors=
            k + 1,

        metric=
            "euclidean",

        n_jobs=
            -1
    )


    model.fit(
        X
    )


    _, idx = (
        model.kneighbors(
            X
        )
    )


    return (
        idx[
            :,
            1:
        ]
    )


idx_pre = knn_indices(
    X_pre,
    K
)

idx_post = knn_indices(
    X_post,
    K
)


jaccard_values = []


for i in range(
    adata_disc.n_obs
):

    a = set(
        idx_pre[
            i
        ]
    )

    b = set(
        idx_post[
            i
        ]
    )


    union = (
        a | b
    )


    intersection = (
        a & b
    )


    jaccard_values.append(

        len(
            intersection
        )
        /
        len(
            union
        )

    )


jaccard_values = np.asarray(
    jaccard_values
)


neighborhood_overlap_summary = pd.DataFrame({

    "Metric": [
        "Mean",
        "Median",
        "P25",
        "P75"
    ],

    "kNN_Jaccard": [

        np.mean(
            jaccard_values
        ),

        np.median(
            jaccard_values
        ),

        np.quantile(
            jaccard_values,
            0.25
        ),

        np.quantile(
            jaccard_values,
            0.75
        )
    ]
})


display(
    neighborhood_overlap_summary
    .round(4)
)


neighborhood_overlap_summary.to_csv(

    RESULTS_DIR /
    "Step23C_PrePost_Neighborhood_Jaccard.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 68
# Multi-resolution Leiden clustering
# ============================================================

STATE_RESOLUTIONS = [
    0.2,
    0.4,
    0.6,
    0.8,
    1.0
]


resolution_rows = []


for res in STATE_RESOLUTIONS:

    key = (
        "state_leiden_r"
        +
        str(res)
        .replace(
            ".",
            "_"
        )
    )


    print(
        f"Running Leiden resolution = {res}"
    )


    sc.tl.leiden(

        adata_disc,

        resolution=
            res,

        neighbors_key=
            "neighbors_harmony_patient",

        key_added=
            key,

        random_state=
            SEED,

        flavor=
            "igraph",

        n_iterations=
            2,

        directed=
            False
    )


    sizes = (
        adata_disc.obs[
            key
        ]
        .value_counts()
    )


    resolution_rows.append({

        "Resolution":
            res,

        "Cluster_column":
            key,

        "N_clusters":
            len(
                sizes
            ),

        "Smallest_cluster":
            int(
                sizes.min()
            ),

        "Largest_cluster":
            int(
                sizes.max()
            ),

        "Median_cluster_size":
            float(
                sizes.median()
            ),

        "Clusters_lt_50":
            int(
                (
                    sizes < 50
                ).sum()
            ),

        "Clusters_lt_100":
            int(
                (
                    sizes < 100
                ).sum()
            )
    })


state_resolution_summary = pd.DataFrame(
    resolution_rows
)


print("=" * 105)
print("STATE CLUSTERING RESOLUTION SUMMARY")
print("=" * 105)


display(
    state_resolution_summary
)


state_resolution_summary.to_csv(

    RESULTS_DIR /
    "Step23C_State_Resolution_Summary.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 69
# Adjacent-resolution ARI
# ============================================================

ari_rows = []


for r1, r2 in [

    (0.2, 0.4),
    (0.4, 0.6),
    (0.6, 0.8),
    (0.8, 1.0)

]:

    key1 = (
        "state_leiden_r"
        +
        str(r1)
        .replace(
            ".",
            "_"
        )
    )


    key2 = (
        "state_leiden_r"
        +
        str(r2)
        .replace(
            ".",
            "_"
        )
    )


    ari = adjusted_rand_score(

        adata_disc.obs[
            key1
        ]
        .astype(str),

        adata_disc.obs[
            key2
        ]
        .astype(str)
    )


    ari_rows.append({

        "Resolution_1":
            r1,

        "Resolution_2":
            r2,

        "ARI":
            ari
    })


state_resolution_ari = pd.DataFrame(
    ari_rows
)


display(
    state_resolution_ari
    .round(4)
)


state_resolution_ari.to_csv(

    RESULTS_DIR /
    "Step23C_State_Resolution_ARI.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C - CELL 70
# Cross-patient support audit for every resolution
# ============================================================

DISCOVERY_PATIENT_ORDER = (
    DISCOVERY_PATIENTS
)


overall_patient_distribution = (

    adata_disc.obs[
        "patient"
    ]
    .astype(str)
    .value_counts(
        normalize=True
    )
    .reindex(
        DISCOVERY_PATIENT_ORDER,
        fill_value=0
    )
    .values
)


def normalized_entropy(
    counts
):

    counts = np.asarray(
        counts,
        dtype=float
    )


    counts = (
        counts[
            counts > 0
        ]
    )


    if len(counts) <= 1:

        return 0.0


    p = (
        counts /
        counts.sum()
    )


    entropy = (
        -np.sum(
            p *
            np.log(
                p
            )
        )
    )


    max_entropy = np.log(
        len(
            DISCOVERY_PATIENT_ORDER
        )
    )


    return float(
        entropy /
        max_entropy
    )


composition_tables = []


for res in STATE_RESOLUTIONS:

    cluster_key = (
        "state_leiden_r"
        +
        str(res)
        .replace(
            ".",
            "_"
        )
    )


    clusters = sorted(

        adata_disc.obs[
            cluster_key
        ]
        .astype(str)
        .unique(),

        key=lambda x:
            int(x)
    )


    rows = []


    for cluster in clusters:

        mask = (
            adata_disc.obs[
                cluster_key
            ]
            .astype(str)
            ==
            cluster
        )


        temp = (
            adata_disc.obs.loc[
                mask
            ]
        )


        n = len(
            temp
        )


        patient_counts = (

            temp[
                "patient"
            ]
            .astype(str)
            .value_counts()
            .reindex(
                DISCOVERY_PATIENT_ORDER,
                fill_value=0
            )

        )


        patient_fraction = (
            patient_counts
            /
            patient_counts.sum()
        )


        dominant_patient = (
            patient_counts.idxmax()
        )


        dominant_patient_pct = (
            patient_counts.max()
            /
            n
            *
            100
        )


        n_patients_ge20 = int(
            (
                patient_counts >= 20
            ).sum()
        )


        n_patients_ge50 = int(
            (
                patient_counts >= 50
            ).sum()
        )


        js_distance = float(

            jensenshannon(

                patient_fraction.values,

                overall_patient_distribution,

                base=2

            )

        )


        site_counts = (
            temp[
                "site"
            ]
            .astype(str)
            .value_counts()
        )


        cnv_counts = (
            temp[
                "cnv_leiden_r0_4"
            ]
            .astype(str)
            .value_counts()
        )


        row = {

            "Resolution":
                res,

            "State_cluster":
                cluster,

            "N_cells":
                n,

            "N_patients_present":
                int(
                    (
                        patient_counts > 0
                    ).sum()
                ),

            "N_patients_ge20":
                n_patients_ge20,

            "N_patients_ge50":
                n_patients_ge50,

            "Dominant_patient":
                dominant_patient,

            "Dominant_patient_pct":
                dominant_patient_pct,

            "Patient_entropy_normalized":
                normalized_entropy(
                    patient_counts.values
                ),

            "Patient_JSD_from_overall":
                js_distance,

            "Dominant_CNV_clone":
                cnv_counts.index[0],

            "Dominant_CNV_clone_pct":
                (
                    cnv_counts.iloc[0]
                    /
                    n
                    *
                    100
                ),

            "Pct_Tumor":
                (
                    site_counts.get(
                        "Tumor",
                        0
                    )
                    /
                    n
                    *
                    100
                ),

            "Pct_PVTT":
                (
                    site_counts.get(
                        "PVTT",
                        0
                    )
                    /
                    n
                    *
                    100
                ),

            "Pct_Lymph":
                (
                    site_counts.get(
                        "Lymph",
                        0
                    )
                    /
                    n
                    *
                    100
                ),

            "Pct_Normal":
                (
                    site_counts.get(
                        "Normal",
                        0
                    )
                    /
                    n
                    *
                    100
                )
        }


        # 每个患者的绝对细胞数
        for patient in DISCOVERY_PATIENT_ORDER:

            row[
                f"N_{patient}"
            ] = int(
                patient_counts[
                    patient
                ]
            )


        rows.append(
            row
        )


    temp_table = pd.DataFrame(
        rows
    )


    composition_tables.append(
        temp_table
    )


state_cluster_composition = pd.concat(

    composition_tables,

    ignore_index=True
)


state_cluster_composition.to_csv(

    RESULTS_DIR /
    "Step23C_State_Cluster_Composition_AllResolutions.csv",

    index=False,
    encoding="utf-8-sig"
)


for res in STATE_RESOLUTIONS:

    print(
        "\n" +
        "=" * 105
    )

    print(
        f"Resolution = {res}"
    )

    print(
        "=" * 105
    )


    display(

        state_cluster_composition[
            state_cluster_composition[
                "Resolution"
            ]
            ==
            res
        ]
        .round(3)

    )

In [ ]:
# ============================================================
# STEP23C - CELL 71
# Cluster x patient tables
# ============================================================

for res in STATE_RESOLUTIONS:

    key = (
        "state_leiden_r"
        +
        str(res)
        .replace(
            ".",
            "_"
        )
    )


    counts = pd.crosstab(

        adata_disc.obs[
            key
        ],

        adata_disc.obs[
            "patient"
        ]
    )


    row_pct = (

        counts
        .div(
            counts.sum(
                axis=1
            ),
            axis=0
        )
        *
        100

    )


    counts.to_csv(

        RESULTS_DIR /
        (
            "Step23C_StateCluster_x_Patient_"
            f"r{str(res).replace('.', '_')}_Counts.csv"
        ),

        encoding="utf-8-sig"
    )


    row_pct.to_csv(

        RESULTS_DIR /
        (
            "Step23C_StateCluster_x_Patient_"
            f"r{str(res).replace('.', '_')}_Percent.csv"
        ),

        encoding="utf-8-sig"
    )

In [ ]:
# ============================================================
# STEP23C - CELL 72
# Multi-resolution cluster UMAPs
# ============================================================

for res in STATE_RESOLUTIONS:

    key = (
        "state_leiden_r"
        +
        str(res)
        .replace(
            ".",
            "_"
        )
    )


    ax = sc.pl.embedding(

        adata_disc,

        basis=
            "umap_harmony_patient",

        color=
            key,

        size=
            8,

        alpha=
            0.82,

        frameon=
            False,

        legend_loc=
            "on data",

        legend_fontsize=
            8,

        title=
            f"Malignant State Candidates (Leiden r={res})",

        show=
            False
    )


    fig = ax.figure

    fig.set_size_inches(
        7.7,
        6.5
    )

    fig.tight_layout()


    save_publication_figure(

        fig,

        (
            "Step23C_StateCandidates_Leiden_"
            f"r{str(res).replace('.', '_')}"
        )

    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23C - CELL 73
# Integration audit figure
# ============================================================

patient_audit = (

    integration_purity[
        integration_purity[
            "Variable"
        ]
        ==
        "patient"
    ]
    .copy()
)


fig, ax = plt.subplots(
    figsize=(6.3, 5.0)
)


x = np.arange(
    len(
        patient_audit
    )
)


bars = ax.bar(

    x,

    patient_audit[
        "Mean_purity"
    ]
)


ax.axhline(

    patient_audit[
        "Expected_random_purity"
    ]
    .iloc[0],

    linestyle="--",

    linewidth=1.2,

    label=
        "Random-mixing expectation"
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    patient_audit[
        "Representation"
    ]
)


ax.set_ylabel(
    "Mean Same-patient kNN Purity"
)

ax.set_title(
    "Patient Mixing Before and After Harmony Integration"
)


ax.legend(
    frameon=False
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23C_07_PatientMixing_PrePostHarmony"

)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23C - CELL 74
# Save clustered Harmony object
# ============================================================

STEP23C_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23C_HARMONY_STATE_CANDIDATES.h5ad"

)


adata_disc.write_h5ad(

    STEP23C_OUTPUT,

    compression="gzip"
)


print("=" * 105)
print("STEP23C OBJECT SAVED")
print("=" * 105)

print(
    STEP23C_OUTPUT
)

print(
    "Shape:",
    adata_disc.shape
)

In [ ]:
# ============================================================
# STEP23C - CELL 75
# Decision log
# ============================================================

step23c_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23C_PatientAware_State_Discovery",

    "Status":
        "HARMONY_AND_MULTIRESOLUTION_CLUSTERING_COMPLETE",

    "Input":
        str(
            INPUT
        ),

    "All_high_confidence_malignant_cells":
        int(
            adata_all.n_obs
        ),

    "State_discovery_cells":
        int(
            adata_disc.n_obs
        ),

    "Discovery_patients":
        DISCOVERY_PATIENTS,

    "Cells_excluded_from_state_discovery":
        int(
            len(
                excluded_cells
            )
        ),

    "Integration_method":
        "Harmony",

    "Harmony_batch_variable":
        "patient",

    "Variables_not_corrected": [
        "site",
        "CNV clone",
        "sample"
    ],

    "Harmony_input":
        "First 30 patient-aware malignant-only unintegrated PCs",

    "Important_rule":
        (
            "Harmony is used only for low-dimensional representation, "
            "neighbor graph, UMAP and candidate clustering. "
            "Gene expression and downstream differential expression "
            "remain based on the original log-normalized expression matrix."
        ),

    "State_resolutions_tested":
        STATE_RESOLUTIONS,

    "Primary_state_resolution":
        "PENDING_REVIEW",

    "Final_state_annotation":
        "NOT_PERFORMED",

    "Next":
        (
            "Select a stable, cross-patient-supported candidate "
            "resolution before marker analysis and biological annotation."
        ),

    "Output":
        str(
            STEP23C_OUTPUT
        )
}


SUMMARY_PATH_C = (

    LOGS_DIR /
    "Step23C_Final_Summary.json"

)


with open(

    SUMMARY_PATH_C,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23c_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_PATH_C
)

In [ ]:
# ============================================================
# STEP23C - CELL 76
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    (
        "Data",
        DATA_DIR
    ),

    (
        "Results",
        RESULTS_DIR
    ),

    (
        "Figures",
        FIGURES_DIR
    ),

    (
        "Logs",
        LOGS_DIR
    )

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH_C = (

    RESULTS_DIR /
    "Step23C_Output_Manifest.csv"

)


manifest_df.to_csv(

    MANIFEST_PATH_C,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Manifest saved:"
)

print(
    MANIFEST_PATH_C
)

In [ ]:
# ============================================================
# STEP23C - CELL 77
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 118)
print("STEP23C FINAL REVIEW")
print("=" * 118)


# ============================================================
# 1. Discovery cohort
# ============================================================

print("\n[1] Discovery cohort")
print("-" * 118)

print(
    "Cells:",
    f"{adata_disc.n_obs:,}"
)

print(
    "Patients:",
    DISCOVERY_PATIENTS
)

print(
    "Excluded cells:",
    len(
        excluded_cells
    )
)


# ============================================================
# 2. Integration silhouette
# ============================================================

print("\n[2] Pre/Post Harmony silhouette")
print("-" * 118)

display(

    integration_silhouette
    .pivot(
        index="Variable",
        columns="Representation",
        values="Silhouette"
    )
    .round(4)

)


# ============================================================
# 3. kNN purity
# ============================================================

print("\n[3] Pre/Post Harmony kNN purity")
print("-" * 118)

display(
    integration_purity
    .round(4)
)


# ============================================================
# 4. Site preservation
# ============================================================

print("\n[4] Within-patient site preservation")
print("-" * 118)

display(

    site_preservation
    .pivot_table(

        index=[
            "Patient",
            "Sites"
        ],

        columns=
            "Representation",

        values=
            "Site_silhouette"
    )
    .round(4)

)


# ============================================================
# 5. Neighborhood overlap
# ============================================================

print("\n[5] Pre/Post neighborhood overlap")
print("-" * 118)

display(
    neighborhood_overlap_summary
    .round(4)
)


# ============================================================
# 6. Resolution summary
# ============================================================

print("\n[6] Leiden resolution summary")
print("-" * 118)

display(
    state_resolution_summary
)


# ============================================================
# 7. Resolution stability
# ============================================================

print("\n[7] Adjacent-resolution ARI")
print("-" * 118)

display(
    state_resolution_ari
    .round(4)
)


# ============================================================
# 8. Key candidate resolutions
# ============================================================

for res in [
    0.4,
    0.6,
    0.8
]:

    print(
        f"\n[8] Cluster composition — resolution {res}"
    )

    print(
        "-" * 118
    )

    display(

        state_cluster_composition[
            state_cluster_composition[
                "Resolution"
            ]
            ==
            res
        ]
        .round(3)

    )


# ============================================================
# 9. Files
# ============================================================

print("\n[9] Files")
print("-" * 118)


important_files = [

    HARMONY_CHECKPOINT,

    STEP23C_OUTPUT,

    RESULTS_DIR /
    "Step23C_Integration_Silhouette.csv",

    RESULTS_DIR /
    "Step23C_Integration_kNN15_Purity.csv",

    RESULTS_DIR /
    "Step23C_WithinPatient_Site_Preservation.csv",

    RESULTS_DIR /
    "Step23C_PrePost_Neighborhood_Jaccard.csv",

    RESULTS_DIR /
    "Step23C_State_Resolution_Summary.csv",

    RESULTS_DIR /
    "Step23C_State_Resolution_ARI.csv",

    RESULTS_DIR /
    "Step23C_State_Cluster_Composition_AllResolutions.csv",

    SUMMARY_PATH_C,

    MANIFEST_PATH_C
]


for path in important_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 118)

print(
    "STEP23C STATUS = HARMONY + STATE CANDIDATES COMPLETE"
)

print(
    "PRIMARY STATE RESOLUTION = PENDING REVIEW"
)

print(
    "DO NOT NAME OR INTERPRET STATES YET."
)

print("=" * 118)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23C-2 - CELL 78
# Load primary Harmony state-candidate object
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

import json
import warnings
import gc

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


INPUT = (
    DATA_DIR /
    "GSE149614_STEP23C_HARMONY_STATE_CANDIDATES.h5ad"
)


assert INPUT.exists()


adata_state = sc.read_h5ad(
    INPUT
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    "state_leiden_r0_4"
]:

    if col in adata_state.obs.columns:

        adata_state.obs[col] = (
            adata_state.obs[col]
            .astype(str)
            .astype("category")
        )


print("=" * 105)
print("STEP23C-2 STABILITY ANALYSIS")
print("=" * 105)

print(
    "Shape:",
    adata_state.shape
)

print(
    "Patients:",
    adata_state.obs[
        "patient"
    ].nunique()
)

print(
    "Primary r0.4 clusters:",
    adata_state.obs[
        "state_leiden_r0_4"
    ].nunique()
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 79
# Publication plotting settings
# All figure text must be English
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


NPG_COLORS = [
    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85",
    "#BDBDBD"
]


def save_publication_figure(
    fig,
    filename
):

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23C-2 - CELL 80
# Build PCA representation excluding PC4
# ============================================================

X_original_pca = (
    adata_state.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :30
    ]
)


pc_indices_without_pc4 = [

    i
    for i in range(30)
    if i != 3
]


X_no_pc4 = (
    X_original_pca[
        :,
        pc_indices_without_pc4
    ]
    .copy()
)


adata_state.obsm[
    "X_pca_for_harmony_noPC4"
] = X_no_pc4


print(
    "Original PCs:",
    X_original_pca.shape
)

print(
    "Without PC4:",
    X_no_pc4.shape
)

print(
    "PCs retained:",
    [
        i + 1
        for i in pc_indices_without_pc4
    ]
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 81
# Harmony sensitivity excluding PC4
# ============================================================

np.random.seed(
    SEED
)


sc.external.pp.harmony_integrate(

    adata_state,

    key=
        "patient",

    basis=
        "X_pca_for_harmony_noPC4",

    adjusted_basis=
        "X_pca_harmony_patient_noPC4"
)


print(
    "Harmony no-PC4 representation:",
    adata_state.obsm[
        "X_pca_harmony_patient_noPC4"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 82
# No-PC4 neighbors + UMAP
# ============================================================

sc.pp.neighbors(

    adata_state,

    use_rep=
        "X_pca_harmony_patient_noPC4",

    n_neighbors=
        15,

    n_pcs=
        29,

    key_added=
        "neighbors_harmony_noPC4",

    random_state=
        SEED
)


sc.tl.umap(

    adata_state,

    neighbors_key=
        "neighbors_harmony_noPC4",

    min_dist=
        0.5,

    random_state=
        SEED
)


adata_state.obsm[
    "X_umap_harmony_noPC4"
] = (
    adata_state.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "No-PC4 UMAP:",
    adata_state.obsm[
        "X_umap_harmony_noPC4"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 83
# Leiden r0.4 without PC4
# ============================================================

sc.tl.leiden(

    adata_state,

    resolution=
        0.4,

    neighbors_key=
        "neighbors_harmony_noPC4",

    key_added=
        "state_noPC4_leiden_r0_4",

    random_state=
        SEED,

    flavor=
        "igraph",

    n_iterations=
        2,

    directed=
        False
)


print(
    "No-PC4 clusters:",
    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ].nunique()
)


display(
    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 84
# Primary vs no-PC4 cluster stability
# ============================================================

ari_primary_noPC4 = adjusted_rand_score(

    adata_state.obs[
        "state_leiden_r0_4"
    ].astype(str),

    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ].astype(str)
)


print("=" * 100)
print("PC4-EXCLUSION SENSITIVITY")
print("=" * 100)

print(
    "Primary r0.4 clusters:",
    adata_state.obs[
        "state_leiden_r0_4"
    ].nunique()
)

print(
    "No-PC4 r0.4 clusters:",
    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_primary_noPC4,
        4
    )
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 85
# Primary vs no-PC4 contingency table
# ============================================================

primary_noPC4_counts = pd.crosstab(

    adata_state.obs[
        "state_leiden_r0_4"
    ],

    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ]
)


primary_noPC4_rowpct = (

    primary_noPC4_counts
    .div(
        primary_noPC4_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


print(
    "Counts:"
)

display(
    primary_noPC4_counts
)


print(
    "\nRow percentages:"
)

display(
    primary_noPC4_rowpct
    .round(2)
)


primary_noPC4_counts.to_csv(

    RESULTS_DIR /
    "Step23C2_Primary_vs_NoPC4_Counts.csv",

    encoding="utf-8-sig"
)


primary_noPC4_rowpct.to_csv(

    RESULTS_DIR /
    "Step23C2_Primary_vs_NoPC4_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 86
# No-PC4 sensitivity UMAP
# ============================================================

ax = sc.pl.embedding(

    adata_state,

    basis=
        "umap_harmony_noPC4",

    color=
        "state_noPC4_leiden_r0_4",

    size=
        8,

    alpha=
        0.82,

    frameon=
        False,

    legend_loc=
        "on data",

    legend_fontsize=
        8,

    title=
        "State Candidates After Excluding PC4",

    show=
        False
)


fig = ax.figure

fig.set_size_inches(
    7.7,
    6.5
)

fig.tight_layout()


save_publication_figure(

    fig,

    "Step23C2_01_StateCandidates_NoPC4_r0_4"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 87
# Patient-capped sensitivity cohort
# ============================================================

PATIENT_CAP = 1500


rng = np.random.default_rng(
    SEED
)


selected_cells = []


for patient in [

    "HCC04",
    "HCC07",
    "HCC08",
    "HCC09",
    "HCC10"

]:

    patient_cells = (

        adata_state.obs_names[
            adata_state.obs[
                "patient"
            ]
            .astype(str)
            ==
            patient
        ]
        .tolist()
    )


    n_take = min(

        PATIENT_CAP,

        len(
            patient_cells
        )
    )


    chosen = rng.choice(

        patient_cells,

        size=
            n_take,

        replace=
            False
    )


    selected_cells.extend(
        chosen.tolist()
    )


adata_cap = (

    adata_state[
        selected_cells,
        :
    ]
    .copy()
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    "state_leiden_r0_4"
]:

    if col in adata_cap.obs.columns:

        adata_cap.obs[col] = (
            adata_cap.obs[col]
            .astype(str)
            .astype("category")
        )


print("=" * 100)
print("PATIENT-CAPPED SENSITIVITY")
print("=" * 100)

print(
    "Cells:",
    f"{adata_cap.n_obs:,}"
)


display(
    adata_cap.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 88
# Save capped cell manifest
# ============================================================

capped_manifest = (

    adata_cap.obs[
        [
            "patient",
            "sample",
            "site",
            "cnv_leiden_r0_4",
            "state_leiden_r0_4"
        ]
    ]
    .copy()
)


capped_manifest.index.name = (
    "Cell"
)


CAPPED_MANIFEST_PATH = (

    RESULTS_DIR /
    "Step23C2_PatientCapped_CellManifest.csv"
)


capped_manifest.to_csv(

    CAPPED_MANIFEST_PATH,

    encoding="utf-8-sig"
)


print(
    "Saved:",
    CAPPED_MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 89
# Re-run Harmony in patient-capped cohort
# ============================================================

adata_cap.obsm[
    "X_pca_for_harmony_capped"
] = (

    adata_cap.obsm[
        "X_pca_malignant_unintegrated"
    ][
        :,
        :30
    ]
    .copy()
)


np.random.seed(
    SEED
)


sc.external.pp.harmony_integrate(

    adata_cap,

    key=
        "patient",

    basis=
        "X_pca_for_harmony_capped",

    adjusted_basis=
        "X_pca_harmony_patient_capped"
)


print(
    "Capped Harmony:",
    adata_cap.obsm[
        "X_pca_harmony_patient_capped"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 90
# Capped neighbors + UMAP
# ============================================================

sc.pp.neighbors(

    adata_cap,

    use_rep=
        "X_pca_harmony_patient_capped",

    n_neighbors=
        15,

    n_pcs=
        30,

    key_added=
        "neighbors_harmony_capped",

    random_state=
        SEED
)


sc.tl.umap(

    adata_cap,

    neighbors_key=
        "neighbors_harmony_capped",

    min_dist=
        0.5,

    random_state=
        SEED
)


adata_cap.obsm[
    "X_umap_harmony_capped"
] = (
    adata_cap.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "Capped UMAP:",
    adata_cap.obsm[
        "X_umap_harmony_capped"
    ].shape
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 91
# Capped Leiden r0.4
# ============================================================

sc.tl.leiden(

    adata_cap,

    resolution=
        0.4,

    neighbors_key=
        "neighbors_harmony_capped",

    key_added=
        "state_capped_leiden_r0_4",

    random_state=
        SEED,

    flavor=
        "igraph",

    n_iterations=
        2,

    directed=
        False
)


print(
    "Capped r0.4 clusters:",
    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ].nunique()
)


display(
    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 92
# Primary vs capped stability
# ============================================================

ari_primary_capped = adjusted_rand_score(

    adata_cap.obs[
        "state_leiden_r0_4"
    ].astype(str),

    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ].astype(str)
)


print("=" * 100)
print("PATIENT-CAPPED SENSITIVITY")
print("=" * 100)

print(
    "Primary clusters on capped cells:",
    adata_cap.obs[
        "state_leiden_r0_4"
    ].nunique()
)

print(
    "Capped-analysis clusters:",
    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_primary_capped,
        4
    )
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 93
# Primary vs capped contingency
# ============================================================

primary_capped_counts = pd.crosstab(

    adata_cap.obs[
        "state_leiden_r0_4"
    ],

    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ]
)


primary_capped_pct = (

    primary_capped_counts
    .div(
        primary_capped_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


print(
    "Counts:"
)

display(
    primary_capped_counts
)


print(
    "\nRow percentages:"
)

display(
    primary_capped_pct
    .round(2)
)


primary_capped_counts.to_csv(

    RESULTS_DIR /
    "Step23C2_Primary_vs_Capped_Counts.csv",

    encoding="utf-8-sig"
)


primary_capped_pct.to_csv(

    RESULTS_DIR /
    "Step23C2_Primary_vs_Capped_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 94
# Patient composition of capped clusters
# ============================================================

capped_cluster_patient_counts = pd.crosstab(

    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ],

    adata_cap.obs[
        "patient"
    ]
)


capped_cluster_patient_pct = (

    capped_cluster_patient_counts
    .div(
        capped_cluster_patient_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


print("=" * 100)
print("CAPPED CLUSTER × PATIENT")
print("=" * 100)

display(
    capped_cluster_patient_counts
)


print(
    "\nRow percentages:"
)

display(
    capped_cluster_patient_pct
    .round(2)
)


capped_cluster_patient_counts.to_csv(

    RESULTS_DIR /
    "Step23C2_CappedCluster_x_Patient_Counts.csv",

    encoding="utf-8-sig"
)


capped_cluster_patient_pct.to_csv(

    RESULTS_DIR /
    "Step23C2_CappedCluster_x_Patient_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 95
# Patient-capped UMAP by patient
# ============================================================

patient_categories = (
    adata_cap.obs[
        "patient"
    ]
    .cat.categories
    .astype(str)
    .tolist()
)


patient_palette = {

    patient:
        NPG_COLORS[
            i %
            len(NPG_COLORS)
        ]

    for i, patient in enumerate(
        patient_categories
    )
}


ax = sc.pl.embedding(

    adata_cap,

    basis=
        "umap_harmony_capped",

    color=
        "patient",

    palette=
        patient_palette,

    size=
        10,

    alpha=
        0.82,

    frameon=
        False,

    legend_loc=
        "right margin",

    title=
        "Patient-capped Harmony Space by Patient",

    show=
        False
)


fig = ax.figure

fig.set_size_inches(
    8.2,
    6.5
)

fig.tight_layout()


save_publication_figure(

    fig,

    "Step23C2_02_PatientCapped_UMAP_ByPatient"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 96
# Patient-capped UMAP by candidate clusters
# ============================================================

ax = sc.pl.embedding(

    adata_cap,

    basis=
        "umap_harmony_capped",

    color=
        "state_capped_leiden_r0_4",

    size=
        10,

    alpha=
        0.82,

    frameon=
        False,

    legend_loc=
        "on data",

    legend_fontsize=
        8,

    title=
        "State Candidates in Patient-capped Sensitivity Analysis",

    show=
        False
)


fig = ax.figure

fig.set_size_inches(
    7.7,
    6.5
)

fig.tight_layout()


save_publication_figure(

    fig,

    "Step23C2_03_PatientCapped_StateCandidates_r0_4"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 97
# Save sensitivity checkpoints
# ============================================================

NO_PC4_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23C2_NoPC4_Sensitivity.h5ad"
)


CAPPED_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23C2_PatientCapped_Sensitivity.h5ad"
)


adata_state.write_h5ad(

    NO_PC4_OUTPUT,

    compression="gzip"
)


adata_cap.write_h5ad(

    CAPPED_OUTPUT,

    compression="gzip"
)


print(
    "[OK]",
    NO_PC4_OUTPUT
)

print(
    "[OK]",
    CAPPED_OUTPUT
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 98
# Stability summary
# ============================================================

stability_summary = pd.DataFrame({

    "Sensitivity_analysis": [

        "Exclude_PC4",
        "Patient_capped_1500"

    ],

    "Primary_resolution": [
        0.4,
        0.4
    ],

    "Primary_clusters": [

        int(
            adata_state.obs[
                "state_leiden_r0_4"
            ].nunique()
        ),

        int(
            adata_cap.obs[
                "state_leiden_r0_4"
            ].nunique()
        )

    ],

    "Sensitivity_clusters": [

        int(
            adata_state.obs[
                "state_noPC4_leiden_r0_4"
            ].nunique()
        ),

        int(
            adata_cap.obs[
                "state_capped_leiden_r0_4"
            ].nunique()
        )

    ],

    "ARI": [

        ari_primary_noPC4,
        ari_primary_capped

    ]
})


display(
    stability_summary
    .round(4)
)


stability_summary.to_csv(

    RESULTS_DIR /
    "Step23C2_State_Stability_Summary.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 99
# Decision log
# ============================================================

step23c2_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23C2_State_Stability_Sensitivity",

    "Status":
        "COMPLETE",

    "Working_resolution":
        0.4,

    "Reason_for_working_resolution":
        (
            "Resolution 0.4 provides a relatively coarse candidate "
            "structure with no clusters smaller than 100 cells and "
            "avoids the very small clusters emerging at higher resolutions."
        ),

    "PC4_sensitivity":
        {

            "Rationale":
                (
                    "PC4 showed the strongest association with mitochondrial "
                    "percentage, detected genes and total counts."
                ),

            "Primary_vs_noPC4_ARI":
                float(
                    ari_primary_noPC4
                )

        },

    "Patient_capped_sensitivity":
        {

            "Cap_per_patient":
                int(
                    PATIENT_CAP
                ),

            "N_cells":
                int(
                    adata_cap.n_obs
                ),

            "Primary_vs_capped_ARI":
                float(
                    ari_primary_capped
                )

        },

    "Final_state_annotation":
        "NOT_PERFORMED",

    "Next_step":
        (
            "Review sensitivity results. If candidate structures are "
            "sufficiently robust, proceed to marker discovery, "
            "cluster-centroid similarity, pathway characterization, "
            "and final malignant-state consolidation."
        )
}


SUMMARY_C2 = (

    LOGS_DIR /
    "Step23C2_Final_Summary.json"
)


with open(

    SUMMARY_C2,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23c2_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_C2
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 100
# Update manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_C2 = (

    RESULTS_DIR /
    "Step23C2_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_C2,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_C2
)

In [ ]:
# ============================================================
# STEP23C-2 - CELL 101
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 118)
print("STEP23C-2 FINAL STABILITY REVIEW")
print("=" * 118)


# ============================================================
# 1. PC4 sensitivity
# ============================================================

print("\n[1] PC4-exclusion sensitivity")
print("-" * 118)

print(
    "Primary clusters:",
    adata_state.obs[
        "state_leiden_r0_4"
    ].nunique()
)

print(
    "No-PC4 clusters:",
    adata_state.obs[
        "state_noPC4_leiden_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_primary_noPC4,
        4
    )
)


print("\nPrimary vs No-PC4 row percentages:")

display(
    primary_noPC4_rowpct
    .round(2)
)


# ============================================================
# 2. Patient-capped cohort
# ============================================================

print("\n[2] Patient-capped cohort")
print("-" * 118)

display(
    adata_cap.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)


# ============================================================
# 3. Capped sensitivity
# ============================================================

print("\n[3] Patient-capped clustering sensitivity")
print("-" * 118)

print(
    "Primary clusters:",
    adata_cap.obs[
        "state_leiden_r0_4"
    ].nunique()
)

print(
    "Capped clusters:",
    adata_cap.obs[
        "state_capped_leiden_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_primary_capped,
        4
    )
)


print("\nPrimary vs Capped row percentages:")

display(
    primary_capped_pct
    .round(2)
)


# ============================================================
# 4. Capped cluster patient composition
# ============================================================

print("\n[4] Capped cluster × patient (%)")
print("-" * 118)

display(
    capped_cluster_patient_pct
    .round(2)
)


# ============================================================
# 5. Summary
# ============================================================

print("\n[5] Overall stability")
print("-" * 118)

display(
    stability_summary
    .round(4)
)


# ============================================================
# 6. Files
# ============================================================

print("\n[6] Files")
print("-" * 118)


for path in [

    NO_PC4_OUTPUT,

    CAPPED_OUTPUT,

    RESULTS_DIR /
    "Step23C2_Primary_vs_NoPC4_Percent.csv",

    RESULTS_DIR /
    "Step23C2_Primary_vs_Capped_Percent.csv",

    RESULTS_DIR /
    "Step23C2_CappedCluster_x_Patient_Percent.csv",

    RESULTS_DIR /
    "Step23C2_State_Stability_Summary.csv",

    SUMMARY_C2,

    MANIFEST_C2

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 118)

print(
    "STEP23C-2 STATUS = COMPLETE"
)

print(
    "FINAL MALIGNANT-STATE ANNOTATION = NOT YET PERFORMED"
)

print(
    "NEXT = marker programs + cluster similarity + state consolidation"
)

print("=" * 118)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23D-1 - CELL 102
# Load primary state-candidate object + raw counts
# ============================================================

from pathlib import Path

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.sparse import vstack

from sklearn.metrics.pairwise import cosine_similarity

import json
import warnings
import gc

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


STATE_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23C_HARMONY_STATE_CANDIDATES.h5ad"
)

RAW_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23A_MALIGNANT_RAW_COUNTS.h5ad"
)


assert STATE_INPUT.exists()
assert RAW_INPUT.exists()


adata_state = sc.read_h5ad(
    STATE_INPUT
)

adata_raw = sc.read_h5ad(
    RAW_INPUT
)


STATE_KEY = "state_leiden_r0_4"


# ------------------------------------------------------------
# Discovery cohort only
# ------------------------------------------------------------

discovery_cells = (
    adata_state.obs_names
)


assert set(
    discovery_cells
).issubset(
    set(
        adata_raw.obs_names
    )
)


adata_raw_disc = (
    adata_raw[
        discovery_cells,
        :
    ]
    .copy()
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    STATE_KEY
]:

    if col in adata_state.obs.columns:

        adata_state.obs[col] = (
            adata_state.obs[col]
            .astype(str)
            .astype("category")
        )


print("=" * 110)
print("STEP23D-1 CONSENSUS STATE EVIDENCE")
print("=" * 110)

print(
    "State object:",
    adata_state.shape
)

print(
    "Raw object:",
    adata_raw_disc.shape
)

print(
    "Primary clusters:",
    adata_state.obs[
        STATE_KEY
    ].nunique()
)

print(
    "Patients:",
    sorted(
        adata_state.obs[
            "patient"
        ]
        .astype(str)
        .unique()
    )
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 103
# Expression / cell alignment audit
# ============================================================

assert (
    adata_state.obs_names
    ==
    adata_raw_disc.obs_names
).all()


assert (
    adata_state.var_names
    ==
    adata_raw_disc.var_names
).all()


# Step23C主对象的X应该是log-normalized expression
X_state = adata_state.X


if sparse.issparse(X_state):

    x_sample = (
        X_state.data[
            :min(
                200000,
                len(X_state.data)
            )
        ]
    )

else:

    x_sample = (
        np.asarray(
            X_state
        )
        .ravel()[
            :200000
        ]
    )


integer_fraction = float(
    np.mean(
        np.isclose(
            x_sample,
            np.round(
                x_sample
            ),
            atol=1e-6
        )
    )
)


print(
    "Cell alignment = PASS"
)

print(
    "Gene alignment = PASS"
)

print(
    "Log-expression integer-like fraction:",
    round(
        integer_fraction,
        4
    )
)


assert integer_fraction < 0.95, (
    "adata_state.X可能不是log-normalized expression。"
)


print(
    "\nExpression matrix = LOG-NORMALIZED PASS"
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 104
# Publication plotting settings
# Figure text: English only
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename
):

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23D-1 - CELL 105
# Primary cluster baseline composition
# ============================================================

primary_cluster_counts = pd.crosstab(

    adata_state.obs[
        STATE_KEY
    ],

    adata_state.obs[
        "patient"
    ]
)


primary_cluster_pct = (

    primary_cluster_counts
    .div(
        primary_cluster_counts.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


primary_cluster_counts.to_csv(

    RESULTS_DIR /
    "Step23D1_PrimaryCluster_x_Patient_Counts.csv",

    encoding="utf-8-sig"
)


primary_cluster_pct.to_csv(

    RESULTS_DIR /
    "Step23D1_PrimaryCluster_x_Patient_Percent.csv",

    encoding="utf-8-sig"
)


print("=" * 100)
print("PRIMARY CLUSTER × PATIENT")
print("=" * 100)

display(
    primary_cluster_counts
)

print("\nPercent:")

display(
    primary_cluster_pct
    .round(2)
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 106
# Cell-level marker discovery
# Exploratory marker discovery, not patient-level validation
# ============================================================

MARKER_KEY = (
    "rank_genes_groups_state_r0_4"
)


sc.tl.rank_genes_groups(

    adata_state,

    groupby=
        STATE_KEY,

    method=
        "wilcoxon",

    use_raw=
        False,

    pts=
        True,

    key_added=
        MARKER_KEY
)


print(
    "rank_genes_groups complete."
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 107
# Export all cell-level markers
# ============================================================

marker_tables = []


clusters = sorted(

    adata_state.obs[
        STATE_KEY
    ]
    .astype(str)
    .unique(),

    key=lambda x:
        int(x)
)


for cluster in clusters:

    df = sc.get.rank_genes_groups_df(

        adata_state,

        group=
            cluster,

        key=
            MARKER_KEY

    )


    df[
        "State_cluster"
    ] = cluster


    marker_tables.append(
        df
    )


all_markers = pd.concat(

    marker_tables,

    ignore_index=True
)


# ------------------------------------------------------------
# Annotate technical gene families
# ------------------------------------------------------------

all_markers[
    "Is_MT"
] = (

    all_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


all_markers[
    "Is_Ribosomal"
] = (

    all_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


ALL_MARKERS_PATH = (

    RESULTS_DIR /
    "Step23D1_CellLevel_AllMarkers.csv"
)


all_markers.to_csv(

    ALL_MARKERS_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "All marker rows:",
    f"{len(all_markers):,}"
)

print(
    "Saved:",
    ALL_MARKERS_PATH
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 108
# Filter candidate biological markers
# ============================================================

candidate_markers = (
    all_markers.copy()
)


# ------------------------------------------------------------
# Basic marker requirements
# ------------------------------------------------------------

candidate_markers = (

    candidate_markers[

        (
            candidate_markers[
                "pvals_adj"
            ]
            <
            0.05
        )

        &

        (
            candidate_markers[
                "logfoldchanges"
            ]
            >
            0.25
        )

        &

        (
            ~candidate_markers[
                "Is_MT"
            ]
        )

        &

        (
            ~candidate_markers[
                "Is_Ribosomal"
            ]
        )

    ]
    .copy()
)


# ------------------------------------------------------------
# If pct expression columns are available
# ------------------------------------------------------------

if (
    "pct_nz_group"
    in candidate_markers.columns
):

    candidate_markers = (

        candidate_markers[
            candidate_markers[
                "pct_nz_group"
            ]
            >=
            0.10
        ]
        .copy()
    )


if (
    "pct_nz_reference"
    in candidate_markers.columns
):

    candidate_markers[
        "Delta_pct"
    ] = (

        candidate_markers[
            "pct_nz_group"
        ]

        -

        candidate_markers[
            "pct_nz_reference"
        ]
    )


    candidate_markers = (

        candidate_markers[
            candidate_markers[
                "Delta_pct"
            ]
            >=
            0.05
        ]
        .copy()
    )


candidate_markers = (

    candidate_markers

    .sort_values(

        [
            "State_cluster",
            "logfoldchanges"
        ],

        ascending=[
            True,
            False
        ]
    )

)


CANDIDATE_MARKER_PATH = (

    RESULTS_DIR /
    "Step23D1_CellLevel_CandidateMarkers.csv"
)


candidate_markers.to_csv(

    CANDIDATE_MARKER_PATH,

    index=False,
    encoding="utf-8-sig"
)


marker_count_summary = (

    candidate_markers
    .groupby(
        "State_cluster"
    )
    .size()
    .rename(
        "N_candidate_markers"
    )
    .reset_index()
)


print("=" * 100)
print("CELL-LEVEL CANDIDATE MARKERS")
print("=" * 100)

display(
    marker_count_summary
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 109
# Top 20 cell-level markers per candidate cluster
# ============================================================

top20_markers = (

    candidate_markers

    .groupby(
        "State_cluster",
        group_keys=False
    )

    .head(
        20
    )

    .copy()
)


TOP20_PATH = (

    RESULTS_DIR /
    "Step23D1_CellLevel_Top20Markers_PerCluster.csv"
)


top20_markers.to_csv(

    TOP20_PATH,

    index=False,
    encoding="utf-8-sig"
)


for cluster in clusters:

    print(
        "\n" +
        "=" * 90
    )

    print(
        f"Cluster {cluster}"
    )

    print(
        "=" * 90
    )


    temp = (

        top20_markers[
            top20_markers[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ]
    )


    display(

        temp[
            [
                "names",
                "logfoldchanges",
                "pvals_adj"
            ]
            +
            (
                [
                    "pct_nz_group",
                    "pct_nz_reference"
                ]

                if (
                    "pct_nz_group"
                    in temp.columns
                )

                else []
            )
        ]
        .head(20)
        .round(4)

    )

In [ ]:
# ============================================================
# STEP23D-1 - CELL 110
# Build patient × candidate-state pseudobulk counts
# ============================================================

# ------------------------------------------------------------
# Transfer state metadata to raw object
# ------------------------------------------------------------

adata_raw_disc.obs[
    "patient"
] = (

    adata_state.obs.loc[
        adata_raw_disc.obs_names,
        "patient"
    ]
    .astype(str)
    .values
)


adata_raw_disc.obs[
    STATE_KEY
] = (

    adata_state.obs.loc[
        adata_raw_disc.obs_names,
        STATE_KEY
    ]
    .astype(str)
    .values
)


adata_raw_disc.obs[
    "patient_state"
] = (

    adata_raw_disc.obs[
        "patient"
    ]
    .astype(str)

    +

    "__State"

    +

    adata_raw_disc.obs[
        STATE_KEY
    ]
    .astype(str)

)


groups = (

    adata_raw_disc.obs[
        "patient_state"
    ]
    .unique()
    .tolist()
)


pb_rows = []
pb_meta = []


for group in groups:

    mask = (

        adata_raw_disc.obs[
            "patient_state"
        ]
        ==
        group
    )


    idx = np.where(
        mask.values
    )[0]


    X_group = (
        adata_raw_disc.X[
            idx,
            :
        ]
    )


    if sparse.issparse(
        X_group
    ):

        summed = (
            X_group.sum(
                axis=0
            )
        )

        summed = sparse.csr_matrix(
            summed
        )

    else:

        summed = sparse.csr_matrix(

            np.sum(
                X_group,
                axis=0,
                keepdims=True
            )

        )


    pb_rows.append(
        summed
    )


    patient = (
        adata_raw_disc.obs.loc[
            mask,
            "patient"
        ]
        .iloc[0]
    )


    state = (
        adata_raw_disc.obs.loc[
            mask,
            STATE_KEY
        ]
        .iloc[0]
    )


    pb_meta.append({

        "patient_state":
            group,

        "patient":
            str(patient),

        "State_cluster":
            str(state),

        "N_cells":
            int(
                mask.sum()
            )
    })


PB_COUNTS = vstack(
    pb_rows
).tocsr()


pb_meta = pd.DataFrame(
    pb_meta
)


adata_pb_counts = ad.AnnData(

    X=
        PB_COUNTS,

    obs=
        pb_meta.set_index(
            "patient_state"
        ),

    var=
        adata_raw_disc.var.copy()
)


print("=" * 100)
print("PATIENT × STATE PSEUDOBULK")
print("=" * 100)

print(
    "Pseudobulk samples:",
    adata_pb_counts.n_obs
)

print(
    "Genes:",
    adata_pb_counts.n_vars
)


display(

    adata_pb_counts.obs
    .sort_values(
        [
            "State_cluster",
            "patient"
        ]
    )

)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 111
# Save raw pseudobulk counts
# ============================================================

PB_COUNTS_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D1_PatientState_Pseudobulk_COUNTS.h5ad"
)


adata_pb_counts.write_h5ad(

    PB_COUNTS_PATH,

    compression="gzip"
)


adata_pb_counts.obs.to_csv(

    RESULTS_DIR /
    "Step23D1_PatientState_Pseudobulk_Metadata.csv",

    encoding="utf-8-sig"
)


print(
    "Saved:",
    PB_COUNTS_PATH
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 112
# Pseudobulk logCPM normalization
# ============================================================

adata_pb_log = (
    adata_pb_counts.copy()
)


sc.pp.normalize_total(

    adata_pb_log,

    target_sum=
        1e6
)


sc.pp.log1p(
    adata_pb_log
)


PB_LOG_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D1_PatientState_Pseudobulk_LogCPM.h5ad"
)


adata_pb_log.write_h5ad(

    PB_LOG_PATH,

    compression="gzip"
)


print(
    "Saved:",
    PB_LOG_PATH
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 113
# Within-patient state-specific expression delta
# ============================================================

X_pb = (
    adata_pb_log.X
)


if sparse.issparse(
    X_pb
):

    X_pb_dense = (
        X_pb
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    X_pb_dense = np.asarray(
        X_pb,
        dtype=np.float32
    )


delta_matrix = np.full(

    X_pb_dense.shape,

    np.nan,

    dtype=np.float32
)


for i in range(
    adata_pb_log.n_obs
):

    patient = (
        str(
            adata_pb_log.obs.iloc[i][
                "patient"
            ]
        )
    )


    state = (
        str(
            adata_pb_log.obs.iloc[i][
                "State_cluster"
            ]
        )
    )


    other_mask = (

        (
            adata_pb_log.obs[
                "patient"
            ]
            .astype(str)
            ==
            patient
        )

        &

        (
            adata_pb_log.obs[
                "State_cluster"
            ]
            .astype(str)
            !=
            state
        )

    )


    other_idx = np.where(
        other_mask.values
    )[0]


    if len(
        other_idx
    ) == 0:

        continue


    other_mean = np.mean(

        X_pb_dense[
            other_idx,
            :
        ],

        axis=0
    )


    delta_matrix[
        i,
        :
    ] = (

        X_pb_dense[
            i,
            :
        ]

        -

        other_mean
    )


adata_pb_delta = ad.AnnData(

    X=
        delta_matrix,

    obs=
        adata_pb_log.obs.copy(),

    var=
        adata_pb_log.var.copy()
)


PB_DELTA_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D1_PatientState_WithinPatient_Delta.h5ad"
)


adata_pb_delta.write_h5ad(

    PB_DELTA_PATH,

    compression="gzip"
)


print(
    "Saved:",
    PB_DELTA_PATH
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 114
# Eligible patient support for each state
# ============================================================

MIN_CELLS_PER_PATIENT_STATE = 20


pb_support = (
    adata_pb_delta.obs
    .copy()
)


pb_support[
    "Eligible"
] = (

    pb_support[
        "N_cells"
    ]
    >=
    MIN_CELLS_PER_PATIENT_STATE
)


state_support_summary = (

    pb_support

    .groupby(
        "State_cluster"
    )

    .agg(

        N_pseudobulks=(
            "patient",
            "size"
        ),

        N_eligible_patients=(
            "Eligible",
            "sum"
        ),

        Total_cells=(
            "N_cells",
            "sum"
        )

    )

    .reset_index()
)


state_support_summary[
    "State_cluster"
] = (

    state_support_summary[
        "State_cluster"
    ]
    .astype(str)
)


print("=" * 100)
print("PATIENT SUPPORT PER CANDIDATE STATE")
print("=" * 100)


display(
    state_support_summary
)


state_support_summary.to_csv(

    RESULTS_DIR /
    "Step23D1_State_PatientSupport.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 115
# Cross-patient marker reproducibility
# ============================================================

gene_to_idx = {

    gene: i
    for i, gene in enumerate(
        adata_pb_delta.var_names
    )
}


marker_repro_rows = []


for cluster in clusters:

    cluster_marker_df = (

        candidate_markers[
            candidate_markers[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ]
        .copy()
    )


    eligible_pb_mask = (

        (
            adata_pb_delta.obs[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        )

        &

        (
            adata_pb_delta.obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    eligible_idx = np.where(
        eligible_pb_mask.values
    )[0]


    eligible_patients = (

        adata_pb_delta.obs.iloc[
            eligible_idx
        ][
            "patient"
        ]
        .astype(str)
        .tolist()
    )


    for _, marker_row in (
        cluster_marker_df
        .iterrows()
    ):

        gene = str(
            marker_row[
                "names"
            ]
        )


        if gene not in gene_to_idx:
            continue


        gene_idx = (
            gene_to_idx[
                gene
            ]
        )


        values = (

            delta_matrix[
                eligible_idx,
                gene_idx
            ]

            if len(
                eligible_idx
            ) > 0

            else np.array(
                []
            )
        )


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ) == 0:

            n_positive = 0
            positive_fraction = np.nan
            median_delta = np.nan
            mean_delta = np.nan

        else:

            n_positive = int(
                np.sum(
                    values > 0
                )
            )

            positive_fraction = float(
                np.mean(
                    values > 0
                )
            )

            median_delta = float(
                np.median(
                    values
                )
            )

            mean_delta = float(
                np.mean(
                    values
                )
            )


        marker_repro_rows.append({

            "State_cluster":
                cluster,

            "Gene":
                gene,

            "CellLevel_logFC":
                float(
                    marker_row[
                        "logfoldchanges"
                    ]
                ),

            "CellLevel_adjP":
                float(
                    marker_row[
                        "pvals_adj"
                    ]
                ),

            "N_eligible_patients":
                len(
                    values
                ),

            "N_positive_patients":
                n_positive,

            "Positive_fraction":
                positive_fraction,

            "Median_withinPatient_delta":
                median_delta,

            "Mean_withinPatient_delta":
                mean_delta,

            "Eligible_patients":
                "|".join(
                    eligible_patients
                )
        })


marker_repro = pd.DataFrame(
    marker_repro_rows
)


MARKER_REPRO_PATH = (

    RESULTS_DIR /
    "Step23D1_Marker_CrossPatient_Reproducibility.csv"
)


marker_repro.to_csv(

    MARKER_REPRO_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Marker reproducibility rows:",
    f"{len(marker_repro):,}"
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 116
# Rank cross-patient reproducible markers
# ============================================================

marker_repro_ranked = (

    marker_repro

    .sort_values(

        [
            "State_cluster",
            "N_positive_patients",
            "Positive_fraction",
            "Median_withinPatient_delta",
            "CellLevel_logFC"
        ],

        ascending=[
            True,
            False,
            False,
            False,
            False
        ]

    )
)


top_repro_markers = (

    marker_repro_ranked

    .groupby(
        "State_cluster",
        group_keys=False
    )

    .head(
        20
    )

    .copy()
)


TOP_REPRO_PATH = (

    RESULTS_DIR /
    "Step23D1_Top20_CrossPatient_Markers_PerCluster.csv"
)


top_repro_markers.to_csv(

    TOP_REPRO_PATH,

    index=False,
    encoding="utf-8-sig"
)


for cluster in clusters:

    print(
        "\n" +
        "=" * 100
    )

    print(
        f"State candidate {cluster}"
    )

    print(
        "=" * 100
    )


    display(

        top_repro_markers[
            top_repro_markers[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ][
            [
                "Gene",
                "CellLevel_logFC",
                "N_eligible_patients",
                "N_positive_patients",
                "Positive_fraction",
                "Median_withinPatient_delta"
            ]
        ]
        .round(4)

    )

In [ ]:
# ============================================================
# STEP23D-1 - CELL 117
# Build patient-adjusted state signatures
# ============================================================

# 使用PCA的HVG
if (
    "highly_variable_for_pca"
    in adata_state.var.columns
):

    signature_genes = (

        adata_state.var_names[
            adata_state.var[
                "highly_variable_for_pca"
            ]
            .astype(bool)
        ]
        .tolist()
    )

else:

    signature_genes = (

        adata_state.var_names[
            adata_state.var[
                "highly_variable"
            ]
            .astype(bool)
        ]
        .tolist()
    )


signature_gene_idx = [

    gene_to_idx[
        gene
    ]

    for gene in signature_genes

    if gene in gene_to_idx
]


signature_genes = [

    gene

    for gene in signature_genes

    if gene in gene_to_idx
]


state_signature_rows = []

state_signature_labels = []


for cluster in clusters:

    mask = (

        (
            adata_pb_delta.obs[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        )

        &

        (
            adata_pb_delta.obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )

    )


    idx = np.where(
        mask.values
    )[0]


    if len(
        idx
    ) == 0:

        state_signature = np.full(

            len(
                signature_gene_idx
            ),

            np.nan,

            dtype=np.float32
        )

    else:

        state_signature = np.nanmean(

            delta_matrix[
                idx,
                :
            ][
                :,
                signature_gene_idx
            ],

            axis=0
        )


    state_signature_rows.append(
        state_signature
    )


    state_signature_labels.append(
        cluster
    )


state_signature_matrix = np.vstack(
    state_signature_rows
)


print(
    "State signature matrix:",
    state_signature_matrix.shape
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 118
# Patient-adjusted state similarity
# ============================================================

state_signature_corr = np.corrcoef(
    state_signature_matrix
)


state_signature_corr_df = pd.DataFrame(

    state_signature_corr,

    index=
        state_signature_labels,

    columns=
        state_signature_labels
)


state_signature_corr_df.index.name = (
    "State_cluster"
)


STATE_CORR_PATH = (

    RESULTS_DIR /
    "Step23D1_State_PatientAdjusted_Correlation.csv"
)


state_signature_corr_df.to_csv(

    STATE_CORR_PATH,

    encoding="utf-8-sig"
)


display(
    state_signature_corr_df
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 119
# Harmony centroid similarity
# ============================================================

X_harmony = (

    adata_state.obsm[
        "X_pca_harmony_patient"
    ]
)


harmony_centroids = []


for cluster in clusters:

    mask = (

        adata_state.obs[
            STATE_KEY
        ]
        .astype(str)
        ==
        cluster
    )


    centroid = np.mean(

        X_harmony[
            mask.values,
            :
        ],

        axis=0
    )


    harmony_centroids.append(
        centroid
    )


harmony_centroids = np.vstack(
    harmony_centroids
)


harmony_cosine = cosine_similarity(
    harmony_centroids
)


harmony_cosine_df = pd.DataFrame(

    harmony_cosine,

    index=
        clusters,

    columns=
        clusters
)


harmony_cosine_df.index.name = (
    "State_cluster"
)


HARMONY_SIM_PATH = (

    RESULTS_DIR /
    "Step23D1_State_HarmonyCentroid_Cosine.csv"
)


harmony_cosine_df.to_csv(

    HARMONY_SIM_PATH,

    encoding="utf-8-sig"
)


display(
    harmony_cosine_df
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 120
# Rank most similar candidate-state pairs
# ============================================================

pair_rows = []


for i in range(
    len(
        clusters
    )
):

    for j in range(
        i + 1,
        len(
            clusters
        )
    ):

        c1 = clusters[i]
        c2 = clusters[j]


        pair_rows.append({

            "State_1":
                c1,

            "State_2":
                c2,

            "PatientAdjusted_Expression_Correlation":
                float(
                    state_signature_corr_df.loc[
                        c1,
                        c2
                    ]
                ),

            "HarmonyCentroid_Cosine":
                float(
                    harmony_cosine_df.loc[
                        c1,
                        c2
                    ]
                )
        })


state_pair_similarity = pd.DataFrame(
    pair_rows
)


state_pair_similarity[
    "Mean_similarity"
] = (

    state_pair_similarity[
        [
            "PatientAdjusted_Expression_Correlation",
            "HarmonyCentroid_Cosine"
        ]
    ]
    .mean(
        axis=1
    )
)


state_pair_similarity = (

    state_pair_similarity
    .sort_values(
        "Mean_similarity",
        ascending=False
    )
)


PAIR_PATH = (

    RESULTS_DIR /
    "Step23D1_StatePair_Similarity.csv"
)


state_pair_similarity.to_csv(

    PAIR_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 100)
print("MOST SIMILAR STATE-CANDIDATE PAIRS")
print("=" * 100)


display(

    state_pair_similarity
    .head(20)
    .round(4)

)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 121
# State similarity heatmap
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.2, 7.0)
)


im = ax.imshow(

    state_signature_corr,

    vmin=-1,
    vmax=1,

    cmap="RdBu_r",

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        len(
            clusters
        )
    )
)

ax.set_yticks(
    np.arange(
        len(
            clusters
        )
    )
)


ax.set_xticklabels(
    clusters
)

ax.set_yticklabels(
    clusters
)


ax.set_xlabel(
    "State Candidate"
)

ax.set_ylabel(
    "State Candidate"
)

ax.set_title(
    "Patient-adjusted Transcriptomic Similarity Between State Candidates"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Pearson Correlation"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D1_01_PatientAdjusted_StateSimilarity"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 122
# Cluster x patient composition heatmap
# ============================================================

patient_pct_matrix = (

    primary_cluster_pct
    .reindex(
        index=[
            str(x)
            for x in clusters
        ],
        fill_value=0
    )
)


# 某些index如果还是category/int，统一转str
patient_pct_matrix.index = (
    patient_pct_matrix.index
    .astype(str)
)


fig, ax = plt.subplots(
    figsize=(8.0, 6.8)
)


im = ax.imshow(

    patient_pct_matrix.values,

    vmin=0,
    vmax=100,

    cmap="Blues",

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        patient_pct_matrix.shape[1]
    )
)

ax.set_xticklabels(
    patient_pct_matrix.columns,
    rotation=45,
    ha="right"
)


ax.set_yticks(
    np.arange(
        patient_pct_matrix.shape[0]
    )
)

ax.set_yticklabels(
    patient_pct_matrix.index
)


ax.set_xlabel(
    "Patient"
)

ax.set_ylabel(
    "State Candidate"
)

ax.set_title(
    "Patient Composition of Malignant State Candidates"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Cell Fraction (%)"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D1_02_StateCandidate_PatientComposition"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 123
# Build cluster stability evidence
# ============================================================

NO_PC4_PATH = (

    RESULTS_DIR /
    "Step23C2_Primary_vs_NoPC4_Percent.csv"
)


CAPPED_PATH = (

    RESULTS_DIR /
    "Step23C2_Primary_vs_Capped_Percent.csv"
)


assert NO_PC4_PATH.exists()
assert CAPPED_PATH.exists()


no_pc4_pct = pd.read_csv(

    NO_PC4_PATH,

    index_col=0
)


capped_pct = pd.read_csv(

    CAPPED_PATH,

    index_col=0
)


no_pc4_pct.index = (
    no_pc4_pct.index
    .astype(str)
)


capped_pct.index = (
    capped_pct.index
    .astype(str)
)


# ------------------------------------------------------------
# r0.4 vs r0.6
# ------------------------------------------------------------

r04_r06 = pd.crosstab(

    adata_state.obs[
        "state_leiden_r0_4"
    ],

    adata_state.obs[
        "state_leiden_r0_6"
    ]
)


r04_r06_pct = (

    r04_r06
    .div(
        r04_r06.sum(
            axis=1
        ),
        axis=0
    )
    *
    100
)


r04_r06_pct.index = (
    r04_r06_pct.index
    .astype(str)
)


stability_rows = []


for cluster in clusters:

    cluster = str(
        cluster
    )


    no_pc4_row = (
        no_pc4_pct.loc[
            cluster
        ]
    )


    capped_row = (
        capped_pct.loc[
            cluster
        ]
    )


    r06_row = (
        r04_r06_pct.loc[
            cluster
        ]
    )


    stability_rows.append({

        "State_cluster":
            cluster,

        "NoPC4_best_match":
            str(
                no_pc4_row.idxmax()
            ),

        "NoPC4_best_match_pct":
            float(
                no_pc4_row.max()
            ),

        "Capped_best_match":
            str(
                capped_row.idxmax()
            ),

        "Capped_best_match_pct":
            float(
                capped_row.max()
            ),

        "r0_6_best_match":
            str(
                r06_row.idxmax()
            ),

        "r0_6_best_match_pct":
            float(
                r06_row.max()
            )
    })


cluster_stability = pd.DataFrame(
    stability_rows
)


display(
    cluster_stability
    .round(2)
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 124
# Master evidence table
# ============================================================

# ------------------------------------------------------------
# Patient composition
# ------------------------------------------------------------

patient_summary_rows = []


for cluster in clusters:

    cluster = str(
        cluster
    )


    row = (
        primary_cluster_counts.loc[
            cluster
        ]
    )


    total = row.sum()


    dominant_patient = (
        row.idxmax()
    )


    patient_summary_rows.append({

        "State_cluster":
            cluster,

        "N_cells":
            int(
                total
            ),

        "N_patients_present":
            int(
                (
                    row > 0
                ).sum()
            ),

        "N_patients_ge20":
            int(
                (
                    row >= 20
                ).sum()
            ),

        "N_patients_ge50":
            int(
                (
                    row >= 50
                ).sum()
            ),

        "Dominant_patient":
            str(
                dominant_patient
            ),

        "Dominant_patient_pct":
            float(
                row.max()
                /
                total
                *
                100
            )
    })


patient_summary = pd.DataFrame(
    patient_summary_rows
)


# ------------------------------------------------------------
# Marker reproducibility summary
# ------------------------------------------------------------

marker_support_summary = (

    marker_repro

    .groupby(
        "State_cluster"
    )

    .agg(

        N_candidate_markers=(
            "Gene",
            "size"
        ),

        Max_patients_supported=(
            "N_eligible_patients",
            "max"
        ),

        N_markers_positive_in_2plus_patients=(

            "N_positive_patients",

            lambda x:
                int(
                    np.sum(
                        x >= 2
                    )
                )
        ),

        N_markers_positive_in_3plus_patients=(

            "N_positive_patients",

            lambda x:
                int(
                    np.sum(
                        x >= 3
                    )
                )
        ),

        Median_marker_positive_fraction=(
            "Positive_fraction",
            "median"
        )
    )

    .reset_index()
)


marker_support_summary[
    "State_cluster"
] = (
    marker_support_summary[
        "State_cluster"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

master_evidence = (

    patient_summary

    .merge(
        cluster_stability,
        on="State_cluster",
        how="left"
    )

    .merge(
        state_support_summary,
        on="State_cluster",
        how="left",
        suffixes=(
            "",
            "_PB"
        )
    )

    .merge(
        marker_support_summary,
        on="State_cluster",
        how="left"
    )
)


MASTER_PATH = (

    RESULTS_DIR /
    "Step23D1_Master_StateCandidate_Evidence.csv"
)


master_evidence.to_csv(

    MASTER_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 110)
print("MASTER STATE-CANDIDATE EVIDENCE")
print("=" * 110)


display(
    master_evidence
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 125
# Save state-evidence object
# ============================================================

EVIDENCE_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D1_STATE_CANDIDATE_EVIDENCE.h5ad"
)


adata_state.write_h5ad(

    EVIDENCE_OUTPUT,

    compression="gzip"
)


print(
    "Saved:",
    EVIDENCE_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 126
# Decision log
# ============================================================

step23d1_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23D1_Consensus_State_Evidence",

    "Status":
        "EVIDENCE_CONSTRUCTION_COMPLETE",

    "Primary_partition":
        "Harmony(patient) + Leiden r=0.4",

    "Primary_clusters":
        int(
            len(
                clusters
            )
        ),

    "Important_rule":
        (
            "Raw Leiden clusters are candidate partitions and are "
            "not treated as final malignant biological states."
        ),

    "Marker_method":
        (
            "Cell-level Wilcoxon markers followed by patient-by-state "
            "pseudobulk consistency assessment."
        ),

    "Pseudobulk_unit":
        "patient × candidate-state",

    "Pseudobulk_normalization":
        "library-size normalized to 1e6 followed by log1p",

    "Cross_patient_marker_logic":
        (
            "For each patient-state unit, expression is compared with "
            "the mean expression of the patient's other candidate states."
        ),

    "State_similarity":
        [
            "Patient-adjusted pseudobulk expression correlation",
            "Harmony centroid cosine similarity"
        ],

    "Stability_inputs":
        [
            "Primary vs no-PC4",
            "Primary vs patient-capped",
            "Primary r0.4 vs r0.6"
        ],

    "Final_state_names":
        "NOT_ASSIGNED",

    "Next_step":
        (
            "Review candidate-state markers, cross-patient reproducibility, "
            "pairwise similarity and stability evidence; then perform "
            "pathway characterization and consensus state consolidation."
        ),

    "Evidence_object":
        str(
            EVIDENCE_OUTPUT
        )
}


SUMMARY_D1 = (

    LOGS_DIR /
    "Step23D1_Final_Summary.json"
)


with open(

    SUMMARY_D1,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23d1_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_D1
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 127
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_D1 = (

    RESULTS_DIR /
    "Step23D1_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_D1,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_D1
)

In [ ]:
# ============================================================
# STEP23D-1 - CELL 128
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 120)
print("STEP23D-1 FINAL REVIEW")
print("=" * 120)


# ============================================================
# 1. Patient support
# ============================================================

print("\n[1] Candidate-state patient support")
print("-" * 120)

display(
    patient_summary
    .round(2)
)


# ============================================================
# 2. Stability
# ============================================================

print("\n[2] Candidate-state stability")
print("-" * 120)

display(
    cluster_stability
    .round(2)
)


# ============================================================
# 3. Pseudobulk support
# ============================================================

print("\n[3] Pseudobulk patient support")
print("-" * 120)

display(
    state_support_summary
)


# ============================================================
# 4. Cross-patient marker evidence
# ============================================================

print("\n[4] Marker reproducibility summary")
print("-" * 120)

display(
    marker_support_summary
    .round(3)
)


# ============================================================
# 5. Master evidence
# ============================================================

print("\n[5] Master evidence table")
print("-" * 120)

display(
    master_evidence
    .round(3)
)


# ============================================================
# 6. Top reproducible markers
# ============================================================

for cluster in clusters:

    print(
        f"\n[6] Top reproducible markers — candidate {cluster}"
    )

    print(
        "-" * 120
    )


    display(

        top_repro_markers[
            top_repro_markers[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ][
            [
                "Gene",
                "CellLevel_logFC",
                "N_eligible_patients",
                "N_positive_patients",
                "Positive_fraction",
                "Median_withinPatient_delta"
            ]
        ]
        .head(10)
        .round(4)

    )


# ============================================================
# 7. Most similar candidate pairs
# ============================================================

print("\n[7] Most similar candidate-state pairs")
print("-" * 120)

display(
    state_pair_similarity
    .head(20)
    .round(4)
)


# ============================================================
# 8. Files
# ============================================================

print("\n[8] Files")
print("-" * 120)


for path in [

    EVIDENCE_OUTPUT,

    PB_COUNTS_PATH,

    PB_LOG_PATH,

    PB_DELTA_PATH,

    ALL_MARKERS_PATH,

    CANDIDATE_MARKER_PATH,

    TOP_REPRO_PATH,

    MARKER_REPRO_PATH,

    STATE_CORR_PATH,

    HARMONY_SIM_PATH,

    PAIR_PATH,

    MASTER_PATH,

    SUMMARY_D1,

    MANIFEST_D1

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 120)

print(
    "STEP23D-1 STATUS = EVIDENCE CONSTRUCTION COMPLETE"
)

print(
    "FINAL STATE CONSOLIDATION = NOT YET PERFORMED"
)

print(
    "NEXT = pathway characterization + consensus consolidation"
)

print("=" * 120)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23D-2 - CELL 102
# Load Step23D-1 evidence object
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse

import json
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D1_STATE_CANDIDATE_EVIDENCE.h5ad"
)


assert INPUT.exists()


adata_d2 = sc.read_h5ad(
    INPUT
)


STATE_KEY = "state_leiden_r0_4"


assert STATE_KEY in adata_d2.obs.columns


adata_d2.obs[STATE_KEY] = (
    adata_d2.obs[STATE_KEY]
    .astype(str)
    .astype("category")
)


print("=" * 105)
print("STEP23D-2 IMMUNE-CONTAMINATION AUDIT")
print("=" * 105)

print(
    "Shape:",
    adata_d2.shape
)

print(
    "Candidate states:",
    adata_d2.obs[
        STATE_KEY
    ].nunique()
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 103
# Expression matrix audit
# ============================================================

X = adata_d2.X


if sparse.issparse(X):
    values = X.data
else:
    values = np.asarray(X).ravel()


values = values[
    values != 0
]


rng = np.random.default_rng(SEED)


if len(values) > 100000:

    test_values = rng.choice(
        values,
        size=100000,
        replace=False
    )

else:

    test_values = values


integer_fraction = float(

    np.mean(

        np.isclose(
            test_values,
            np.round(test_values),
            atol=1e-6
        )

    )

)


print(
    "Integer-like fraction:",
    round(integer_fraction, 4)
)


if integer_fraction > 0.95:

    print(
        "WARNING: X looks count-like."
    )

else:

    print(
        "Expression matrix is consistent with normalized/log-transformed data."
    )

In [ ]:
# ============================================================
# STEP23D-2 - CELL 104
# Define lineage programs
# ============================================================

LINEAGE_PROGRAMS = {

    "T_NK_program": [
        "CD3D",
        "CD3E",
        "TRAC",
        "CD2",
        "IL7R",
        "CCL5",
        "GZMK",
        "GZMA",
        "NKG7",
        "GNLY"
    ],

    "B_Plasma_program": [
        "CD79A",
        "MS4A1",
        "CD37",
        "JCHAIN",
        "MZB1",
        "IGKC",
        "IGLC2",
        "TNFRSF17",
        "SDC1"
    ],

    "Myeloid_program": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "CTSS",
        "AIF1",
        "C1QC",
        "FCGR3A",
        "LILRB1",
        "MSR1"
    ],

    "Epithelial_program": [
        "EPCAM",
        "KRT7",
        "KRT8",
        "KRT18",
        "KRT19",
        "KRT17",
        "TACSTD2"
    ],

    "Hepatic_program": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "HPD",
        "CPS1",
        "CYP8B1",
        "MAT1A"
    ],

    "Tumor_epithelial_program": [
        "GPC3",
        "AFP",
        "S100P",
        "SPINK1",
        "AGR2",
        "CLDN6",
        "PRAME",
        "KRT19",
        "EPCAM"
    ]
}


program_audit_rows = []


for score_name, genes in LINEAGE_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata_d2.var_names
    ]


    missing = [
        gene
        for gene in genes
        if gene not in adata_d2.var_names
    ]


    print(
        "\n",
        score_name
    )

    print(
        "Detected:",
        detected
    )

    print(
        "Missing:",
        missing
    )


    assert len(detected) >= 3


    sc.tl.score_genes(

        adata_d2,

        gene_list=
            detected,

        score_name=
            score_name,

        ctrl_size=
            min(
                50,
                max(
                    10,
                    len(detected) * 5
                )
            ),

        n_bins=
            25,

        random_state=
            SEED,

        use_raw=
            False
    )


    for gene in genes:

        program_audit_rows.append({

            "Program":
                score_name,

            "Gene":
                gene,

            "Detected":
                gene in adata_d2.var_names

        })


program_gene_audit = pd.DataFrame(
    program_audit_rows
)


program_gene_audit.to_csv(

    RESULTS_DIR /
    "Step23D2_LineageProgram_GeneAudit.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 105
# Median lineage scores by candidate state
# ============================================================

score_columns = list(
    LINEAGE_PROGRAMS.keys()
)


state_lineage_summary = (

    adata_d2.obs

    .groupby(
        STATE_KEY,
        observed=True
    )[
        score_columns
    ]

    .median()

    .reset_index()
)


state_lineage_summary[
    STATE_KEY
] = (
    state_lineage_summary[
        STATE_KEY
    ]
    .astype(str)
)


state_lineage_summary = (

    state_lineage_summary
    .sort_values(
        STATE_KEY,
        key=lambda x:
            x.astype(int)
    )

    .reset_index(
        drop=True
    )
)


print("=" * 105)
print("MEDIAN LINEAGE PROGRAMS BY CANDIDATE STATE")
print("=" * 105)


display(
    state_lineage_summary
    .round(3)
)


state_lineage_summary.to_csv(

    RESULTS_DIR /
    "Step23D2_State_LineageProgram_Summary.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 106
# Per-cell expressed lineage-gene counts
# ============================================================

CORE_PANELS = {

    "T_core": [
        "CD3D",
        "CD3E",
        "TRAC",
        "CD2",
        "IL7R",
        "CCL5"
    ],

    "BPlasma_core": [
        "CD79A",
        "MS4A1",
        "JCHAIN",
        "MZB1",
        "IGKC",
        "TNFRSF17"
    ],

    "Myeloid_core": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "CTSS",
        "C1QC",
        "FCGR3A"
    ],

    "Epithelial_core": [
        "EPCAM",
        "KRT7",
        "KRT8",
        "KRT18",
        "KRT19",
        "TACSTD2"
    ],

    "Hepatic_core": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "CPS1"
    ]
}


def expressed_gene_count(
    adata,
    genes
):

    genes = [
        g
        for g in genes
        if g in adata.var_names
    ]


    X_sub = (
        adata[
            :,
            genes
        ]
        .X
    )


    if sparse.issparse(X_sub):

        counts = np.asarray(
            (
                X_sub > 0
            ).sum(axis=1)
        ).ravel()

    else:

        counts = np.sum(
            X_sub > 0,
            axis=1
        )


    return (
        counts.astype(int),
        len(genes)
    )


core_sizes = {}


for panel_name, genes in CORE_PANELS.items():

    counts, n_genes = (
        expressed_gene_count(
            adata_d2,
            genes
        )
    )


    adata_d2.obs[
        f"{panel_name}_expressed_n"
    ] = counts


    core_sizes[
        panel_name
    ] = n_genes


print(
    "Core panel sizes:",
    core_sizes
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 107
# Candidate-state lineage identity audit
# ============================================================

identity_rows = []


states = sorted(

    adata_d2.obs[
        STATE_KEY
    ]
    .astype(str)
    .unique(),

    key=lambda x:
        int(x)
)


for state in states:

    mask = (
        adata_d2.obs[
            STATE_KEY
        ]
        .astype(str)
        ==
        state
    )


    temp = (
        adata_d2.obs.loc[
            mask
        ]
    )


    row = {

        "State_cluster":
            state,

        "N_cells":
            len(temp)

    }


    for panel_name in CORE_PANELS.keys():

        column = (
            f"{panel_name}_expressed_n"
        )


        row[
            f"Median_{panel_name}_genes"
        ] = float(
            temp[column]
            .median()
        )


        row[
            f"Pct_{panel_name}_ge3"
        ] = float(

            (
                temp[column]
                >=
                3
            )
            .mean()
            *
            100

        )


    identity_rows.append(
        row
    )


state_identity_audit = pd.DataFrame(
    identity_rows
)


print("=" * 115)
print("CANDIDATE-STATE LINEAGE IDENTITY AUDIT")
print("=" * 115)


display(
    state_identity_audit
    .round(2)
)


state_identity_audit.to_csv(

    RESULTS_DIR /
    "Step23D2_State_LineageIdentity_Audit.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 108
# QC / doublet / CNV audit
# ============================================================

possible_doublet_cols = [

    col
    for col in adata_d2.obs.columns

    if (
        "doublet"
        in col.lower()

        or

        "scrublet"
        in col.lower()
    )

]


print(
    "Doublet/Scrublet columns:",
    possible_doublet_cols
)


audit_numeric_cols = [

    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "cnv_burden_primary"

]


audit_numeric_cols += [

    col
    for col in possible_doublet_cols

    if pd.api.types.is_numeric_dtype(
        adata_d2.obs[col]
    )
]


audit_numeric_cols = list(
    dict.fromkeys(
        [
            col
            for col in audit_numeric_cols
            if col in adata_d2.obs.columns
        ]
    )
)


qc_state_summary = (

    adata_d2.obs

    .groupby(
        STATE_KEY,
        observed=True
    )[
        audit_numeric_cols
    ]

    .median()

    .reset_index()
)


print("=" * 110)
print("QC / CNV / DOUBLET AUDIT BY CANDIDATE STATE")
print("=" * 110)


display(
    qc_state_summary
    .round(4)
)


qc_state_summary.to_csv(

    RESULTS_DIR /
    "Step23D2_State_QC_Doublet_CNV_Audit.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 109
# Key marker detection fractions
# ============================================================

KEY_MARKERS = [

    # pan immune
    "PTPRC",

    # T/NK
    "CD3D",
    "CD3E",
    "TRAC",
    "CD2",
    "IL7R",
    "CCL5",

    # plasma/B
    "JCHAIN",
    "MZB1",
    "IGKC",
    "CD79A",

    # myeloid
    "LST1",
    "TYROBP",
    "FCER1G",
    "C1QC",

    # epithelial
    "EPCAM",
    "KRT8",
    "KRT18",
    "KRT19",

    # hepatocyte
    "ALB",
    "APOA1",
    "CPS1",

    # tumor-associated
    "GPC3",
    "AFP",
    "S100P",
    "SPINK1"
]


KEY_MARKERS = [
    gene
    for gene in KEY_MARKERS
    if gene in adata_d2.var_names
]


marker_fraction_rows = []


for state in states:

    mask = (
        adata_d2.obs[
            STATE_KEY
        ]
        .astype(str)
        ==
        state
    )


    adata_temp = (
        adata_d2[
            mask,
            KEY_MARKERS
        ]
    )


    X_temp = (
        adata_temp.X
    )


    if sparse.issparse(
        X_temp
    ):

        expression_fraction = (

            np.asarray(
                (
                    X_temp > 0
                )
                .mean(axis=0)
            )
            .ravel()

        )

    else:

        expression_fraction = (

            np.mean(
                X_temp > 0,
                axis=0
            )

        )


    for gene, fraction in zip(
        KEY_MARKERS,
        expression_fraction
    ):

        marker_fraction_rows.append({

            "State_cluster":
                state,

            "Gene":
                gene,

            "Pct_expressing":
                float(
                    fraction
                    *
                    100
                )

        })


marker_fraction = pd.DataFrame(
    marker_fraction_rows
)


marker_fraction.to_csv(

    RESULTS_DIR /
    "Step23D2_KeyMarker_ExpressionFraction.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 110
# Lineage program heatmap
# ============================================================

heatmap_df = (

    state_lineage_summary
    .set_index(
        STATE_KEY
    )[
        score_columns
    ]
    .copy()
)


# column-wise z score
heatmap_z = (

    heatmap_df
    -
    heatmap_df.mean(
        axis=0
    )

) / (

    heatmap_df.std(
        axis=0
    )
    +
    1e-8

)


fig, ax = plt.subplots(
    figsize=(8.2, 6.2)
)


im = ax.imshow(

    heatmap_z.values,

    aspect=
        "auto",

    cmap=
        "coolwarm",

    vmin=
        -2,

    vmax=
        2
)


ax.set_xticks(
    np.arange(
        heatmap_z.shape[1]
    )
)


ax.set_xticklabels(

    [
        x.replace(
            "_program",
            ""
        )
        .replace(
            "_",
            " "
        )

        for x in heatmap_z.columns
    ],

    rotation=45,

    ha="right"
)


ax.set_yticks(
    np.arange(
        heatmap_z.shape[0]
    )
)


ax.set_yticklabels(
    heatmap_z.index
)


ax.set_xlabel(
    "Lineage Program"
)

ax.set_ylabel(
    "Candidate State"
)

ax.set_title(
    "Lineage Program Audit of Malignant-state Candidates"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Z-scored Median Program Score"
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step23D2_01_LineageProgram_Heatmap.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step23D2_01_LineageProgram_Heatmap.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 111
# Focused audit of candidates 5 and 6
# ============================================================

focus_mask = (

    adata_d2.obs[
        STATE_KEY
    ]
    .astype(str)
    .isin(
        [
            "5",
            "6"
        ]
    )

)


focus_cols = [

    "patient",
    "sample",
    "site",
    STATE_KEY,

    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",

    "cnv_leiden_r0_4",
    "cnv_burden_primary",

    "T_NK_program",
    "B_Plasma_program",
    "Myeloid_program",
    "Epithelial_program",
    "Hepatic_program",
    "Tumor_epithelial_program",

    "T_core_expressed_n",
    "BPlasma_core_expressed_n",
    "Myeloid_core_expressed_n",
    "Epithelial_core_expressed_n",
    "Hepatic_core_expressed_n"
]


focus_cols += [

    col
    for col in possible_doublet_cols
    if col not in focus_cols

]


focus_cols = [

    col
    for col in focus_cols
    if col in adata_d2.obs.columns
]


candidate56_audit = (

    adata_d2.obs.loc[
        focus_mask,
        focus_cols
    ]
    .copy()
)


candidate56_audit.index.name = (
    "Cell"
)


candidate56_audit.to_csv(

    RESULTS_DIR /
    "Step23D2_Candidate5_6_CellLevel_Audit.csv",

    encoding="utf-8-sig"
)


print(
    "Candidate 5/6 cells:",
    len(
        candidate56_audit
    )
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 112
# Save audit checkpoint
# ============================================================

AUDIT_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D2_LINEAGE_CONTAMINATION_AUDIT.h5ad"
)


adata_d2.write_h5ad(

    AUDIT_OUTPUT,

    compression="gzip"
)


print(
    "Saved:",
    AUDIT_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-2 - CELL 113
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 118)
print("STEP23D-2 LINEAGE CONTAMINATION REVIEW")
print("=" * 118)


print("\n[1] Median lineage program scores")
print("-" * 118)

display(
    state_lineage_summary
    .round(3)
)


print("\n[2] Core lineage-gene expression")
print("-" * 118)

display(
    state_identity_audit
    .round(2)
)


print("\n[3] QC / CNV / doublet audit")
print("-" * 118)

display(
    qc_state_summary
    .round(4)
)


print("\n[4] Candidate 5 detailed marker fractions")
print("-" * 118)

display(

    marker_fraction[
        marker_fraction[
            "State_cluster"
        ]
        .astype(str)
        ==
        "5"
    ]

    .sort_values(
        "Pct_expressing",
        ascending=False
    )

    .round(2)

)


print("\n[5] Candidate 6 detailed marker fractions")
print("-" * 118)

display(

    marker_fraction[
        marker_fraction[
            "State_cluster"
        ]
        .astype(str)
        ==
        "6"
    ]

    .sort_values(
        "Pct_expressing",
        ascending=False
    )

    .round(2)

)


print("\n[6] Candidate 5/6 cell counts by patient")
print("-" * 118)

display(

    pd.crosstab(

        adata_d2.obs.loc[
            focus_mask,
            STATE_KEY
        ],

        adata_d2.obs.loc[
            focus_mask,
            "patient"
        ]

    )

)


print("\n[7] Files")
print("-" * 118)


for path in [

    AUDIT_OUTPUT,

    RESULTS_DIR /
    "Step23D2_State_LineageProgram_Summary.csv",

    RESULTS_DIR /
    "Step23D2_State_LineageIdentity_Audit.csv",

    RESULTS_DIR /
    "Step23D2_State_QC_Doublet_CNV_Audit.csv",

    RESULTS_DIR /
    "Step23D2_KeyMarker_ExpressionFraction.csv",

    RESULTS_DIR /
    "Step23D2_Candidate5_6_CellLevel_Audit.csv"

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 118)

print(
    "STEP23D-2 STATUS = LINEAGE AUDIT COMPLETE"
)

print(
    "NO CANDIDATE HAS BEEN REMOVED AUTOMATICALLY."
)

print(
    "NEXT = decide contamination exclusions, then pathway characterization and state consolidation."
)

print("=" * 118)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23D-3 - CELL 141
# Build state-analysis eligibility across all malignant cells
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.stats import spearmanr

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

from sklearn.neighbors import NearestNeighbors

import json
import warnings
import gc

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


RAW_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23A_MALIGNANT_RAW_COUNTS.h5ad"
)

AUDIT_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D2_LINEAGE_CONTAMINATION_AUDIT.h5ad"
)


assert RAW_INPUT.exists()
assert AUDIT_INPUT.exists()


adata_raw_all = sc.read_h5ad(
    RAW_INPUT
)

adata_audit = sc.read_h5ad(
    AUDIT_INPUT
)


OLD_STATE_KEY = "state_leiden_r0_4"


print("=" * 110)
print("STEP23D-3 LINEAGE-CLEAN MALIGNANT-STATE RECONSTRUCTION")
print("=" * 110)

print(
    "All malignant cells:",
    f"{adata_raw_all.n_obs:,}"
)

print(
    "Previous discovery cells:",
    f"{adata_audit.n_obs:,}"
)


# ============================================================
# Initialize status
# ============================================================

adata_raw_all.obs[
    "state_source_cluster_r0_4"
] = pd.NA


adata_raw_all.obs[
    "state_analysis_eligible"
] = False


adata_raw_all.obs[
    "state_analysis_exclusion_reason"
] = (
    "Projection_only_insufficient_patient_support"
)


# ============================================================
# Cells that entered previous Step23 discovery
# ============================================================

common_cells = (
    adata_raw_all.obs_names
    .intersection(
        adata_audit.obs_names
    )
)


adata_raw_all.obs.loc[
    common_cells,
    "state_source_cluster_r0_4"
] = (

    adata_audit.obs.loc[
        common_cells,
        OLD_STATE_KEY
    ]
    .astype(str)
    .values
)


adata_raw_all.obs.loc[
    common_cells,
    "state_analysis_eligible"
] = True


adata_raw_all.obs.loc[
    common_cells,
    "state_analysis_exclusion_reason"
] = "Eligible_after_lineage_audit"


# ============================================================
# Candidate 5 exclusion
# ============================================================

candidate5_cells = (
    adata_audit.obs_names[
        adata_audit.obs[
            OLD_STATE_KEY
        ]
        .astype(str)
        ==
        "5"
    ]
)


adata_raw_all.obs.loc[
    candidate5_cells,
    "state_analysis_eligible"
] = False


adata_raw_all.obs.loc[
    candidate5_cells,
    "state_analysis_exclusion_reason"
] = (
    "Mixed_myeloid_plasma_malignant_lineage"
)


# ============================================================
# Candidate 6 exclusion
# ============================================================

candidate6_cells = (
    adata_audit.obs_names[
        adata_audit.obs[
            OLD_STATE_KEY
        ]
        .astype(str)
        ==
        "6"
    ]
)


adata_raw_all.obs.loc[
    candidate6_cells,
    "state_analysis_eligible"
] = False


adata_raw_all.obs.loc[
    candidate6_cells,
    "state_analysis_exclusion_reason"
] = (
    "Mixed_Tcell_malignant_lineage"
)


# ============================================================
# Audit
# ============================================================

eligibility_summary = (

    adata_raw_all.obs[
        "state_analysis_exclusion_reason"
    ]
    .value_counts()
    .rename_axis(
        "Reason"
    )
    .reset_index(
        name="N_cells"
    )
)


eligibility_summary[
    "Pct_all_malignant"
] = (

    eligibility_summary[
        "N_cells"
    ]
    /
    adata_raw_all.n_obs
    *
    100
)


print("\nEligibility summary:")

display(
    eligibility_summary
    .round(3)
)


print(
    "\nEligible cells:",
    int(
        adata_raw_all.obs[
            "state_analysis_eligible"
        ].sum()
    )
)


assert (
    adata_raw_all.obs[
        "state_analysis_eligible"
    ].sum()
    ==
    12057
), (
    "Expected 12,057 lineage-clean discovery cells."
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 142
# Save eligibility and lineage-clean raw counts
# ============================================================

ELIGIBILITY_PATH = (

    RESULTS_DIR /
    "Step23D3_StateAnalysis_Eligibility.csv"
)


eligibility_columns = [

    "patient",
    "sample",
    "site",

    "state_source_cluster_r0_4",

    "state_analysis_eligible",
    "state_analysis_exclusion_reason",

    "cnv_leiden_r0_4",
    "cnv_burden_primary",

    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt"
]


eligibility_columns = [

    col
    for col in eligibility_columns
    if col in adata_raw_all.obs.columns
]


eligibility_table = (
    adata_raw_all.obs[
        eligibility_columns
    ]
    .copy()
)


eligibility_table.index.name = "Cell"


eligibility_table.to_csv(
    ELIGIBILITY_PATH,
    encoding="utf-8-sig"
)


# ============================================================
# Clean discovery object
# ============================================================

eligible_cells = (
    adata_raw_all.obs_names[
        adata_raw_all.obs[
            "state_analysis_eligible"
        ]
    ]
)


adata_clean_raw = (

    adata_raw_all[
        eligible_cells,
        :
    ]
    .copy()
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    "state_source_cluster_r0_4"
]:

    if col in adata_clean_raw.obs.columns:

        adata_clean_raw.obs[col] = (
            adata_clean_raw.obs[col]
            .astype(str)
            .astype("category")
        )


CLEAN_RAW_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)


adata_clean_raw.write_h5ad(
    CLEAN_RAW_OUTPUT,
    compression="gzip"
)


print("=" * 100)
print("LINEAGE-CLEAN RAW DATASET")
print("=" * 100)

print(
    "Shape:",
    adata_clean_raw.shape
)

print(
    "Saved:",
    CLEAN_RAW_OUTPUT
)


print("\nCells by patient:")

display(
    adata_clean_raw.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 143
# Re-normalize lineage-clean malignant cells
# ============================================================

# ------------------------------------------------------------
# Verify counts
# ------------------------------------------------------------

X = adata_clean_raw.X


if sparse.issparse(X):

    nonzero = X.data

else:

    nonzero = np.asarray(
        X
    ).ravel()

    nonzero = nonzero[
        nonzero != 0
    ]


rng = np.random.default_rng(
    SEED
)


if len(nonzero) > 200000:

    values = rng.choice(
        nonzero,
        size=200000,
        replace=False
    )

else:

    values = nonzero


integer_fraction = float(

    np.mean(

        np.isclose(
            values,
            np.round(values),
            atol=1e-6
        )

    )

)


print(
    "Raw integer-like fraction:",
    round(integer_fraction, 6)
)


assert integer_fraction > 0.999


# ============================================================
# Normalize
# ============================================================

adata_clean = (
    adata_clean_raw.copy()
)


sc.pp.normalize_total(

    adata_clean,

    target_sum=1e4
)


sc.pp.log1p(
    adata_clean
)


CLEAN_LOG_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_LOGNORM_ALL_GENES.h5ad"
)


adata_clean.write_h5ad(
    CLEAN_LOG_OUTPUT,
    compression="gzip"
)


print(
    "Saved:",
    CLEAN_LOG_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 144
# Recompute patient-aware HVGs
# ============================================================

N_TOP_HVG = 3000
MIN_PATIENT_CELLS_FOR_HVG = 20


patient_n = (

    adata_clean.obs[
        "patient"
    ]
    .astype(str)
    .value_counts()
)


hvg_patients_clean = (

    patient_n[
        patient_n
        >=
        MIN_PATIENT_CELLS_FOR_HVG
    ]
    .index
    .tolist()
)


print(
    "Patients included in HVG fitting:",
    hvg_patients_clean
)


hvg_mask = (

    adata_clean.obs[
        "patient"
    ]
    .astype(str)
    .isin(
        hvg_patients_clean
    )
)


adata_hvg_fit = (

    adata_clean[
        hvg_mask,
        :
    ]
    .copy()
)


sc.pp.highly_variable_genes(

    adata_hvg_fit,

    flavor="seurat",

    n_top_genes=
        N_TOP_HVG,

    batch_key=
        "patient",

    inplace=True
)


adata_clean.var[
    "highly_variable"
] = False


selected_hvg_genes = (

    adata_hvg_fit.var_names[
        adata_hvg_fit.var[
            "highly_variable"
        ]
    ]
    .tolist()
)


adata_clean.var.loc[
    selected_hvg_genes,
    "highly_variable"
] = True


for col in [

    "means",
    "dispersions",
    "dispersions_norm",
    "highly_variable_nbatches",
    "highly_variable_intersection"

]:

    if col in adata_hvg_fit.var.columns:

        adata_clean.var[col] = (

            adata_hvg_fit.var[
                col
            ]
            .reindex(
                adata_clean.var_names
            )
            .values

        )


# ============================================================
# Remove mitochondrial HVGs only from PCA
# ============================================================

is_mt = (
    adata_clean.var_names
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


adata_clean.var[
    "highly_variable_for_pca"
] = (

    adata_clean.var[
        "highly_variable"
    ].astype(bool)

    &

    (~np.asarray(is_mt))
)


print(
    "HVG:",
    int(
        adata_clean.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "PCA genes:",
    int(
        adata_clean.var[
            "highly_variable_for_pca"
        ].sum()
    )
)


# ============================================================
# Cross-patient HVG support
# ============================================================

if (
    "highly_variable_nbatches"
    in adata_clean.var.columns
):

    hvg_support_clean = (

        adata_clean.var.loc[
            adata_clean.var[
                "highly_variable"
            ],
            "highly_variable_nbatches"
        ]

        .value_counts()

        .sort_index()

        .rename_axis(
            "N_patients_HVG"
        )

        .reset_index(
            name="N_genes"
        )
    )

else:

    hvg_support_clean = pd.DataFrame()


display(
    hvg_support_clean
)


adata_clean.var.to_csv(

    RESULTS_DIR /
    "Step23D3_HVG_Full_Table.csv",

    encoding="utf-8-sig"
)


hvg_support_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_HVG_CrossPatient_Support.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 145
# Rebuild lineage-clean PCA
# ============================================================

pca_genes_clean = (

    adata_clean.var_names[
        adata_clean.var[
            "highly_variable_for_pca"
        ]
    ]
    .tolist()
)


adata_pca_clean = (

    adata_clean[
        :,
        pca_genes_clean
    ]
    .copy()
)


sc.pp.scale(

    adata_pca_clean,

    max_value=10
)


sc.tl.pca(

    adata_pca_clean,

    n_comps=50,

    svd_solver="arpack",

    random_state=
        SEED
)


adata_clean.obsm[
    "X_pca_lineage_clean_unintegrated"
] = (

    adata_pca_clean.obsm[
        "X_pca"
    ]
    .copy()
)


variance_ratio = (

    adata_pca_clean.uns[
        "pca"
    ][
        "variance_ratio"
    ]
)


pca_variance_clean = pd.DataFrame({

    "PC":
        np.arange(
            1,
            len(variance_ratio) + 1
        ),

    "VarianceRatio":
        variance_ratio
})


pca_variance_clean[
    "CumulativeVariance"
] = (

    pca_variance_clean[
        "VarianceRatio"
    ]
    .cumsum()
)


pca_variance_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_PCA_Variance.csv",

    index=False,
    encoding="utf-8-sig"
)


print(
    "PCA representation:",
    adata_clean.obsm[
        "X_pca_lineage_clean_unintegrated"
    ].shape
)


display(
    pca_variance_clean
    .head(20)
    .round(5)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 146
# Re-audit PCA × QC correlations
# ============================================================

X_pca_clean = (

    adata_clean.obsm[
        "X_pca_lineage_clean_unintegrated"
    ][
        :,
        :30
    ]
)


qc_vars = [

    col
    for col in [

        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt"

    ]

    if col in adata_clean.obs.columns
]


qc_rows = []


for pc_idx in range(20):

    pc_values = (
        X_pca_clean[
            :,
            pc_idx
        ]
    )


    for variable in qc_vars:

        values = (

            adata_clean.obs[
                variable
            ]
            .astype(float)
            .values
        )


        rho, p = spearmanr(

            pc_values,
            values
        )


        qc_rows.append({

            "PC":
                pc_idx + 1,

            "QC_variable":
                variable,

            "Spearman_rho":
                rho,

            "Abs_rho":
                abs(rho),

            "P_value":
                p

        })


pc_qc_clean = pd.DataFrame(
    qc_rows
)


pc_qc_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_PC_QC_Correlations.csv",

    index=False,
    encoding="utf-8-sig"
)


qc_max_clean = (

    pc_qc_clean

    .sort_values(
        "Abs_rho",
        ascending=False
    )

    .groupby(
        "QC_variable",
        as_index=False
    )

    .first()
)


print("=" * 100)
print("LINEAGE-CLEAN PCA × QC")
print("=" * 100)


display(
    qc_max_clean
    .round(4)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 147
# Save clean pre-Harmony checkpoint
# ============================================================

PREHARMONY_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_PREHARMONY.h5ad"
)


adata_clean.write_h5ad(

    PREHARMONY_OUTPUT,

    compression="gzip"
)


print(
    "Saved:",
    PREHARMONY_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 148
# Re-run Harmony on lineage-clean malignant cells
# ============================================================

N_PCS_HARMONY = 30


adata_clean.obsm[
    "X_pca_for_harmony_clean"
] = (

    adata_clean.obsm[
        "X_pca_lineage_clean_unintegrated"
    ][
        :,
        :N_PCS_HARMONY
    ]
    .copy()
)


np.random.seed(
    SEED
)


sc.external.pp.harmony_integrate(

    adata_clean,

    key=
        "patient",

    basis=
        "X_pca_for_harmony_clean",

    adjusted_basis=
        "X_pca_harmony_lineage_clean"
)


print(
    "Harmony representation:",
    adata_clean.obsm[
        "X_pca_harmony_lineage_clean"
    ].shape
)


HARMONY_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_HARMONY_CHECKPOINT.h5ad"
)


adata_clean.write_h5ad(

    HARMONY_OUTPUT,

    compression="gzip"
)


print(
    "Saved:",
    HARMONY_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 149
# Build lineage-clean Harmony graph
# ============================================================

N_NEIGHBORS = 15


sc.pp.neighbors(

    adata_clean,

    use_rep=
        "X_pca_harmony_lineage_clean",

    n_pcs=
        30,

    n_neighbors=
        N_NEIGHBORS,

    key_added=
        "neighbors_lineage_clean",

    random_state=
        SEED
)


sc.tl.umap(

    adata_clean,

    neighbors_key=
        "neighbors_lineage_clean",

    min_dist=
        0.5,

    random_state=
        SEED
)


adata_clean.obsm[
    "X_umap_lineage_clean"
] = (

    adata_clean.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "UMAP:",
    adata_clean.obsm[
        "X_umap_lineage_clean"
    ].shape
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 150
# Integration audit
# ============================================================

X_pre = (

    adata_clean.obsm[
        "X_pca_lineage_clean_unintegrated"
    ][
        :,
        :30
    ]
)


X_post = (

    adata_clean.obsm[
        "X_pca_harmony_lineage_clean"
    ][
        :,
        :30
    ]
)


integration_rows = []


for variable in [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"

]:

    if variable not in adata_clean.obs.columns:
        continue


    labels = (

        adata_clean.obs[
            variable
        ]
        .astype(str)
        .values
    )


    for representation, matrix in [

        (
            "Unintegrated",
            X_pre
        ),

        (
            "Harmony_patient",
            X_post
        )

    ]:

        score = silhouette_score(

            matrix,

            labels,

            sample_size=
                min(
                    3000,
                    len(labels)
                ),

            random_state=
                SEED
        )


        integration_rows.append({

            "Representation":
                representation,

            "Variable":
                variable,

            "Silhouette":
                score

        })


integration_silhouette_clean = pd.DataFrame(
    integration_rows
)


integration_silhouette_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_Integration_Silhouette.csv",

    index=False,
    encoding="utf-8-sig"
)


display(

    integration_silhouette_clean

    .pivot(
        index="Variable",
        columns="Representation",
        values="Silhouette"
    )

    .round(4)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 151
# kNN purity before and after Harmony
# ============================================================

def calculate_knn_purity(
    matrix,
    labels,
    k=15
):

    model = NearestNeighbors(

        n_neighbors=
            k + 1,

        metric="euclidean",

        n_jobs=-1
    )


    model.fit(
        matrix
    )


    _, indices = model.kneighbors(
        matrix
    )


    indices = (
        indices[
            :,
            1:
        ]
    )


    labels = np.asarray(
        labels
    )


    neighbor_labels = (
        labels[
            indices
        ]
    )


    purity = np.mean(

        neighbor_labels

        ==

        labels[
            :,
            None
        ],

        axis=1
    )


    return purity


purity_rows = []


for representation, matrix in [

    ("Unintegrated", X_pre),
    ("Harmony_patient", X_post)

]:

    for variable in [

        "patient",
        "cnv_leiden_r0_4",
        "site"

    ]:

        labels = (
            adata_clean.obs[
                variable
            ]
            .astype(str)
            .values
        )


        purity = calculate_knn_purity(

            matrix,
            labels,
            k=15
        )


        proportions = (

            pd.Series(
                labels
            )
            .value_counts(
                normalize=True
            )
            .values
        )


        random_expectation = float(

            np.sum(
                proportions ** 2
            )
        )


        purity_rows.append({

            "Representation":
                representation,

            "Variable":
                variable,

            "Mean_purity":
                float(
                    np.mean(purity)
                ),

            "Median_purity":
                float(
                    np.median(purity)
                ),

            "Expected_random_purity":
                random_expectation,

            "Observed_to_random_ratio":
                float(
                    np.mean(purity)
                    /
                    random_expectation
                )
        })


integration_purity_clean = pd.DataFrame(
    purity_rows
)


integration_purity_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_Integration_kNN15_Purity.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    integration_purity_clean
    .round(4)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 152
# Within-patient site preservation
# ============================================================

MIN_SITE_CELLS = 20


site_rows = []


for patient in sorted(

    adata_clean.obs[
        "patient"
    ]
    .astype(str)
    .unique()

):

    patient_mask = (

        adata_clean.obs[
            "patient"
        ]
        .astype(str)
        ==
        patient
    )


    site_n = (

        adata_clean.obs.loc[
            patient_mask,
            "site"
        ]

        .astype(str)

        .value_counts()
    )


    eligible_sites = (

        site_n[
            site_n >= MIN_SITE_CELLS
        ]
        .index
        .tolist()
    )


    if len(
        eligible_sites
    ) < 2:

        continue


    mask = (

        patient_mask

        &

        adata_clean.obs[
            "site"
        ]
        .astype(str)
        .isin(
            eligible_sites
        )
    )


    idx = np.where(
        mask.values
    )[0]


    labels = (

        adata_clean.obs.loc[
            mask,
            "site"
        ]

        .astype(str)
        .values
    )


    for representation, matrix in [

        ("Unintegrated", X_pre),
        ("Harmony_patient", X_post)

    ]:

        score = silhouette_score(

            matrix[
                idx,
                :
            ],

            labels,

            sample_size=
                min(
                    2000,
                    len(labels)
                ),

            random_state=
                SEED
        )


        site_rows.append({

            "Patient":
                patient,

            "Sites":
                "|".join(
                    eligible_sites
                ),

            "Representation":
                representation,

            "N_cells":
                len(labels),

            "Site_silhouette":
                score

        })


site_preservation_clean = pd.DataFrame(
    site_rows
)


site_preservation_clean.to_csv(

    RESULTS_DIR /
    "Step23D3_WithinPatient_Site_Preservation.csv",

    index=False,
    encoding="utf-8-sig"
)


display(

    site_preservation_clean

    .pivot_table(

        index=[
            "Patient",
            "Sites"
        ],

        columns=
            "Representation",

        values=
            "Site_silhouette"
    )

    .round(4)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 153
# Publication UMAPs
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False


NPG_COLORS = [

    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85"
]


SITE_COLORS = {

    "Normal":
        "#4DBBD5",

    "Tumor":
        "#E64B35",

    "PVTT":
        "#00A087",

    "Lymph":
        "#3C5488"
}


def save_publication_figure(
    fig,
    filename
):

    fig.savefig(

        FIGURES_DIR /
        f"{filename}.png",

        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(

        FIGURES_DIR /
        f"{filename}.pdf",

        bbox_inches="tight",
        facecolor="white"
    )


plot_specs = [

    (
        "patient",
        "Lineage-clean Malignant-cell Space by Patient",
        "Step23D3_01_Clean_UMAP_ByPatient"
    ),

    (
        "site",
        "Lineage-clean Malignant-cell Space by Sampling Site",
        "Step23D3_02_Clean_UMAP_BySite"
    ),

    (
        "cnv_leiden_r0_4",
        "Lineage-clean Malignant-cell Space by CNV Clone",
        "Step23D3_03_Clean_UMAP_ByCNVClone"
    )
]


for variable, title, filename in plot_specs:

    adata_clean.obs[
        variable
    ] = (

        adata_clean.obs[
            variable
        ]
        .astype(str)
        .astype("category")
    )


    categories = (

        adata_clean.obs[
            variable
        ]
        .cat.categories
        .astype(str)
        .tolist()
    )


    if variable == "site":

        palette = {

            category:
                SITE_COLORS.get(
                    category,
                    "#BDBDBD"
                )

            for category in categories
        }

    else:

        palette = {

            category:
                NPG_COLORS[
                    i %
                    len(NPG_COLORS)
                ]

            for i, category
            in enumerate(
                categories
            )
        }


    ax = sc.pl.embedding(

        adata_clean,

        basis=
            "umap_lineage_clean",

        color=
            variable,

        palette=
            palette,

        size=8,

        alpha=0.80,

        frameon=False,

        legend_loc=
            "right margin",

        title=
            title,

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        8.3,
        6.5
    )

    fig.tight_layout()


    save_publication_figure(
        fig,
        filename
    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 154
# Multi-resolution clustering after lineage cleanup
# ============================================================

STATE_RESOLUTIONS = [

    0.2,
    0.4,
    0.6,
    0.8,
    1.0

]


resolution_rows = []


for res in STATE_RESOLUTIONS:

    key = (

        "state_clean_leiden_r"
        +
        str(res)
        .replace(
            ".",
            "_"
        )

    )


    sc.tl.leiden(

        adata_clean,

        resolution=
            res,

        neighbors_key=
            "neighbors_lineage_clean",

        key_added=
            key,

        random_state=
            SEED,

        flavor="igraph",

        n_iterations=2,

        directed=False
    )


    sizes = (

        adata_clean.obs[
            key
        ]
        .value_counts()
    )


    resolution_rows.append({

        "Resolution":
            res,

        "Cluster_column":
            key,

        "N_clusters":
            len(sizes),

        "Smallest_cluster":
            int(
                sizes.min()
            ),

        "Largest_cluster":
            int(
                sizes.max()
            ),

        "Median_cluster_size":
            float(
                sizes.median()
            ),

        "Clusters_lt50":
            int(
                (
                    sizes < 50
                ).sum()
            ),

        "Clusters_lt100":
            int(
                (
                    sizes < 100
                ).sum()
            )

    })


clean_resolution_summary = pd.DataFrame(
    resolution_rows
)


clean_resolution_summary.to_csv(

    RESULTS_DIR /
    "Step23D3_Clean_State_Resolution_Summary.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    clean_resolution_summary
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 155
# Adjacent-resolution ARI
# ============================================================

ari_rows = []


for r1, r2 in [

    (0.2, 0.4),
    (0.4, 0.6),
    (0.6, 0.8),
    (0.8, 1.0)

]:

    key1 = (
        "state_clean_leiden_r"
        +
        str(r1).replace(
            ".",
            "_"
        )
    )


    key2 = (
        "state_clean_leiden_r"
        +
        str(r2).replace(
            ".",
            "_"
        )
    )


    ari = adjusted_rand_score(

        adata_clean.obs[
            key1
        ].astype(str),

        adata_clean.obs[
            key2
        ].astype(str)

    )


    ari_rows.append({

        "Resolution_1":
            r1,

        "Resolution_2":
            r2,

        "ARI":
            ari
    })


clean_resolution_ari = pd.DataFrame(
    ari_rows
)


clean_resolution_ari.to_csv(

    RESULTS_DIR /
    "Step23D3_Clean_State_Resolution_ARI.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    clean_resolution_ari
    .round(4)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 156
# Patient composition across resolutions
# ============================================================

composition_rows = []


for res in STATE_RESOLUTIONS:

    key = (
        "state_clean_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )


    for cluster in sorted(

        adata_clean.obs[
            key
        ]
        .astype(str)
        .unique(),

        key=lambda x:
            int(x)

    ):

        mask = (

            adata_clean.obs[
                key
            ]
            .astype(str)
            ==
            cluster

        )


        temp = (
            adata_clean.obs.loc[
                mask
            ]
        )


        patient_counts = (

            temp[
                "patient"
            ]
            .astype(str)
            .value_counts()
        )


        n = len(temp)


        row = {

            "Resolution":
                res,

            "Cluster":
                cluster,

            "N_cells":
                n,

            "N_patients_present":
                int(
                    (
                        patient_counts > 0
                    ).sum()
                ),

            "N_patients_ge20":
                int(
                    (
                        patient_counts >= 20
                    ).sum()
                ),

            "N_patients_ge50":
                int(
                    (
                        patient_counts >= 50
                    ).sum()
                ),

            "Dominant_patient":
                patient_counts.index[0],

            "Dominant_patient_pct":
                float(
                    patient_counts.iloc[0]
                    /
                    n
                    *
                    100
                )
        }


        for patient in sorted(

            adata_clean.obs[
                "patient"
            ]
            .astype(str)
            .unique()

        ):

            row[
                f"N_{patient}"
            ] = int(
                patient_counts.get(
                    patient,
                    0
                )
            )


        composition_rows.append(
            row
        )


clean_state_composition = pd.DataFrame(
    composition_rows
)


clean_state_composition.to_csv(

    RESULTS_DIR /
    "Step23D3_Clean_State_Composition_AllResolutions.csv",

    index=False,
    encoding="utf-8-sig"
)


print("Resolution 0.4:")

display(

    clean_state_composition[
        clean_state_composition[
            "Resolution"
        ]
        ==
        0.4
    ]
    .round(2)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 157
# Old primary vs lineage-clean r0.4
# ============================================================

NEW_STATE_KEY = (
    "state_clean_leiden_r0_4"
)


old_labels = (

    adata_clean.obs[
        "state_source_cluster_r0_4"
    ]
    .astype(str)
)


new_labels = (

    adata_clean.obs[
        NEW_STATE_KEY
    ]
    .astype(str)
)


old_vs_clean_ari = adjusted_rand_score(

    old_labels,
    new_labels
)


old_clean_counts = pd.crosstab(

    old_labels,
    new_labels
)


old_clean_pct = (

    old_clean_counts

    .div(
        old_clean_counts.sum(
            axis=1
        ),
        axis=0
    )

    *
    100
)


print("=" * 105)
print("OLD vs LINEAGE-CLEAN r0.4")
print("=" * 105)

print(
    "ARI:",
    round(
        old_vs_clean_ari,
        4
    )
)


display(
    old_clean_pct
    .round(2)
)


old_clean_counts.to_csv(

    RESULTS_DIR /
    "Step23D3_Old_vs_Clean_r0_4_Counts.csv",

    encoding="utf-8-sig"
)


old_clean_pct.to_csv(

    RESULTS_DIR /
    "Step23D3_Old_vs_Clean_r0_4_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 158
# Re-audit lineage programs after reconstruction
# ============================================================

LINEAGE_PROGRAMS = {

    "T_NK_program": [

        "CD3D",
        "CD3E",
        "TRAC",
        "CD2",
        "IL7R",
        "CCL5",
        "GZMK",
        "GZMA",
        "NKG7",
        "GNLY"

    ],

    "B_Plasma_program": [

        "CD79A",
        "MS4A1",
        "CD37",
        "JCHAIN",
        "MZB1",
        "IGKC",
        "IGLC2",
        "TNFRSF17",
        "SDC1"

    ],

    "Myeloid_program": [

        "LST1",
        "TYROBP",
        "FCER1G",
        "CTSS",
        "AIF1",
        "C1QC",
        "FCGR3A",
        "LILRB1",
        "MSR1"

    ],

    "Epithelial_program": [

        "EPCAM",
        "KRT7",
        "KRT8",
        "KRT18",
        "KRT19",
        "KRT17",
        "TACSTD2"

    ],

    "Hepatic_program": [

        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "HPD",
        "CPS1",
        "CYP8B1",
        "MAT1A"

    ],

    "Tumor_epithelial_program": [

        "GPC3",
        "AFP",
        "S100P",
        "SPINK1",
        "AGR2",
        "CLDN6",
        "PRAME",
        "KRT19",
        "EPCAM"

    ]
}


for score_name, genes in LINEAGE_PROGRAMS.items():

    detected = [

        gene
        for gene in genes
        if gene in adata_clean.var_names

    ]


    sc.tl.score_genes(

        adata_clean,

        gene_list=
            detected,

        score_name=
            score_name,

        ctrl_size=
            min(
                50,
                max(
                    10,
                    len(detected) * 5
                )
            ),

        n_bins=25,

        random_state=
            SEED,

        use_raw=False
    )


score_columns = list(
    LINEAGE_PROGRAMS.keys()
)


clean_lineage_summary = (

    adata_clean.obs

    .groupby(
        NEW_STATE_KEY,
        observed=True
    )[
        score_columns
    ]

    .median()

    .reset_index()
)


clean_lineage_summary.to_csv(

    RESULTS_DIR /
    "Step23D3_Clean_r0_4_LineageProgram_Summary.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    clean_lineage_summary
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 159
# Lineage program heatmap
# ============================================================

heatmap_df = (

    clean_lineage_summary

    .set_index(
        NEW_STATE_KEY
    )[
        score_columns
    ]
)


heatmap_z = (

    heatmap_df
    -
    heatmap_df.mean(
        axis=0
    )

) / (

    heatmap_df.std(
        axis=0
    )
    +
    1e-8

)


fig, ax = plt.subplots(
    figsize=(8.3, 6.4)
)


im = ax.imshow(

    heatmap_z.values,

    aspect="auto",

    cmap="coolwarm",

    vmin=-2,
    vmax=2
)


ax.set_xticks(
    np.arange(
        heatmap_z.shape[1]
    )
)


ax.set_xticklabels(

    [

        col.replace(
            "_program",
            ""
        )
        .replace(
            "_",
            " "
        )

        for col in heatmap_z.columns
    ],

    rotation=45,
    ha="right"
)


ax.set_yticks(
    np.arange(
        heatmap_z.shape[0]
    )
)


ax.set_yticklabels(
    heatmap_z.index
)


ax.set_xlabel(
    "Lineage Program"
)

ax.set_ylabel(
    "Lineage-clean State Candidate"
)

ax.set_title(
    "Lineage Program Audit After Mixed-lineage Exclusion"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Z-scored Median Program Score"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D3_04_Clean_State_LineageProgram_Heatmap"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 160
# Clean r0.4 candidate-state UMAP
# ============================================================

ax = sc.pl.embedding(

    adata_clean,

    basis=
        "umap_lineage_clean",

    color=
        NEW_STATE_KEY,

    size=8,

    alpha=0.82,

    frameon=False,

    legend_loc=
        "on data",

    legend_fontsize=8,

    title=
        "Lineage-clean Malignant State Candidates (Leiden r=0.4)",

    show=False
)


fig = ax.figure

fig.set_size_inches(
    7.8,
    6.5
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D3_05_Clean_StateCandidates_r0_4"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 161
# Save reconstructed lineage-clean object
# ============================================================

CLEAN_STATE_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_STATE_CANDIDATES.h5ad"
)


adata_clean.write_h5ad(

    CLEAN_STATE_OUTPUT,

    compression="gzip"
)


print("=" * 105)
print("LINEAGE-CLEAN STATE OBJECT SAVED")
print("=" * 105)

print(
    CLEAN_STATE_OUTPUT
)

print(
    "Shape:",
    adata_clean.shape
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 162
# Decision record
# ============================================================

step23d3_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23D3_LineageClean_State_Reconstruction",

    "Status":
        "RECONSTRUCTION_COMPLETE",

    "Original_high_confidence_malignant_cells":
        12406,

    "Projection_only_cells":
        3,

    "Mixed_lineage_candidate5_cells":
        225,

    "Mixed_lineage_candidate6_cells":
        121,

    "Lineage_clean_state_discovery_cells":
        int(
            adata_clean.n_obs
        ),

    "Historical_malignancy_labels_modified":
        False,

    "State_analysis_exclusion_logic": {

        "Candidate_5":
            (
                "Mixed myeloid/plasma-associated and malignant "
                "lineage expression; excluded only from malignant "
                "state construction."
            ),

        "Candidate_6":
            (
                "Strong T-cell lineage expression combined with "
                "malignant/hepatic expression; excluded only from "
                "malignant state construction."
            )
    },

    "Reconstruction":
        (
            "Rebuilt from raw counts after lineage exclusion; "
            "normalization, patient-aware HVG, PCA, Harmony(patient), "
            "neighbors, UMAP and Leiden were all recomputed."
        ),

    "Integration_variables_corrected": [
        "patient"
    ],

    "Variables_not_corrected": [
        "site",
        "CNV clone"
    ],

    "Primary_resolution":
        "PENDING_REVIEW",

    "Old_vs_clean_r0_4_ARI":
        float(
            old_vs_clean_ari
        ),

    "Final_state_annotation":
        "NOT_PERFORMED",

    "Next_step":
        (
            "Review lineage-clean state structure and resolution; "
            "then perform final marker/pathway characterization "
            "and consensus state consolidation."
        ),

    "Output":
        str(
            CLEAN_STATE_OUTPUT
        )
}


SUMMARY_D3 = (

    LOGS_DIR /
    "Step23D3_Final_Summary.json"
)


with open(

    SUMMARY_D3,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23d3_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_D3
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 163
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_D3 = (

    RESULTS_DIR /
    "Step23D3_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_D3,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_D3
)

In [ ]:
# ============================================================
# STEP23D-3 - CELL 164
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 120)
print("STEP23D-3 FINAL REVIEW")
print("=" * 120)


# ============================================================
# 1. Eligibility
# ============================================================

print("\n[1] State-analysis eligibility")
print("-" * 120)

display(
    eligibility_summary
    .round(3)
)


print(
    "\nLineage-clean cells:",
    f"{adata_clean.n_obs:,}"
)


# ============================================================
# 2. Patient composition
# ============================================================

print("\n[2] Lineage-clean cells by patient")
print("-" * 120)

display(

    adata_clean.obs[
        "patient"
    ]

    .value_counts()

    .to_frame(
        "N_cells"
    )
)


# ============================================================
# 3. HVG
# ============================================================

print("\n[3] Patient-aware HVG support")
print("-" * 120)

print(
    "HVG:",
    int(
        adata_clean.var[
            "highly_variable"
        ].sum()
    )
)

print(
    "PCA genes:",
    int(
        adata_clean.var[
            "highly_variable_for_pca"
        ].sum()
    )
)

display(
    hvg_support_clean
)


# ============================================================
# 4. QC-PC relationship
# ============================================================

print("\n[4] Maximum QC association with first 20 PCs")
print("-" * 120)

display(
    qc_max_clean
    .round(4)
)


# ============================================================
# 5. Integration silhouette
# ============================================================

print("\n[5] Pre/Post Harmony silhouette")
print("-" * 120)

display(

    integration_silhouette_clean

    .pivot(
        index="Variable",
        columns="Representation",
        values="Silhouette"
    )

    .round(4)
)


# ============================================================
# 6. kNN purity
# ============================================================

print("\n[6] Pre/Post Harmony kNN purity")
print("-" * 120)

display(
    integration_purity_clean
    .round(4)
)


# ============================================================
# 7. Site preservation
# ============================================================

print("\n[7] Within-patient site preservation")
print("-" * 120)

display(

    site_preservation_clean

    .pivot_table(

        index=[
            "Patient",
            "Sites"
        ],

        columns=
            "Representation",

        values=
            "Site_silhouette"
    )

    .round(4)
)


# ============================================================
# 8. Resolution summary
# ============================================================

print("\n[8] Clean Leiden resolution summary")
print("-" * 120)

display(
    clean_resolution_summary
)


# ============================================================
# 9. Resolution ARI
# ============================================================

print("\n[9] Adjacent-resolution ARI")
print("-" * 120)

display(
    clean_resolution_ari
    .round(4)
)


# ============================================================
# 10. Old vs clean r0.4
# ============================================================

print("\n[10] Old vs lineage-clean r0.4")
print("-" * 120)

print(
    "ARI:",
    round(
        old_vs_clean_ari,
        4
    )
)

display(
    old_clean_pct
    .round(2)
)


# ============================================================
# 11. Clean r0.4 composition
# ============================================================

print("\n[11] Clean r0.4 patient composition")
print("-" * 120)

display(

    clean_state_composition[
        clean_state_composition[
            "Resolution"
        ]
        ==
        0.4
    ]

    .round(2)
)


# ============================================================
# 12. Lineage audit
# ============================================================

print("\n[12] Clean r0.4 lineage programs")
print("-" * 120)

display(
    clean_lineage_summary
    .round(3)
)


# ============================================================
# 13. Files
# ============================================================

print("\n[13] Files")
print("-" * 120)


for path in [

    ELIGIBILITY_PATH,

    CLEAN_RAW_OUTPUT,

    CLEAN_LOG_OUTPUT,

    PREHARMONY_OUTPUT,

    HARMONY_OUTPUT,

    CLEAN_STATE_OUTPUT,

    RESULTS_DIR /
    "Step23D3_HVG_Full_Table.csv",

    RESULTS_DIR /
    "Step23D3_PC_QC_Correlations.csv",

    RESULTS_DIR /
    "Step23D3_Integration_Silhouette.csv",

    RESULTS_DIR /
    "Step23D3_Integration_kNN15_Purity.csv",

    RESULTS_DIR /
    "Step23D3_Clean_State_Resolution_Summary.csv",

    RESULTS_DIR /
    "Step23D3_Clean_State_Resolution_ARI.csv",

    RESULTS_DIR /
    "Step23D3_Old_vs_Clean_r0_4_Percent.csv",

    RESULTS_DIR /
    "Step23D3_Clean_r0_4_LineageProgram_Summary.csv",

    SUMMARY_D3,

    MANIFEST_D3

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 120)

print(
    "STEP23D-3 STATUS = LINEAGE-CLEAN RECONSTRUCTION COMPLETE"
)

print(
    "PRIMARY CLEAN STATE RESOLUTION = PENDING REVIEW"
)

print(
    "FINAL BIOLOGICAL STATE ANNOTATION = NOT YET PERFORMED"
)

print(
    "NEXT = marker/pathway characterization + consensus consolidation"
)

print("=" * 120)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23D-4A - CELL 165
# Load lineage-clean malignant-state object
# ============================================================

from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np

from sklearn.metrics import adjusted_rand_score

import json
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_STATE_CANDIDATES.h5ad"
)

assert INPUT.exists()


adata_clean = sc.read_h5ad(
    INPUT
)


PRIMARY_KEY = (
    "state_clean_leiden_r0_4"
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    PRIMARY_KEY
]:

    if col in adata_clean.obs.columns:

        adata_clean.obs[col] = (
            adata_clean.obs[col]
            .astype(str)
            .astype("category")
        )


print("=" * 110)
print("STEP23D-4A FINAL CLEAN-STATE STABILITY LOCK")
print("=" * 110)

print(
    "Cells:",
    f"{adata_clean.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_clean.n_vars:,}"
)

print(
    "Primary r0.4 clusters:",
    adata_clean.obs[
        PRIMARY_KEY
    ].nunique()
)

print("\nPatients:")

display(
    adata_clean.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 166
# Clean no-PC4 Harmony sensitivity
# ============================================================

X_pca_30 = (
    adata_clean.obsm[
        "X_pca_lineage_clean_unintegrated"
    ][
        :,
        :30
    ]
)


# PC4 = Python column index 3
keep_pc_idx = [

    i
    for i in range(30)
    if i != 3
]


adata_clean.obsm[
    "X_pca_clean_noPC4"
] = (

    X_pca_30[
        :,
        keep_pc_idx
    ]
    .copy()
)


print(
    "Original Harmony input:",
    X_pca_30.shape
)

print(
    "No-PC4 input:",
    adata_clean.obsm[
        "X_pca_clean_noPC4"
    ].shape
)


np.random.seed(SEED)


sc.external.pp.harmony_integrate(

    adata_clean,

    key=
        "patient",

    basis=
        "X_pca_clean_noPC4",

    adjusted_basis=
        "X_pca_harmony_clean_noPC4"
)


print(
    "Harmony no-PC4:",
    adata_clean.obsm[
        "X_pca_harmony_clean_noPC4"
    ].shape
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 167
# Clean no-PC4 graph and clustering
# ============================================================

sc.pp.neighbors(

    adata_clean,

    use_rep=
        "X_pca_harmony_clean_noPC4",

    n_neighbors=
        15,

    n_pcs=
        29,

    key_added=
        "neighbors_clean_noPC4",

    random_state=
        SEED
)


sc.tl.leiden(

    adata_clean,

    resolution=
        0.4,

    neighbors_key=
        "neighbors_clean_noPC4",

    key_added=
        "state_clean_noPC4_r0_4",

    random_state=
        SEED,

    flavor=
        "igraph",

    n_iterations=
        2,

    directed=
        False
)


print(
    "No-PC4 clusters:",
    adata_clean.obs[
        "state_clean_noPC4_r0_4"
    ].nunique()
)


display(
    adata_clean.obs[
        "state_clean_noPC4_r0_4"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 168
# Primary vs clean no-PC4
# ============================================================

ari_clean_noPC4 = adjusted_rand_score(

    adata_clean.obs[
        PRIMARY_KEY
    ].astype(str),

    adata_clean.obs[
        "state_clean_noPC4_r0_4"
    ].astype(str)
)


clean_noPC4_counts = pd.crosstab(

    adata_clean.obs[
        PRIMARY_KEY
    ],

    adata_clean.obs[
        "state_clean_noPC4_r0_4"
    ]
)


clean_noPC4_pct = (

    clean_noPC4_counts

    .div(
        clean_noPC4_counts.sum(
            axis=1
        ),
        axis=0
    )

    *
    100
)


print("=" * 105)
print("CLEAN PRIMARY vs CLEAN NO-PC4")
print("=" * 105)

print(
    "ARI:",
    round(
        ari_clean_noPC4,
        4
    )
)


display(
    clean_noPC4_pct
    .round(2)
)


clean_noPC4_counts.to_csv(

    RESULTS_DIR /
    "Step23D4A_Primary_vs_NoPC4_Counts.csv",

    encoding="utf-8-sig"
)


clean_noPC4_pct.to_csv(

    RESULTS_DIR /
    "Step23D4A_Primary_vs_NoPC4_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 169
# Build lineage-clean patient-capped cohort
# ============================================================

PATIENT_CAP = 1500


rng = np.random.default_rng(
    SEED
)


selected_cells = []


for patient in sorted(

    adata_clean.obs[
        "patient"
    ]
    .astype(str)
    .unique()

):

    cells = (

        adata_clean.obs_names[
            adata_clean.obs[
                "patient"
            ]
            .astype(str)
            ==
            patient
        ]
        .tolist()
    )


    if len(cells) <= PATIENT_CAP:

        chosen = cells

    else:

        chosen = (

            rng.choice(

                cells,

                size=
                    PATIENT_CAP,

                replace=
                    False

            )

            .tolist()
        )


    selected_cells.extend(
        chosen
    )


adata_cap = (

    adata_clean[
        selected_cells,
        :
    ]
    .copy()
)


for col in [
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    PRIMARY_KEY
]:

    if col in adata_cap.obs.columns:

        adata_cap.obs[col] = (
            adata_cap.obs[col]
            .astype(str)
            .astype("category")
        )


print(
    "Capped cells:",
    f"{adata_cap.n_obs:,}"
)


display(
    adata_cap.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 170
# Save capped cell manifest
# ============================================================

cap_manifest_cols = [

    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4",
    PRIMARY_KEY

]


cap_manifest_cols = [

    col
    for col in cap_manifest_cols
    if col in adata_cap.obs.columns
]


cap_manifest = (
    adata_cap.obs[
        cap_manifest_cols
    ]
    .copy()
)


cap_manifest.index.name = "Cell"


CAP_MANIFEST_PATH = (

    RESULTS_DIR /
    "Step23D4A_Clean_PatientCapped_CellManifest.csv"
)


cap_manifest.to_csv(

    CAP_MANIFEST_PATH,

    encoding="utf-8-sig"
)


print(
    "Saved:",
    CAP_MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 171
# Re-run Harmony in clean capped cohort
# ============================================================

adata_cap.obsm[
    "X_pca_for_harmony_clean_capped"
] = (

    adata_cap.obsm[
        "X_pca_lineage_clean_unintegrated"
    ][
        :,
        :30
    ]
    .copy()
)


np.random.seed(
    SEED
)


sc.external.pp.harmony_integrate(

    adata_cap,

    key=
        "patient",

    basis=
        "X_pca_for_harmony_clean_capped",

    adjusted_basis=
        "X_pca_harmony_clean_capped"
)


print(
    "Capped Harmony:",
    adata_cap.obsm[
        "X_pca_harmony_clean_capped"
    ].shape
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 172
# Clean capped clustering
# ============================================================

sc.pp.neighbors(

    adata_cap,

    use_rep=
        "X_pca_harmony_clean_capped",

    n_neighbors=
        15,

    n_pcs=
        30,

    key_added=
        "neighbors_clean_capped",

    random_state=
        SEED
)


sc.tl.leiden(

    adata_cap,

    resolution=
        0.4,

    neighbors_key=
        "neighbors_clean_capped",

    key_added=
        "state_clean_capped_r0_4",

    random_state=
        SEED,

    flavor=
        "igraph",

    n_iterations=
        2,

    directed=
        False
)


print(
    "Capped clusters:",
    adata_cap.obs[
        "state_clean_capped_r0_4"
    ].nunique()
)


display(
    adata_cap.obs[
        "state_clean_capped_r0_4"
    ]
    .value_counts()
    .sort_index()
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 173
# Primary vs clean capped clustering
# ============================================================

ari_clean_capped = adjusted_rand_score(

    adata_cap.obs[
        PRIMARY_KEY
    ].astype(str),

    adata_cap.obs[
        "state_clean_capped_r0_4"
    ].astype(str)
)


clean_capped_counts = pd.crosstab(

    adata_cap.obs[
        PRIMARY_KEY
    ],

    adata_cap.obs[
        "state_clean_capped_r0_4"
    ]
)


clean_capped_pct = (

    clean_capped_counts

    .div(
        clean_capped_counts.sum(
            axis=1
        ),
        axis=0
    )

    *
    100
)


print("=" * 105)
print("CLEAN PRIMARY vs CLEAN PATIENT-CAPPED")
print("=" * 105)

print(
    "ARI:",
    round(
        ari_clean_capped,
        4
    )
)


display(
    clean_capped_pct
    .round(2)
)


clean_capped_counts.to_csv(

    RESULTS_DIR /
    "Step23D4A_Primary_vs_Capped_Counts.csv",

    encoding="utf-8-sig"
)


clean_capped_pct.to_csv(

    RESULTS_DIR /
    "Step23D4A_Primary_vs_Capped_Percent.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 174
# Cluster-level stability evidence
# ============================================================

clean_noPC4_pct.index = (
    clean_noPC4_pct.index
    .astype(str)
)


clean_capped_pct.index = (
    clean_capped_pct.index
    .astype(str)
)


primary_clusters = sorted(

    adata_clean.obs[
        PRIMARY_KEY
    ]
    .astype(str)
    .unique(),

    key=lambda x:
        int(x)
)


cluster_stability_rows = []


for cluster in primary_clusters:

    no_pc4_row = (
        clean_noPC4_pct.loc[
            cluster
        ]
    )


    capped_row = (
        clean_capped_pct.loc[
            cluster
        ]
    )


    cluster_stability_rows.append({

        "Primary_cluster":
            cluster,

        "NoPC4_best_match":
            str(
                no_pc4_row.idxmax()
            ),

        "NoPC4_best_match_pct":
            float(
                no_pc4_row.max()
            ),

        "Capped_best_match":
            str(
                capped_row.idxmax()
            ),

        "Capped_best_match_pct":
            float(
                capped_row.max()
            ),

        "Mean_best_match_pct":
            float(
                np.mean(
                    [
                        no_pc4_row.max(),
                        capped_row.max()
                    ]
                )
            )
    })


clean_cluster_stability = pd.DataFrame(
    cluster_stability_rows
)


clean_cluster_stability.to_csv(

    RESULTS_DIR /
    "Step23D4A_CleanCluster_Stability.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    clean_cluster_stability
    .round(2)
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 175
# Resolution mapping evidence
# ============================================================

resolution_stability_rows = []


for sensitivity_key, label in [

    (
        "state_clean_leiden_r0_2",
        "r0.2"
    ),

    (
        "state_clean_leiden_r0_6",
        "r0.6"
    )

]:

    ct = pd.crosstab(

        adata_clean.obs[
            PRIMARY_KEY
        ],

        adata_clean.obs[
            sensitivity_key
        ]
    )


    pct = (

        ct.div(
            ct.sum(
                axis=1
            ),
            axis=0
        )

        *
        100
    )


    pct.index = (
        pct.index
        .astype(str)
    )


    for cluster in primary_clusters:

        row = (
            pct.loc[
                cluster
            ]
        )


        resolution_stability_rows.append({

            "Primary_cluster":
                cluster,

            "Sensitivity":
                label,

            "Best_match":
                str(
                    row.idxmax()
                ),

            "Best_match_pct":
                float(
                    row.max()
                )
        })


resolution_stability = pd.DataFrame(
    resolution_stability_rows
)


display(
    resolution_stability
    .pivot(
        index=
            "Primary_cluster",

        columns=
            "Sensitivity",

        values=
            "Best_match_pct"
    )
    .round(2)
)


resolution_stability.to_csv(

    RESULTS_DIR /
    "Step23D4A_Resolution_Stability.csv",

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 176
# Final structural stability master table
# ============================================================

res_wide = (

    resolution_stability

    .pivot(

        index=
            "Primary_cluster",

        columns=
            "Sensitivity",

        values=
            "Best_match_pct"
    )

    .reset_index()

)


res_wide.columns.name = None


res_wide = res_wide.rename(

    columns={

        "r0.2":
            "r0_2_best_match_pct",

        "r0.6":
            "r0_6_best_match_pct"

    }

)


final_structural_stability = (

    clean_cluster_stability

    .merge(

        res_wide,

        on=
            "Primary_cluster",

        how=
            "left"
    )

)


final_structural_stability[
    "Mean_structural_retention_pct"
] = (

    final_structural_stability[
        [
            "NoPC4_best_match_pct",
            "Capped_best_match_pct",
            "r0_2_best_match_pct",
            "r0_6_best_match_pct"
        ]
    ]
    .mean(
        axis=1
    )
)


FINAL_STABILITY_PATH = (

    RESULTS_DIR /
    "Step23D4A_Final_Structural_Stability.csv"
)


final_structural_stability.to_csv(

    FINAL_STABILITY_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 110)
print("FINAL STRUCTURAL STABILITY")
print("=" * 110)


display(
    final_structural_stability
    .round(2)
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 177
# Save clean sensitivity objects
# ============================================================

NO_PC4_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D4A_CLEAN_NoPC4_Sensitivity.h5ad"
)


CAPPED_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D4A_CLEAN_PatientCapped_Sensitivity.h5ad"
)


adata_clean.write_h5ad(

    NO_PC4_OUTPUT,

    compression="gzip"
)


adata_cap.write_h5ad(

    CAPPED_OUTPUT,

    compression="gzip"
)


print(
    "[OK]",
    NO_PC4_OUTPUT.name
)

print(
    "[OK]",
    CAPPED_OUTPUT.name
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 178
# Structural decision record
# ============================================================

step23d4a_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23D4A_Final_Clean_State_Stability",

    "Lineage_clean_cells":
        int(
            adata_clean.n_obs
        ),

    "Primary_representation":
        "Harmony(patient)",

    "Primary_resolution":
        0.4,

    "Primary_cluster_key":
        PRIMARY_KEY,

    "Primary_cluster_number":
        int(
            adata_clean.obs[
                PRIMARY_KEY
            ].nunique()
        ),

    "Resolution_rationale":
        (
            "Clean r0.4 contains nine clusters with no clusters "
            "smaller than 100 cells; r0.2-to-r0.4 ARI is high, "
            "whereas r0.6 introduces a seven-cell cluster."
        ),

    "Sensitivity_framework": [

        "Clean no-PC4 Harmony",
        "Clean patient-capped Harmony",
        "Leiden r0.2",
        "Leiden r0.6"

    ],

    "Primary_vs_noPC4_ARI":
        float(
            ari_clean_noPC4
        ),

    "Primary_vs_patientCapped_ARI":
        float(
            ari_clean_capped
        ),

    "Final_biological_state_names":
        "NOT_ASSIGNED",

    "Next_step":
        (
            "Perform clean-cluster marker discovery, patient-state "
            "pseudobulk reproducibility, biological program scoring "
            "and pathway characterization before final consensus "
            "state consolidation."
        )
}


SUMMARY_D4A = (

    LOGS_DIR /
    "Step23D4A_Final_Summary.json"
)


with open(

    SUMMARY_D4A,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23d4a_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_D4A
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 179
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_D4A = (

    RESULTS_DIR /
    "Step23D4A_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_D4A,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_D4A
)

In [ ]:
# ============================================================
# STEP23D-4A - CELL 180
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 120)
print("STEP23D-4A FINAL CLEAN-STATE STABILITY REVIEW")
print("=" * 120)


print("\n[1] Primary framework")
print("-" * 120)

print(
    "Lineage-clean cells:",
    f"{adata_clean.n_obs:,}"
)

print(
    "Primary resolution:",
    0.4
)

print(
    "Primary clusters:",
    adata_clean.obs[
        PRIMARY_KEY
    ].nunique()
)


print("\n[2] Clean no-PC4 sensitivity")
print("-" * 120)

print(
    "Clusters:",
    adata_clean.obs[
        "state_clean_noPC4_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_clean_noPC4,
        4
    )
)

display(
    clean_noPC4_pct
    .round(2)
)


print("\n[3] Clean patient-capped sensitivity")
print("-" * 120)

display(
    adata_cap.obs[
        "patient"
    ]
    .value_counts()
    .to_frame(
        "N_cells"
    )
)

print(
    "Clusters:",
    adata_cap.obs[
        "state_clean_capped_r0_4"
    ].nunique()
)

print(
    "ARI:",
    round(
        ari_clean_capped,
        4
    )
)

display(
    clean_capped_pct
    .round(2)
)


print("\n[4] Cluster-level structural stability")
print("-" * 120)

display(
    final_structural_stability
    .round(2)
)


print("\n[5] Files")
print("-" * 120)


for path in [

    NO_PC4_OUTPUT,

    CAPPED_OUTPUT,

    CAP_MANIFEST_PATH,

    RESULTS_DIR /
    "Step23D4A_Primary_vs_NoPC4_Percent.csv",

    RESULTS_DIR /
    "Step23D4A_Primary_vs_Capped_Percent.csv",

    RESULTS_DIR /
    "Step23D4A_Resolution_Stability.csv",

    FINAL_STABILITY_PATH,

    SUMMARY_D4A,

    MANIFEST_D4A

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 120)

print(
    "STEP23D-4A STATUS = FINAL STRUCTURAL STABILITY COMPLETE"
)

print(
    "PRIMARY CLEAN PARTITION = LEIDEN r=0.4"
)

print(
    "BIOLOGICAL STATE NAMES = NOT YET ASSIGNED"
)

print(
    "NEXT = CLEAN MARKERS + PSEUDOBULK + PATHWAYS + CONSENSUS STATE CONSOLIDATION"
)

print("=" * 120)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23D-4B - CELL 181
# Load structurally locked lineage-clean state object
# ============================================================

from pathlib import Path

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.sparse import vstack
from scipy.stats import hypergeom

from sklearn.metrics.pairwise import cosine_similarity

import json
import warnings
import gc

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"
REFERENCE_DIR = ROOT / "Reference"

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


STATE_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_STATE_CANDIDATES.h5ad"
)

RAW_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)

STABILITY_INPUT = (
    RESULTS_DIR /
    "Step23D4A_Final_Structural_Stability.csv"
)


assert STATE_INPUT.exists()
assert RAW_INPUT.exists()
assert STABILITY_INPUT.exists()


adata = sc.read_h5ad(
    STATE_INPUT
)

adata_raw = sc.read_h5ad(
    RAW_INPUT
)

structural_stability = pd.read_csv(
    STABILITY_INPUT
)


STATE_KEY = "state_clean_leiden_r0_4"


for col in [
    STATE_KEY,
    "patient",
    "sample",
    "site",
    "cnv_leiden_r0_4"
]:

    if col in adata.obs.columns:

        adata.obs[col] = (
            adata.obs[col]
            .astype(str)
            .astype("category")
        )


assert (
    adata.obs_names
    ==
    adata_raw.obs_names
).all()


assert (
    adata.var_names
    ==
    adata_raw.var_names
).all()


clusters = sorted(
    adata.obs[
        STATE_KEY
    ]
    .astype(str)
    .unique(),
    key=lambda x: int(x)
)


print("=" * 115)
print("STEP23D-4B CLEAN BIOLOGICAL STATE CHARACTERIZATION")
print("=" * 115)

print(
    "Cells:",
    f"{adata.n_obs:,}"
)

print(
    "Genes:",
    f"{adata.n_vars:,}"
)

print(
    "Primary clusters:",
    clusters
)

print(
    "N clusters:",
    len(clusters)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 182
# Publication plotting settings
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename
):

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        FIGURES_DIR /
        f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    print(
        "Saved:",
        filename
    )

In [ ]:
# ============================================================
# STEP23D-4B - CELL 183
# Cell-level marker discovery
# ============================================================

MARKER_KEY = (
    "rank_genes_groups_clean_r0_4"
)


sc.tl.rank_genes_groups(

    adata,

    groupby=
        STATE_KEY,

    method=
        "wilcoxon",

    use_raw=
        False,

    pts=
        True,

    key_added=
        MARKER_KEY
)


marker_tables = []


for cluster in clusters:

    temp = sc.get.rank_genes_groups_df(

        adata,

        group=
            cluster,

        key=
            MARKER_KEY
    )


    temp[
        "State_cluster"
    ] = cluster


    marker_tables.append(
        temp
    )


all_markers = pd.concat(
    marker_tables,
    ignore_index=True
)


all_markers[
    "Is_MT"
] = (
    all_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.startswith("MT-")
)


all_markers[
    "Is_Ribosomal"
] = (
    all_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


ALL_MARKERS_PATH = (
    RESULTS_DIR /
    "Step23D4B_Clean_AllMarkers.csv"
)


all_markers.to_csv(
    ALL_MARKERS_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Marker rows:",
    f"{len(all_markers):,}"
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 184
# Candidate biological markers
# ============================================================

candidate_markers = (

    all_markers[
        (
            all_markers[
                "pvals_adj"
            ]
            <
            0.05
        )

        &

        (
            all_markers[
                "logfoldchanges"
            ]
            >
            0.25
        )

        &

        (
            ~all_markers[
                "Is_MT"
            ]
        )

        &

        (
            ~all_markers[
                "Is_Ribosomal"
            ]
        )
    ]
    .copy()
)


if "pct_nz_group" in candidate_markers.columns:

    candidate_markers = (

        candidate_markers[
            candidate_markers[
                "pct_nz_group"
            ]
            >=
            0.10
        ]
        .copy()
    )


if (
    "pct_nz_group"
    in candidate_markers.columns

    and

    "pct_nz_reference"
    in candidate_markers.columns
):

    candidate_markers[
        "Delta_pct"
    ] = (

        candidate_markers[
            "pct_nz_group"
        ]

        -

        candidate_markers[
            "pct_nz_reference"
        ]
    )


    candidate_markers = (

        candidate_markers[
            candidate_markers[
                "Delta_pct"
            ]
            >=
            0.05
        ]
        .copy()
    )


candidate_markers = (

    candidate_markers

    .sort_values(

        [
            "State_cluster",
            "logfoldchanges"
        ],

        ascending=[
            True,
            False
        ]
    )
)


CANDIDATE_MARKERS_PATH = (

    RESULTS_DIR /
    "Step23D4B_Clean_CandidateMarkers.csv"
)


candidate_markers.to_csv(

    CANDIDATE_MARKERS_PATH,

    index=False,
    encoding="utf-8-sig"
)


marker_count = (

    candidate_markers

    .groupby(
        "State_cluster"
    )

    .size()

    .rename(
        "N_candidate_markers"
    )

    .reset_index()
)


display(
    marker_count
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 185
# Build patient × clean-state pseudobulk counts
# ============================================================

adata_raw.obs[
    "patient"
] = (

    adata.obs.loc[
        adata_raw.obs_names,
        "patient"
    ]
    .astype(str)
    .values
)


adata_raw.obs[
    STATE_KEY
] = (

    adata.obs.loc[
        adata_raw.obs_names,
        STATE_KEY
    ]
    .astype(str)
    .values
)


adata_raw.obs[
    "patient_state"
] = (

    adata_raw.obs[
        "patient"
    ]
    .astype(str)

    +

    "__State"

    +

    adata_raw.obs[
        STATE_KEY
    ]
    .astype(str)
)


pb_rows = []
pb_meta_rows = []


for group in (
    adata_raw.obs[
        "patient_state"
    ]
    .unique()
):

    mask = (
        adata_raw.obs[
            "patient_state"
        ]
        ==
        group
    )


    idx = np.where(
        mask.values
    )[0]


    X_group = (
        adata_raw.X[
            idx,
            :
        ]
    )


    if sparse.issparse(
        X_group
    ):

        summed = sparse.csr_matrix(
            X_group.sum(
                axis=0
            )
        )

    else:

        summed = sparse.csr_matrix(

            np.sum(
                X_group,
                axis=0,
                keepdims=True
            )
        )


    pb_rows.append(
        summed
    )


    patient = (
        adata_raw.obs.loc[
            mask,
            "patient"
        ]
        .iloc[0]
    )


    state = (
        adata_raw.obs.loc[
            mask,
            STATE_KEY
        ]
        .iloc[0]
    )


    pb_meta_rows.append({

        "patient_state":
            group,

        "patient":
            str(patient),

        "State_cluster":
            str(state),

        "N_cells":
            int(
                mask.sum()
            )
    })


pb_counts = vstack(
    pb_rows
).tocsr()


pb_meta = pd.DataFrame(
    pb_meta_rows
)


adata_pb_counts = ad.AnnData(

    X=
        pb_counts,

    obs=
        pb_meta.set_index(
            "patient_state"
        ),

    var=
        adata_raw.var.copy()
)


print(
    "Pseudobulk samples:",
    adata_pb_counts.n_obs
)


display(

    adata_pb_counts.obs

    .sort_values(
        [
            "State_cluster",
            "patient"
        ]
    )
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 186
# Save counts and generate logCPM
# ============================================================

PB_COUNTS_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D4B_Clean_PatientState_COUNTS.h5ad"
)


adata_pb_counts.write_h5ad(
    PB_COUNTS_PATH,
    compression="gzip"
)


adata_pb_counts.obs.to_csv(

    RESULTS_DIR /
    "Step23D4B_Clean_PatientState_Metadata.csv",

    encoding="utf-8-sig"
)


adata_pb_log = (
    adata_pb_counts.copy()
)


sc.pp.normalize_total(
    adata_pb_log,
    target_sum=1e6
)


sc.pp.log1p(
    adata_pb_log
)


PB_LOG_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D4B_Clean_PatientState_LogCPM.h5ad"
)


adata_pb_log.write_h5ad(
    PB_LOG_PATH,
    compression="gzip"
)


print(
    "[OK]",
    PB_COUNTS_PATH.name
)

print(
    "[OK]",
    PB_LOG_PATH.name
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 187
# Within-patient pseudobulk expression delta
# ============================================================

if sparse.issparse(
    adata_pb_log.X
):

    X_pb = (
        adata_pb_log.X
        .toarray()
        .astype(np.float32)
    )

else:

    X_pb = np.asarray(
        adata_pb_log.X,
        dtype=np.float32
    )


delta_matrix = np.full(

    X_pb.shape,

    np.nan,

    dtype=np.float32
)


for i in range(
    adata_pb_log.n_obs
):

    patient = str(
        adata_pb_log.obs.iloc[i][
            "patient"
        ]
    )


    state = str(
        adata_pb_log.obs.iloc[i][
            "State_cluster"
        ]
    )


    other_mask = (

        (
            adata_pb_log.obs[
                "patient"
            ]
            .astype(str)
            ==
            patient
        )

        &

        (
            adata_pb_log.obs[
                "State_cluster"
            ]
            .astype(str)
            !=
            state
        )
    )


    other_idx = np.where(
        other_mask.values
    )[0]


    if len(other_idx) == 0:
        continue


    other_mean = np.mean(
        X_pb[
            other_idx,
            :
        ],
        axis=0
    )


    delta_matrix[
        i,
        :
    ] = (

        X_pb[
            i,
            :
        ]

        -

        other_mean
    )


adata_pb_delta = ad.AnnData(

    X=
        delta_matrix,

    obs=
        adata_pb_log.obs.copy(),

    var=
        adata_pb_log.var.copy()
)


PB_DELTA_PATH = (

    DATA_DIR /
    "GSE149614_STEP23D4B_Clean_PatientState_Delta.h5ad"
)


adata_pb_delta.write_h5ad(
    PB_DELTA_PATH,
    compression="gzip"
)


print(
    "Saved:",
    PB_DELTA_PATH
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 188
# Patient-level state support
# ============================================================

MIN_CELLS_PER_PATIENT_STATE = 20


pb_support = (
    adata_pb_delta.obs
    .copy()
)


pb_support[
    "Eligible"
] = (

    pb_support[
        "N_cells"
    ]
    >=
    MIN_CELLS_PER_PATIENT_STATE
)


state_support = (

    pb_support

    .groupby(
        "State_cluster"
    )

    .agg(

        N_pseudobulks=(
            "patient",
            "size"
        ),

        N_eligible_patients=(
            "Eligible",
            "sum"
        ),

        Total_cells=(
            "N_cells",
            "sum"
        )
    )

    .reset_index()
)


def support_class(
    n
):

    if n >= 3:
        return "Cross_patient_3plus"

    elif n == 2:
        return "Cross_patient_2"

    elif n == 1:
        return "Single_patient_supported"

    else:
        return "Insufficient_patient_support"


state_support[
    "Patient_support_class"
] = (

    state_support[
        "N_eligible_patients"
    ]
    .apply(
        support_class
    )
)


state_support.to_csv(

    RESULTS_DIR /
    "Step23D4B_State_PatientSupport.csv",

    index=False,
    encoding="utf-8-sig"
)


display(
    state_support
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 189
# Cross-patient marker reproducibility
# ============================================================

gene_to_idx = {

    gene: i

    for i, gene
    in enumerate(
        adata_pb_delta.var_names
    )
}


marker_repro_rows = []


for cluster in clusters:

    marker_df = (

        candidate_markers[
            candidate_markers[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ]
    )


    eligible_mask = (

        (
            adata_pb_delta.obs[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        )

        &

        (
            adata_pb_delta.obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    eligible_idx = np.where(
        eligible_mask.values
    )[0]


    eligible_patients = (

        adata_pb_delta.obs.iloc[
            eligible_idx
        ][
            "patient"
        ]
        .astype(str)
        .tolist()
    )


    for _, row in marker_df.iterrows():

        gene = str(
            row[
                "names"
            ]
        )


        if gene not in gene_to_idx:
            continue


        gene_idx = (
            gene_to_idx[
                gene
            ]
        )


        values = (
            delta_matrix[
                eligible_idx,
                gene_idx
            ]
        )


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(values) == 0:

            n_positive = 0
            positive_fraction = np.nan
            median_delta = np.nan
            mean_delta = np.nan

        else:

            n_positive = int(
                np.sum(
                    values > 0
                )
            )

            positive_fraction = float(
                np.mean(
                    values > 0
                )
            )

            median_delta = float(
                np.median(
                    values
                )
            )

            mean_delta = float(
                np.mean(
                    values
                )
            )


        marker_repro_rows.append({

            "State_cluster":
                cluster,

            "Gene":
                gene,

            "CellLevel_logFC":
                float(
                    row[
                        "logfoldchanges"
                    ]
                ),

            "CellLevel_adjP":
                float(
                    row[
                        "pvals_adj"
                    ]
                ),

            "N_eligible_patients":
                len(values),

            "N_positive_patients":
                n_positive,

            "Positive_fraction":
                positive_fraction,

            "Median_withinPatient_delta":
                median_delta,

            "Mean_withinPatient_delta":
                mean_delta,

            "Eligible_patients":
                "|".join(
                    eligible_patients
                )
        })


marker_repro = pd.DataFrame(
    marker_repro_rows
)


marker_repro = (

    marker_repro

    .sort_values(

        [
            "State_cluster",
            "N_positive_patients",
            "Positive_fraction",
            "Median_withinPatient_delta",
            "CellLevel_logFC"
        ],

        ascending=[
            True,
            False,
            False,
            False,
            False
        ]
    )
)


MARKER_REPRO_PATH = (

    RESULTS_DIR /
    "Step23D4B_Clean_Marker_Reproducibility.csv"
)


marker_repro.to_csv(

    MARKER_REPRO_PATH,

    index=False,
    encoding="utf-8-sig"
)


top20_repro = (

    marker_repro

    .groupby(
        "State_cluster",
        group_keys=False
    )

    .head(20)

    .copy()
)


TOP20_REPRO_PATH = (

    RESULTS_DIR /
    "Step23D4B_Clean_Top20_ReproducibleMarkers.csv"
)


top20_repro.to_csv(

    TOP20_REPRO_PATH,

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 190
# Patient-adjusted state transcriptomic similarity
# ============================================================

if (
    "highly_variable_for_pca"
    in adata.var.columns
):

    signature_genes = (

        adata.var_names[
            adata.var[
                "highly_variable_for_pca"
            ]
            .astype(bool)
        ]
        .tolist()
    )

else:

    signature_genes = (

        adata.var_names[
            adata.var[
                "highly_variable"
            ]
            .astype(bool)
        ]
        .tolist()
    )


signature_genes = [

    gene
    for gene in signature_genes
    if gene in gene_to_idx
]


signature_gene_idx = [

    gene_to_idx[
        gene
    ]
    for gene in signature_genes
]


state_signature_rows = []


for cluster in clusters:

    mask = (

        (
            adata_pb_delta.obs[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        )

        &

        (
            adata_pb_delta.obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    idx = np.where(
        mask.values
    )[0]


    if len(idx) == 0:

        signature = np.full(
            len(signature_gene_idx),
            np.nan
        )

    else:

        signature = np.nanmean(

            delta_matrix[
                idx,
                :
            ][
                :,
                signature_gene_idx
            ],

            axis=0
        )


    state_signature_rows.append(
        signature
    )


state_signature_matrix = np.vstack(
    state_signature_rows
)


state_corr = np.corrcoef(
    state_signature_matrix
)


state_corr_df = pd.DataFrame(

    state_corr,

    index=
        clusters,

    columns=
        clusters
)


state_corr_df.index.name = (
    "State_cluster"
)


STATE_CORR_PATH = (

    RESULTS_DIR /
    "Step23D4B_PatientAdjusted_StateCorrelation.csv"
)


state_corr_df.to_csv(
    STATE_CORR_PATH,
    encoding="utf-8-sig"
)


display(
    state_corr_df
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 191
# Curated malignant-cell biological programs
# ============================================================

BIO_PROGRAMS = {

    "Proliferation_G2M": [

        "MKI67", "TOP2A", "CCNB1", "CCNB2",
        "CDK1", "CDC20", "UBE2C", "BIRC5",
        "CENPF", "PLK1", "AURKA", "AURKB",
        "KIF11", "KIF20A", "CDCA8"

    ],

    "DNA_Replication_S": [

        "MCM2", "MCM3", "MCM4", "MCM5",
        "MCM6", "MCM7", "PCNA", "RRM2",
        "TYMS", "FEN1", "CDC6", "RFC4",
        "HELLS", "GINS1", "GINS2"

    ],

    "Hypoxia": [

        "CA9", "VEGFA", "SLC2A1", "BNIP3",
        "BNIP3L", "NDRG1", "EGLN3", "LDHA",
        "PDK1", "ENO1"

    ],

    "Glycolysis": [

        "HK2", "PFKP", "ALDOA", "GAPDH",
        "PGK1", "ENO1", "PKM", "LDHA",
        "SLC2A1", "TPI1"

    ],

    "Hepatic_Differentiation": [

        "ALB", "APOA1", "APOA2", "TTR",
        "ASGR1", "ASGR2", "CPS1", "HPD",
        "MAT1A", "HNF4A", "F7", "PCK1"

    ],

    "Bile_Acid_Metabolism": [

        "CYP8B1", "CYP7A1", "SLC10A1", "ABCB11",
        "ABCC2", "BAAT", "HSD17B13", "SULT2A1",
        "NR1H4", "AKR1D1"

    ],

    "Fatty_Acid_Metabolism": [

        "ACADM", "ACADL", "CPT1A", "CPT2",
        "HADHA", "HADHB", "ACAA2", "ECHS1",
        "FABP1", "PPARA", "HMGCS2"

    ],

    "Progenitor_Epithelial": [

        "EPCAM", "KRT19", "KRT7", "KRT17",
        "TACSTD2", "PROM1", "SOX9", "KRT8",
        "KRT18", "S100P", "SPINK1", "AGR2"

    ],

    "Fetal_HCC_Program": [

        "AFP", "GPC3", "DLK1", "IGF2",
        "PROM1", "EPCAM", "KRT19", "SOX9"

    ],

    "EMT_ECM": [

        "VIM", "FN1", "TGFBI", "SPARC",
        "COL1A1", "COL1A2", "COL3A1",
        "TAGLN", "ACTA2", "SNAI1",
        "SNAI2", "ZEB1", "ZEB2"

    ],

    "TGFb_Response": [

        "TGFB1", "TGFBI", "SERPINE1",
        "PMEPA1", "CTGF", "SMAD7",
        "ITGA5", "FN1", "VIM"

    ],

    "Inflammatory_NFkB": [

        "CXCL1", "CXCL2", "CXCL3",
        "CCL20", "NFKBIA", "TNFAIP3",
        "ICAM1", "IL6", "IER3"

    ],

    "Stress_UPR": [

        "HSPA5", "XBP1", "ATF4",
        "DDIT3", "HERPUD1", "DNAJB9",
        "HSP90B1", "PPP1R15A"

    ],

    "Wnt_BetaCatenin_Target": [

        "AXIN2", "GLUL", "LGR5",
        "TBX3", "RNF43", "SP5",
        "NOTUM", "NKD1"

    ]
}


program_audit_rows = []


for program, genes in BIO_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata.var_names
    ]


    missing = [
        gene
        for gene in genes
        if gene not in adata.var_names
    ]


    program_audit_rows.append({

        "Program":
            program,

        "N_defined":
            len(genes),

        "N_detected":
            len(detected),

        "Detected_fraction":
            len(detected)
            /
            len(genes),

        "Detected_genes":
            "|".join(detected),

        "Missing_genes":
            "|".join(missing)
    })


program_audit = pd.DataFrame(
    program_audit_rows
)


PROGRAM_AUDIT_PATH = (

    RESULTS_DIR /
    "Step23D4B_BiologicalProgram_GeneAudit.csv"
)


program_audit.to_csv(

    PROGRAM_AUDIT_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    program_audit[
        [
            "Program",
            "N_defined",
            "N_detected",
            "Detected_fraction"
        ]
    ]
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 192
# Score biological programs
# ============================================================

program_score_cols = []


for program, genes in BIO_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata.var_names
    ]


    if len(detected) < 3:

        print(
            f"[SKIP] {program}: only {len(detected)} genes detected"
        )

        continue


    score_col = (
        "Program__"
        +
        program
    )


    sc.tl.score_genes(

        adata,

        gene_list=
            detected,

        score_name=
            score_col,

        ctrl_size=
            min(
                50,
                max(
                    15,
                    len(detected) * 3
                )
            ),

        n_bins=
            25,

        random_state=
            SEED,

        use_raw=
            False
    )


    program_score_cols.append(
        score_col
    )


print(
    "Programs scored:",
    len(program_score_cols)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 193
# Patient-adjusted biological program evidence
# ============================================================

program_pb = (

    adata.obs[
        [
            "patient",
            STATE_KEY
        ]
        +
        program_score_cols
    ]

    .copy()
)


program_pb[
    "patient"
] = (
    program_pb[
        "patient"
    ]
    .astype(str)
)


program_pb[
    STATE_KEY
] = (
    program_pb[
        STATE_KEY
    ]
    .astype(str)
)


# ------------------------------------------------------------
# Median program score for each patient × state
# ------------------------------------------------------------

patient_state_program = (

    program_pb

    .groupby(
        [
            "patient",
            STATE_KEY
        ],
        observed=True
    )[
        program_score_cols
    ]

    .median()

    .reset_index()
)


# ------------------------------------------------------------
# Add cell number
# ------------------------------------------------------------

state_cell_n = (

    adata.obs

    .groupby(
        [
            adata.obs[
                "patient"
            ].astype(str),

            adata.obs[
                STATE_KEY
            ].astype(str)
        ],

        observed=True
    )

    .size()

    .rename(
        "N_cells"
    )

    .reset_index()
)


state_cell_n.columns = [
    "patient",
    STATE_KEY,
    "N_cells"
]


patient_state_program = (

    patient_state_program

    .merge(

        state_cell_n,

        on=[
            "patient",
            STATE_KEY
        ],

        how="left"
    )
)


# ------------------------------------------------------------
# Within-patient adjustment
# ------------------------------------------------------------

program_delta_rows = []


for _, row in (
    patient_state_program
    .iterrows()
):

    patient = str(
        row[
            "patient"
        ]
    )

    state = str(
        row[
            STATE_KEY
        ]
    )


    other = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                .astype(str)
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                .astype(str)
                !=
                state
            )
        ]
    )


    out = {

        "patient":
            patient,

        "State_cluster":
            state,

        "N_cells":
            int(
                row[
                    "N_cells"
                ]
            )
    }


    for score_col in program_score_cols:

        if len(other) == 0:

            delta = np.nan

        else:

            delta = (

                float(
                    row[
                        score_col
                    ]
                )

                -

                float(
                    other[
                        score_col
                    ]
                    .mean()
                )
            )


        out[
            score_col
        ] = delta


    program_delta_rows.append(
        out
    )


patient_state_program_delta = pd.DataFrame(
    program_delta_rows
)


PATIENT_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step23D4B_PatientState_ProgramDelta.csv"
)


patient_state_program_delta.to_csv(

    PATIENT_PROGRAM_PATH,

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 194
# State-level patient-adjusted program summary
# ============================================================

eligible_program_pb = (

    patient_state_program_delta[
        patient_state_program_delta[
            "N_cells"
        ]
        >=
        MIN_CELLS_PER_PATIENT_STATE
    ]
    .copy()
)


state_program_delta = (

    eligible_program_pb

    .groupby(
        "State_cluster"
    )[
        program_score_cols
    ]

    .mean()

    .reindex(
        clusters
    )
)


state_program_delta.columns = [

    col.replace(
        "Program__",
        ""
    )

    for col in state_program_delta.columns
]


STATE_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step23D4B_State_PatientAdjusted_Programs.csv"
)


state_program_delta.to_csv(
    STATE_PROGRAM_PATH,
    encoding="utf-8-sig"
)


display(
    state_program_delta
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 195
# Patient-adjusted biological program heatmap
# ============================================================

program_z = (

    state_program_delta

    -

    state_program_delta.mean(
        axis=0
    )

) / (

    state_program_delta.std(
        axis=0
    )

    +
    1e-8
)


fig, ax = plt.subplots(
    figsize=(11.5, 7.0)
)


im = ax.imshow(

    program_z.values,

    aspect="auto",

    cmap="RdBu_r",

    vmin=-2.5,
    vmax=2.5
)


ax.set_xticks(
    np.arange(
        program_z.shape[1]
    )
)


ax.set_xticklabels(

    [

        col.replace(
            "_",
            " "
        )

        for col in program_z.columns
    ],

    rotation=55,
    ha="right"
)


ax.set_yticks(
    np.arange(
        program_z.shape[0]
    )
)


ax.set_yticklabels(
    program_z.index
)


ax.set_xlabel(
    "Biological Program"
)

ax.set_ylabel(
    "Malignant State Candidate"
)

ax.set_title(
    "Patient-adjusted Biological Programs Across Malignant State Candidates"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)


cbar.set_label(
    "Relative Program Enrichment (Z-score)"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D4B_01_PatientAdjusted_BiologicalPrograms"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 196
# Patient-adjusted transcriptomic state similarity
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.2, 6.4)
)


im = ax.imshow(

    state_corr,

    cmap="RdBu_r",

    vmin=-1,
    vmax=1,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        len(clusters)
    )
)


ax.set_xticklabels(
    clusters
)


ax.set_yticks(
    np.arange(
        len(clusters)
    )
)


ax.set_yticklabels(
    clusters
)


ax.set_xlabel(
    "Malignant State Candidate"
)

ax.set_ylabel(
    "Malignant State Candidate"
)

ax.set_title(
    "Patient-adjusted Transcriptomic Similarity"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Pearson Correlation"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step23D4B_02_PatientAdjusted_StateSimilarity"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 197
# Nearest transcriptomic neighbor
# ============================================================

nearest_rows = []


for i, cluster in enumerate(
    clusters
):

    values = (
        state_corr[
            i,
            :
        ]
        .copy()
    )


    values[i] = -np.inf


    j = int(
        np.nanargmax(
            values
        )
    )


    nearest_rows.append({

        "State_cluster":
            cluster,

        "Closest_state":
            clusters[j],

        "Closest_state_correlation":
            float(
                values[j]
            )
    })


nearest_state = pd.DataFrame(
    nearest_rows
)


display(
    nearest_state
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 198
# Top biological programs per state
# ============================================================

top_program_rows = []


for cluster in clusters:

    row = (
        state_program_delta.loc[
            cluster
        ]
        .sort_values(
            ascending=False
        )
    )


    for rank, (
        program,
        value
    ) in enumerate(
        row.head(5).items(),
        start=1
    ):

        top_program_rows.append({

            "State_cluster":
                cluster,

            "Rank":
                rank,

            "Program":
                program,

            "PatientAdjusted_mean_delta":
                float(
                    value
                )
        })


top_programs = pd.DataFrame(
    top_program_rows
)


TOP_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step23D4B_Top5_Programs_PerState.csv"
)


top_programs.to_csv(

    TOP_PROGRAM_PATH,

    index=False,
    encoding="utf-8-sig"
)


for cluster in clusters:

    print(
        "\nState",
        cluster
    )


    display(

        top_programs[
            top_programs[
                "State_cluster"
            ]
            ==
            cluster
        ]
        .round(4)
    )

In [ ]:
# ============================================================
# STEP23D-4B - CELL 199
# OPTIONAL offline GMT enrichment
# No internet access required
# ============================================================

GMT_DIR = (
    REFERENCE_DIR /
    "GMT"
)

GMT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


gmt_files = list(
    GMT_DIR.glob(
        "*.gmt"
    )
)


def parse_gmt(
    path
):

    gene_sets = {}


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            parts = (
                line.rstrip("\n")
                .split("\t")
            )


            if len(parts) < 3:
                continue


            name = parts[0]

            genes = set(
                parts[2:]
            )


            gene_sets[
                name
            ] = genes


    return gene_sets


def bh_adjust(
    pvalues
):

    pvalues = np.asarray(
        pvalues,
        dtype=float
    )


    n = len(pvalues)


    order = np.argsort(
        pvalues
    )


    ranked = pvalues[
        order
    ]


    adjusted = (

        ranked
        *
        n
        /
        np.arange(
            1,
            n + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    output = np.empty(
        n
    )


    output[
        order
    ] = adjusted


    return output


ora_results = []


if len(gmt_files) == 0:

    print(
        "No local GMT files found."
    )

    print(
        "Offline GO/KEGG/Hallmark ORA skipped."
    )


else:

    background = set(
        adata.var_names
    )


    M = len(background)


    for gmt_path in gmt_files:

        print(
            "Reading:",
            gmt_path.name
        )


        gene_sets = parse_gmt(
            gmt_path
        )


        for cluster in clusters:

            temp = (

                marker_repro[
                    marker_repro[
                        "State_cluster"
                    ]
                    .astype(str)
                    ==
                    cluster
                ]

                .copy()
            )


            # Prefer reproducible positive markers
            temp = temp[
                (
                    temp[
                        "N_positive_patients"
                    ]
                    >=
                    2
                )

                &

                (
                    temp[
                        "Median_withinPatient_delta"
                    ]
                    >
                    0
                )
            ]


            if len(temp) < 20:

                temp = (

                    marker_repro[
                        marker_repro[
                            "State_cluster"
                        ]
                        .astype(str)
                        ==
                        cluster
                    ]

                    .head(200)
                )


            query = set(
                temp[
                    "Gene"
                ]
            )


            query = (
                query
                &
                background
            )


            N = len(query)


            if N == 0:
                continue


            local_rows = []


            for pathway, genes in (
                gene_sets.items()
            ):

                genes = (
                    genes
                    &
                    background
                )


                K = len(genes)


                overlap = (
                    query
                    &
                    genes
                )


                k = len(overlap)


                if k < 3:
                    continue


                p = hypergeom.sf(
                    k - 1,
                    M,
                    K,
                    N
                )


                local_rows.append({

                    "GMT_file":
                        gmt_path.name,

                    "State_cluster":
                        cluster,

                    "Pathway":
                        pathway,

                    "Query_size":
                        N,

                    "Pathway_size":
                        K,

                    "Overlap":
                        k,

                    "P_value":
                        p,

                    "Genes":
                        "|".join(
                            sorted(
                                overlap
                            )
                        )
                })


            if len(local_rows) > 0:

                local_df = pd.DataFrame(
                    local_rows
                )


                local_df[
                    "FDR_BH"
                ] = bh_adjust(
                    local_df[
                        "P_value"
                    ].values
                )


                ora_results.append(
                    local_df
                )


if len(ora_results) > 0:

    ora_df = pd.concat(
        ora_results,
        ignore_index=True
    )


    ora_df = (

        ora_df

        .sort_values(
            [
                "State_cluster",
                "FDR_BH",
                "P_value"
            ]
        )
    )


else:

    ora_df = pd.DataFrame()


ORA_PATH = (

    RESULTS_DIR /
    "Step23D4B_Offline_GMT_ORA.csv"
)


ora_df.to_csv(

    ORA_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "GMT files:",
    len(gmt_files)
)

print(
    "ORA rows:",
    len(ora_df)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 200
# Consensus evidence master table
# ============================================================

structural_stability[
    "Primary_cluster"
] = (
    structural_stability[
        "Primary_cluster"
    ]
    .astype(str)
)


state_support[
    "State_cluster"
] = (
    state_support[
        "State_cluster"
    ]
    .astype(str)
)


nearest_state[
    "State_cluster"
] = (
    nearest_state[
        "State_cluster"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# Top markers as text
# ------------------------------------------------------------

top_marker_text_rows = []


for cluster in clusters:

    temp = (

        marker_repro[
            marker_repro[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ]
        .head(10)
    )


    top_marker_text_rows.append({

        "State_cluster":
            cluster,

        "Top10_markers":
            "|".join(
                temp[
                    "Gene"
                ]
                .astype(str)
                .tolist()
            ),

        "N_markers_positive_2plus":
            int(
                (
                    marker_repro[
                        marker_repro[
                            "State_cluster"
                        ]
                        .astype(str)
                        ==
                        cluster
                    ][
                        "N_positive_patients"
                    ]
                    >=
                    2
                )
                .sum()
            ),

        "N_markers_positive_3plus":
            int(
                (
                    marker_repro[
                        marker_repro[
                            "State_cluster"
                        ]
                        .astype(str)
                        ==
                        cluster
                    ][
                        "N_positive_patients"
                    ]
                    >=
                    3
                )
                .sum()
            )
    })


marker_summary = pd.DataFrame(
    top_marker_text_rows
)


# ------------------------------------------------------------
# Top three programs
# ------------------------------------------------------------

program_summary_rows = []


for cluster in clusters:

    temp = (

        top_programs[
            top_programs[
                "State_cluster"
            ]
            ==
            cluster
        ]
        .sort_values(
            "Rank"
        )
    )


    row = {

        "State_cluster":
            cluster
    }


    for rank in [
        1,
        2,
        3
    ]:

        hit = temp[
            temp[
                "Rank"
            ]
            ==
            rank
        ]


        if len(hit) == 1:

            row[
                f"TopProgram_{rank}"
            ] = (
                hit.iloc[0][
                    "Program"
                ]
            )

            row[
                f"TopProgram_{rank}_delta"
            ] = float(
                hit.iloc[0][
                    "PatientAdjusted_mean_delta"
                ]
            )


    program_summary_rows.append(
        row
    )


program_summary = pd.DataFrame(
    program_summary_rows
)


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

consensus_evidence = (

    structural_stability

    .rename(
        columns={
            "Primary_cluster":
                "State_cluster"
        }
    )

    .merge(
        state_support,
        on=
            "State_cluster",
        how=
            "left"
    )

    .merge(
        marker_summary,
        on=
            "State_cluster",
        how=
            "left"
    )

    .merge(
        nearest_state,
        on=
            "State_cluster",
        how=
            "left"
    )

    .merge(
        program_summary,
        on=
            "State_cluster",
        how=
            "left"
    )
)


CONSENSUS_PATH = (

    RESULTS_DIR /
    "Step23D4B_Consensus_State_Evidence.csv"
)


consensus_evidence.to_csv(

    CONSENSUS_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    consensus_evidence
    .round(3)
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 201
# Save evidence checkpoint
# ============================================================

EVIDENCE_OUTPUT = (

    DATA_DIR /
    "GSE149614_STEP23D4B_CLEAN_STATE_BIOLOGICAL_EVIDENCE.h5ad"
)


adata.write_h5ad(

    EVIDENCE_OUTPUT,

    compression="gzip"
)


print(
    "Saved:",
    EVIDENCE_OUTPUT
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 202
# Decision record
# ============================================================

step23d4b_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23D4B_Clean_State_Biological_Characterization",

    "Lineage_clean_cells":
        int(
            adata.n_obs
        ),

    "Primary_cluster_key":
        STATE_KEY,

    "Primary_resolution":
        0.4,

    "Primary_cluster_number":
        len(clusters),

    "Cell_level_marker_method":
        "Wilcoxon rank-sum",

    "Patient_level_unit":
        "patient × malignant-state candidate",

    "Minimum_cells_for_patient_state_support":
        int(
            MIN_CELLS_PER_PATIENT_STATE
        ),

    "Patient_adjustment":
        (
            "Each patient-state pseudobulk or program score is "
            "contrasted against the mean of the same patient's "
            "other malignant-state candidates."
        ),

    "Biological_program_strategy":
        (
            "Transparent curated malignant-cell programs were used "
            "for state characterization. These are not claimed to be "
            "the original MSigDB Hallmark gene sets."
        ),

    "Offline_GMT_files_detected":
        len(gmt_files),

    "Bulk_risk_features_used_for_state_definition":
        False,

    "Final_state_names":
        "PENDING_CONSENSUS_REVIEW",

    "Next_step":
        (
            "Review structural stability, patient support, reproducible "
            "markers, transcriptomic similarity and biological programs; "
            "then lock consensus malignant states before bulk-risk projection."
        )
}


SUMMARY_D4B = (

    LOGS_DIR /
    "Step23D4B_Final_Summary.json"
)


with open(

    SUMMARY_D4B,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step23d4b_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_D4B
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 203
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_D4B = (

    RESULTS_DIR /
    "Step23D4B_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_D4B,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_D4B
)

In [ ]:
# ============================================================
# STEP23D-4B - CELL 204
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 125)
print("STEP23D-4B FINAL BIOLOGICAL EVIDENCE REVIEW")
print("=" * 125)


print("\n[1] Patient-level state support")
print("-" * 125)

display(
    state_support
)


print("\n[2] Structural stability")
print("-" * 125)

display(
    structural_stability
    .round(2)
)


print("\n[3] Patient-adjusted biological programs")
print("-" * 125)

display(
    state_program_delta
    .round(3)
)


print("\n[4] Patient-adjusted transcriptomic similarity")
print("-" * 125)

display(
    state_corr_df
    .round(3)
)


print("\n[5] Top reproducible markers")
print("-" * 125)


for cluster in clusters:

    print(
        f"\nState candidate {cluster}"
    )


    display(

        marker_repro[
            marker_repro[
                "State_cluster"
            ]
            .astype(str)
            ==
            cluster
        ][
            [
                "Gene",
                "CellLevel_logFC",
                "N_eligible_patients",
                "N_positive_patients",
                "Positive_fraction",
                "Median_withinPatient_delta"
            ]
        ]
        .head(10)
        .round(4)
    )


print("\n[6] Top biological programs")
print("-" * 125)


display(
    top_programs
    .round(4)
)


print("\n[7] Nearest transcriptomic states")
print("-" * 125)

display(
    nearest_state
    .round(4)
)


print("\n[8] Consensus evidence table")
print("-" * 125)

display(
    consensus_evidence
    .round(3)
)


print("\n[9] Optional offline GMT enrichment")
print("-" * 125)

print(
    "GMT files detected:",
    len(gmt_files)
)

print(
    "ORA rows:",
    len(ora_df)
)


if len(ora_df) > 0:

    display(
        ora_df
        .head(30)
    )


print("\n[10] Files")
print("-" * 125)


for path in [

    EVIDENCE_OUTPUT,

    PB_COUNTS_PATH,

    PB_LOG_PATH,

    PB_DELTA_PATH,

    ALL_MARKERS_PATH,

    CANDIDATE_MARKERS_PATH,

    MARKER_REPRO_PATH,

    TOP20_REPRO_PATH,

    STATE_CORR_PATH,

    PROGRAM_AUDIT_PATH,

    PATIENT_PROGRAM_PATH,

    STATE_PROGRAM_PATH,

    TOP_PROGRAM_PATH,

    ORA_PATH,

    CONSENSUS_PATH,

    SUMMARY_D4B,

    MANIFEST_D4B

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP23D-4B STATUS = BIOLOGICAL EVIDENCE COMPLETE"
)

print(
    "RAW LEIDEN CLUSTERS ARE STILL NOT FINAL BIOLOGICAL STATE NAMES"
)

print(
    "NEXT = CONSENSUS CONSOLIDATION + FINAL MALIGNANT-STATE LOCK"
)

print("=" * 125)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP23 FINAL LOCK - CELL 205
# Load biological evidence object
# ============================================================

from pathlib import Path

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.sparse import vstack

import json
import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


EVIDENCE_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D4B_CLEAN_STATE_BIOLOGICAL_EVIDENCE.h5ad"
)

RAW_INPUT = (
    DATA_DIR /
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)


assert EVIDENCE_INPUT.exists()
assert RAW_INPUT.exists()


adata = sc.read_h5ad(
    EVIDENCE_INPUT
)

adata_raw = sc.read_h5ad(
    RAW_INPUT
)


SOURCE_KEY = (
    "state_clean_leiden_r0_4"
)


assert (
    adata.obs_names
    ==
    adata_raw.obs_names
).all()


assert (
    adata.var_names
    ==
    adata_raw.var_names
).all()


print("=" * 115)
print("STEP23 FINAL MALIGNANT-STATE LOCK")
print("=" * 115)

print(
    "Cells:",
    f"{adata.n_obs:,}"
)

print(
    "Genes:",
    f"{adata.n_vars:,}"
)

print(
    "Source Leiden clusters:",
    adata.obs[
        SOURCE_KEY
    ].nunique()
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 206
# Consensus state mapping
# ============================================================

FINAL_MAPPING = {

    "0": {
        "id":
            "MS1",

        "name":
            "Hypoxia-TGFb-Inflammatory",

        "display":
            "MS1 Hypoxia-TGFβ-Inflammatory",

        "family":
            "Stress_Plasticity",

        "scope":
            "Cross_patient_supported"
    },

    "1": {
        "id":
            "MS2",

        "name":
            "S100P-SPINK1-Epithelial",

        "display":
            "MS2 S100P-SPINK1 Epithelial",

        "family":
            "Epithelial_Secretory",

        "scope":
            "Cross_patient_supported"
    },

    "2": {
        "id":
            "MS3",

        "name":
            "Cycling-G2M",

        "display":
            "MS3 Cycling-G2M",

        "family":
            "Proliferative",

        "scope":
            "Cross_patient_supported"
    },

    "3": {
        "id":
            "MS4",

        "name":
            "HCC08-Replication-Fetal-Progenitor",

        "display":
            "MS4 HCC08 Replication-Fetal/Progenitor",

        "family":
            "Proliferative_Progenitor",

        "scope":
            "Patient_specific_HCC08"
    },

    "4": {
        "id":
            "MS4",

        "name":
            "HCC08-Replication-Fetal-Progenitor",

        "display":
            "MS4 HCC08 Replication-Fetal/Progenitor",

        "family":
            "Proliferative_Progenitor",

        "scope":
            "Patient_specific_HCC08"
    },

    "5": {
        "id":
            "MS5",

        "name":
            "Stress-Inflammatory-Progenitor",

        "display":
            "MS5 Stress-Inflammatory-Progenitor",

        "family":
            "Stress_Plasticity",

        "scope":
            "Cross_patient_supported"
    },

    "6": {
        "id":
            "MS6",

        "name":
            "Hepatic-Differentiated",

        "display":
            "MS6 Hepatic-Differentiated",

        "family":
            "Hepatic_Metabolic",

        "scope":
            "Cross_patient_supported"
    },

    "7": {
        "id":
            "MS7",

        "name":
            "HCC04-Glycolytic-Hepatic",

        "display":
            "MS7 HCC04 Glycolytic-Hepatic",

        "family":
            "Hepatic_Metabolic",

        "scope":
            "Patient_specific_HCC04"
    },

    "8": {
        "id":
            "MS8",

        "name":
            "HCC04-Hepatic-BileAcid-Metabolic",

        "display":
            "MS8 HCC04 Hepatic-Bile-Acid Metabolic",

        "family":
            "Hepatic_Metabolic",

        "scope":
            "Patient_specific_HCC04"
    }
}


source_cluster = (
    adata.obs[
        SOURCE_KEY
    ]
    .astype(str)
)


adata.obs[
    "final_malignant_state_id"
] = source_cluster.map(

    {
        key:
            value["id"]

        for key, value
        in FINAL_MAPPING.items()
    }
)


adata.obs[
    "final_malignant_state"
] = source_cluster.map(

    {
        key:
            value["display"]

        for key, value
        in FINAL_MAPPING.items()
    }
)


adata.obs[
    "final_state_family"
] = source_cluster.map(

    {
        key:
            value["family"]

        for key, value
        in FINAL_MAPPING.items()
    }
)


adata.obs[
    "final_state_scope"
] = source_cluster.map(

    {
        key:
            value["scope"]

        for key, value
        in FINAL_MAPPING.items()
    }
)


for col in [

    "final_malignant_state_id",
    "final_malignant_state",
    "final_state_family",
    "final_state_scope"

]:

    adata.obs[col] = (
        adata.obs[col]
        .astype("category")
    )


assert (
    adata.obs[
        "final_malignant_state"
    ]
    .isna()
    .sum()
    ==
    0
)


print(
    "Final malignant states:",
    adata.obs[
        "final_malignant_state_id"
    ].nunique()
)


display(

    pd.crosstab(

        adata.obs[
            SOURCE_KEY
        ],

        adata.obs[
            "final_malignant_state"
        ]

    )
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 207
# Save mapping and rationale
# ============================================================

mapping_rows = [

    {
        "Source_cluster":
            "0",

        "Final_state_id":
            "MS1",

        "Final_state_name":
            "Hypoxia-TGFb-Inflammatory",

        "State_scope":
            "Cross_patient_supported",

        "Decision":
            "Retain",

        "Rationale":
            (
                "Cross-patient support with reproducible IGFBP3, "
                "SLC2A1, PMEPA1, AXL and enrichment of inflammatory, "
                "TGFb, stress and hypoxia programs."
            )
    },

    {
        "Source_cluster":
            "1",

        "Final_state_id":
            "MS2",

        "Final_state_name":
            "S100P-SPINK1-Epithelial",

        "State_scope":
            "Cross_patient_supported",

        "Decision":
            "Retain",

        "Rationale":
            (
                "Four-patient support with reproducible S100P, "
                "SPINK1, AGR2 and TDO2 expression."
            )
    },

    {
        "Source_cluster":
            "2",

        "Final_state_id":
            "MS3",

        "Final_state_name":
            "Cycling-G2M",

        "State_scope":
            "Cross_patient_supported",

        "Decision":
            "Retain",

        "Rationale":
            (
                "Strong cross-patient G2M program with PLK1, CCNB1, "
                "TOP2A, CENPA, HMMR and related mitotic genes."
            )
    },

    {
        "Source_cluster":
            "3|4",

        "Final_state_id":
            "MS4",

        "Final_state_name":
            "HCC08-Replication-Fetal-Progenitor",

        "State_scope":
            "Patient_specific_HCC08",

        "Decision":
            "Merge",

        "Rationale":
            (
                "Both clusters are HCC08-specific, show moderate "
                "patient-adjusted transcriptomic similarity, and collapse "
                "to the same clusters in no-PC4 and patient-capped "
                "sensitivity analyses. They are retained as one "
                "patient-specific replication/progenitor continuum."
            )
    },

    {
        "Source_cluster":
            "5",

        "Final_state_id":
            "MS5",

        "Final_state_name":
            "Stress-Inflammatory-Progenitor",

        "State_scope":
            "Cross_patient_supported",

        "Decision":
            "Retain",

        "Rationale":
            (
                "Two-patient support with reproducible EREG, CXCL1, "
                "CXCL3, VCAM1 and MYC together with stress, progenitor "
                "and inflammatory programs."
            )
    },

    {
        "Source_cluster":
            "6",

        "Final_state_id":
            "MS6",

        "Final_state_name":
            "Hepatic-Differentiated",

        "State_scope":
            "Cross_patient_supported",

        "Decision":
            "Retain",

        "Rationale":
            (
                "Two-patient support with strong hepatic differentiation "
                "and metabolic genes including KNG1, HP, HRG, APOC3 "
                "and ALDOB."
            )
    },

    {
        "Source_cluster":
            "7",

        "Final_state_id":
            "MS7",

        "Final_state_name":
            "HCC04-Glycolytic-Hepatic",

        "State_scope":
            "Patient_specific_HCC04",

        "Decision":
            "Retain",

        "Rationale":
            (
                "HCC04-specific but structurally reproducible branch "
                "with glycolytic/hepatic features. Retained separately "
                "from MS8 despite transcriptomic similarity."
            )
    },

    {
        "Source_cluster":
            "8",

        "Final_state_id":
            "MS8",

        "Final_state_name":
            "HCC04-Hepatic-BileAcid-Metabolic",

        "State_scope":
            "Patient_specific_HCC04",

        "Decision":
            "Retain",

        "Rationale":
            (
                "HCC04-specific and structurally reproducible hepatic, "
                "fatty-acid and bile-acid metabolic branch. Closely "
                "related to MS7 but consistently separable."
            )
    }
]


final_mapping_table = pd.DataFrame(
    mapping_rows
)


MAPPING_PATH = (

    RESULTS_DIR /
    "Step23_Final_MalignantState_Mapping.csv"
)


final_mapping_table.to_csv(

    MAPPING_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    final_mapping_table
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 208
# Final state patient composition
# ============================================================

final_state_counts = pd.crosstab(

    adata.obs[
        "final_malignant_state"
    ],

    adata.obs[
        "patient"
    ]
)


final_state_pct = (

    final_state_counts

    .div(
        final_state_counts.sum(
            axis=1
        ),
        axis=0
    )

    *
    100
)


final_state_counts.to_csv(

    RESULTS_DIR /
    "Step23_Final_State_x_Patient_Counts.csv",

    encoding="utf-8-sig"
)


final_state_pct.to_csv(

    RESULTS_DIR /
    "Step23_Final_State_x_Patient_Percent.csv",

    encoding="utf-8-sig"
)


display(
    final_state_counts
)

display(
    final_state_pct
    .round(2)
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 209
# Recompute markers for FINAL consensus states
# ============================================================

FINAL_MARKER_KEY = (
    "rank_genes_groups_final_malignant_states"
)


sc.tl.rank_genes_groups(

    adata,

    groupby=
        "final_malignant_state_id",

    method=
        "wilcoxon",

    use_raw=
        False,

    pts=
        True,

    key_added=
        FINAL_MARKER_KEY
)


final_marker_tables = []


final_state_ids = [

    f"MS{i}"
    for i in range(
        1,
        9
    )
]


for state_id in final_state_ids:

    df = sc.get.rank_genes_groups_df(

        adata,

        group=
            state_id,

        key=
            FINAL_MARKER_KEY
    )


    df[
        "Final_state_id"
    ] = state_id


    final_marker_tables.append(
        df
    )


final_markers = pd.concat(

    final_marker_tables,

    ignore_index=True
)


final_markers[
    "Is_MT"
] = (

    final_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


final_markers[
    "Is_Ribosomal"
] = (

    final_markers[
        "names"
    ]
    .astype(str)
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


FINAL_ALL_MARKERS_PATH = (

    RESULTS_DIR /
    "Step23_Final_AllState_Markers.csv"
)


final_markers.to_csv(

    FINAL_ALL_MARKERS_PATH,

    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 210
# Final biological marker shortlist
# ============================================================

final_candidate_markers = (

    final_markers[
        (
            final_markers[
                "pvals_adj"
            ]
            <
            0.05
        )

        &

        (
            final_markers[
                "logfoldchanges"
            ]
            >
            0.25
        )

        &

        (
            ~final_markers[
                "Is_MT"
            ]
        )

        &

        (
            ~final_markers[
                "Is_Ribosomal"
            ]
        )
    ]
    .copy()
)


if (
    "pct_nz_group"
    in final_candidate_markers.columns
):

    final_candidate_markers = (

        final_candidate_markers[
            final_candidate_markers[
                "pct_nz_group"
            ]
            >=
            0.10
        ]
    )


if (
    "pct_nz_group"
    in final_candidate_markers.columns

    and

    "pct_nz_reference"
    in final_candidate_markers.columns
):

    final_candidate_markers[
        "Delta_pct"
    ] = (

        final_candidate_markers[
            "pct_nz_group"
        ]

        -

        final_candidate_markers[
            "pct_nz_reference"
        ]
    )


    final_candidate_markers = (

        final_candidate_markers[
            final_candidate_markers[
                "Delta_pct"
            ]
            >=
            0.05
        ]
    )


final_candidate_markers = (

    final_candidate_markers

    .sort_values(

        [
            "Final_state_id",
            "logfoldchanges"
        ],

        ascending=[
            True,
            False
        ]
    )
)


final_top20_markers = (

    final_candidate_markers

    .groupby(
        "Final_state_id",
        group_keys=False
    )

    .head(
        20
    )
)


FINAL_TOP20_PATH = (

    RESULTS_DIR /
    "Step23_Final_Top20_Markers_PerState.csv"
)


final_top20_markers.to_csv(

    FINAL_TOP20_PATH,

    index=False,
    encoding="utf-8-sig"
)


for state_id in final_state_ids:

    print(
        "\n",
        "=" * 90,
        sep=""
    )

    print(
        state_id
    )

    display(

        final_top20_markers[
            final_top20_markers[
                "Final_state_id"
            ]
            ==
            state_id
        ][
            [
                "names",
                "logfoldchanges",
                "pvals_adj"
            ]
        ]
        .head(10)
        .round(4)
    )

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 211
# Patient × FINAL state pseudobulk
# ============================================================

adata_raw.obs[
    "patient"
] = (

    adata.obs.loc[
        adata_raw.obs_names,
        "patient"
    ]
    .astype(str)
    .values
)


adata_raw.obs[
    "final_malignant_state_id"
] = (

    adata.obs.loc[
        adata_raw.obs_names,
        "final_malignant_state_id"
    ]
    .astype(str)
    .values
)


adata_raw.obs[
    "patient_final_state"
] = (

    adata_raw.obs[
        "patient"
    ]
    .astype(str)

    +

    "__"

    +

    adata_raw.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
)


pb_rows = []
pb_meta_rows = []


for group in (
    adata_raw.obs[
        "patient_final_state"
    ]
    .unique()
):

    mask = (

        adata_raw.obs[
            "patient_final_state"
        ]
        ==
        group
    )


    idx = np.where(
        mask.values
    )[0]


    X_group = (
        adata_raw.X[
            idx,
            :
        ]
    )


    if sparse.issparse(
        X_group
    ):

        summed = sparse.csr_matrix(
            X_group.sum(
                axis=0
            )
        )

    else:

        summed = sparse.csr_matrix(

            np.sum(
                X_group,
                axis=0,
                keepdims=True
            )
        )


    pb_rows.append(
        summed
    )


    patient = (
        adata_raw.obs.loc[
            mask,
            "patient"
        ]
        .iloc[0]
    )


    state_id = (
        adata_raw.obs.loc[
            mask,
            "final_malignant_state_id"
        ]
        .iloc[0]
    )


    pb_meta_rows.append({

        "patient_final_state":
            group,

        "patient":
            str(patient),

        "Final_state_id":
            str(state_id),

        "N_cells":
            int(
                mask.sum()
            )
    })


final_pb_counts = vstack(
    pb_rows
).tocsr()


final_pb_meta = pd.DataFrame(
    pb_meta_rows
)


adata_final_pb = ad.AnnData(

    X=
        final_pb_counts,

    obs=
        final_pb_meta.set_index(
            "patient_final_state"
        ),

    var=
        adata_raw.var.copy()
)


FINAL_PB_COUNTS_PATH = (

    DATA_DIR /
    "GSE149614_STEP23_FINAL_State_Pseudobulk_COUNTS.h5ad"
)


adata_final_pb.write_h5ad(

    FINAL_PB_COUNTS_PATH,

    compression="gzip"
)


adata_final_pb.obs.to_csv(

    RESULTS_DIR /
    "Step23_Final_State_Pseudobulk_Metadata.csv",

    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 212
# Final patient support
# ============================================================

MIN_CELLS_PER_PATIENT_STATE = 20


final_support = (

    adata_final_pb.obs

    .assign(

        Eligible=lambda x:

            x[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
    )

    .groupby(
        "Final_state_id"
    )

    .agg(

        N_pseudobulks=(
            "patient",
            "size"
        ),

        N_eligible_patients=(
            "Eligible",
            "sum"
        ),

        Total_cells=(
            "N_cells",
            "sum"
        )
    )

    .reset_index()
)


def final_support_label(
    n
):

    if n >= 3:

        return (
            "Cross_patient_3plus"
        )

    if n == 2:

        return (
            "Cross_patient_2"
        )

    if n == 1:

        return (
            "Patient_specific"
        )

    return (
        "Insufficient_support"
    )


final_support[
    "Support_class"
] = (

    final_support[
        "N_eligible_patients"
    ]
    .apply(
        final_support_label
    )
)


FINAL_SUPPORT_PATH = (

    RESULTS_DIR /
    "Step23_Final_State_PatientSupport.csv"
)


final_support.to_csv(

    FINAL_SUPPORT_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    final_support
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 213
# Final-state biological program summary
# ============================================================

program_score_cols = [

    col
    for col in adata.obs.columns
    if col.startswith(
        "Program__"
    )
]


program_df = (

    adata.obs[
        [
            "patient",
            "final_malignant_state_id"
        ]
        +
        program_score_cols
    ]
    .copy()
)


program_df[
    "patient"
] = (
    program_df[
        "patient"
    ]
    .astype(str)
)


program_df[
    "final_malignant_state_id"
] = (
    program_df[
        "final_malignant_state_id"
    ]
    .astype(str)
)


patient_state_program = (

    program_df

    .groupby(
        [
            "patient",
            "final_malignant_state_id"
        ]
    )[
        program_score_cols
    ]

    .median()

    .reset_index()
)


cell_n = (

    adata.obs

    .groupby(
        [
            adata.obs[
                "patient"
            ]
            .astype(str),

            adata.obs[
                "final_malignant_state_id"
            ]
            .astype(str)
        ]
    )

    .size()

    .rename(
        "N_cells"
    )

    .reset_index()
)


cell_n.columns = [

    "patient",
    "final_malignant_state_id",
    "N_cells"
]


patient_state_program = (

    patient_state_program

    .merge(

        cell_n,

        on=[
            "patient",
            "final_malignant_state_id"
        ],

        how="left"
    )
)


program_delta_rows = []


for _, row in (
    patient_state_program.iterrows()
):

    patient = str(
        row[
            "patient"
        ]
    )


    state_id = str(
        row[
            "final_malignant_state_id"
        ]
    )


    others = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                .astype(str)
                ==
                patient
            )

            &

            (
                patient_state_program[
                    "final_malignant_state_id"
                ]
                .astype(str)
                !=
                state_id
            )
        ]
    )


    out = {

        "patient":
            patient,

        "Final_state_id":
            state_id,

        "N_cells":
            int(
                row[
                    "N_cells"
                ]
            )
    }


    for col in program_score_cols:

        if len(others) == 0:

            value = np.nan

        else:

            value = (

                float(
                    row[col]
                )

                -

                float(
                    others[col]
                    .mean()
                )
            )


        out[
            col.replace(
                "Program__",
                ""
            )
        ] = value


    program_delta_rows.append(
        out
    )


final_program_delta_patient = pd.DataFrame(
    program_delta_rows
)


final_program_cols = [

    col
    for col in final_program_delta_patient.columns
    if col not in [

        "patient",
        "Final_state_id",
        "N_cells"
    ]
]


final_program_summary = (

    final_program_delta_patient[
        final_program_delta_patient[
            "N_cells"
        ]
        >=
        MIN_CELLS_PER_PATIENT_STATE
    ]

    .groupby(
        "Final_state_id"
    )[
        final_program_cols
    ]

    .mean()

    .reindex(
        final_state_ids
    )
)


FINAL_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step23_Final_State_BiologicalPrograms.csv"
)


final_program_summary.to_csv(

    FINAL_PROGRAM_PATH,

    encoding="utf-8-sig"
)


display(
    final_program_summary
    .round(3)
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 214
# Final malignant-state UMAP
# ============================================================

FINAL_COLORS = {

    "MS1 Hypoxia-TGFβ-Inflammatory":
        "#E64B35",

    "MS2 S100P-SPINK1 Epithelial":
        "#4DBBD5",

    "MS3 Cycling-G2M":
        "#00A087",

    "MS4 HCC08 Replication-Fetal/Progenitor":
        "#3C5488",

    "MS5 Stress-Inflammatory-Progenitor":
        "#F39B7F",

    "MS6 Hepatic-Differentiated":
        "#8491B4",

    "MS7 HCC04 Glycolytic-Hepatic":
        "#91D1C2",

    "MS8 HCC04 Hepatic-Bile-Acid Metabolic":
        "#7E6148"
}


state_order = [

    "MS1 Hypoxia-TGFβ-Inflammatory",
    "MS2 S100P-SPINK1 Epithelial",
    "MS3 Cycling-G2M",
    "MS4 HCC08 Replication-Fetal/Progenitor",
    "MS5 Stress-Inflammatory-Progenitor",
    "MS6 Hepatic-Differentiated",
    "MS7 HCC04 Glycolytic-Hepatic",
    "MS8 HCC04 Hepatic-Bile-Acid Metabolic"
]


adata.obs[
    "final_malignant_state"
] = pd.Categorical(

    adata.obs[
        "final_malignant_state"
    ],

    categories=
        state_order,

    ordered=True
)


ax = sc.pl.embedding(

    adata,

    basis=
        "umap_lineage_clean",

    color=
        "final_malignant_state",

    palette=
        FINAL_COLORS,

    size=
        8,

    alpha=
        0.82,

    frameon=
        False,

    legend_loc=
        "right margin",

    title=
        "Final Consensus Malignant States",

    show=
        False
)


fig = ax.figure

fig.set_size_inches(
    9.0,
    6.7
)

fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_01_ConsensusMalignantStates_UMAP.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_01_ConsensusMalignantStates_UMAP.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 215
# Final biological-program heatmap
# ============================================================

final_program_z = (

    final_program_summary

    -

    final_program_summary.mean(
        axis=0
    )

) / (

    final_program_summary.std(
        axis=0
    )

    +
    1e-8
)


fig, ax = plt.subplots(
    figsize=(11.5, 6.8)
)


im = ax.imshow(

    final_program_z.values,

    cmap=
        "RdBu_r",

    vmin=
        -2.5,

    vmax=
        2.5,

    aspect=
        "auto"
)


ax.set_xticks(
    np.arange(
        final_program_z.shape[1]
    )
)


ax.set_xticklabels(

    [
        col.replace(
            "_",
            " "
        )

        for col in final_program_z.columns
    ],

    rotation=
        55,

    ha=
        "right"
)


ax.set_yticks(
    np.arange(
        len(final_state_ids)
    )
)


ax.set_yticklabels(
    final_state_ids
)


ax.set_xlabel(
    "Biological Program"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "Patient-adjusted Biological Programs of Final Malignant States"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)


cbar.set_label(
    "Relative Program Enrichment (Z-score)"
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_02_BiologicalProgram_Heatmap.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_02_BiologicalProgram_Heatmap.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 216
# Final state × patient composition
# ============================================================

plot_pct = (

    final_state_pct
    .copy()
)


plot_pct.index = [

    str(x).split()[0]

    for x in plot_pct.index
]


fig, ax = plt.subplots(
    figsize=(7.8, 6.3)
)


im = ax.imshow(

    plot_pct.values,

    cmap=
        "Blues",

    vmin=
        0,

    vmax=
        100,

    aspect=
        "auto"
)


ax.set_xticks(
    np.arange(
        plot_pct.shape[1]
    )
)


ax.set_xticklabels(
    plot_pct.columns
)


ax.set_yticks(
    np.arange(
        plot_pct.shape[0]
    )
)


ax.set_yticklabels(
    plot_pct.index
)


ax.set_xlabel(
    "Patient"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "Patient Composition of Final Malignant States"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Cell Fraction (%)"
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_03_StatePatientComposition.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step23_Final_03_StatePatientComposition.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 217
# Final state summary table
# ============================================================

state_name_lookup = (

    adata.obs[
        [
            "final_malignant_state_id",
            "final_malignant_state",
            "final_state_family",
            "final_state_scope"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        "final_malignant_state_id"
    )
)


final_summary_table = (

    state_name_lookup

    .merge(

        final_support,

        left_on=
            "final_malignant_state_id",

        right_on=
            "Final_state_id",

        how=
            "left"
    )

    .drop(
        columns=[
            "Final_state_id"
        ]
    )
)


top_markers_text = (

    final_top20_markers

    .groupby(
        "Final_state_id"
    )[
        "names"
    ]

    .apply(
        lambda x:
            "|".join(
                x.head(10)
                .astype(str)
            )
    )

    .rename(
        "Top10_markers"
    )

    .reset_index()
)


final_summary_table = (

    final_summary_table

    .merge(

        top_markers_text,

        left_on=
            "final_malignant_state_id",

        right_on=
            "Final_state_id",

        how=
            "left"
    )

    .drop(
        columns=[
            "Final_state_id"
        ]
    )
)


FINAL_SUMMARY_TABLE_PATH = (

    RESULTS_DIR /
    "Step23_Final_MalignantState_Summary.csv"
)


final_summary_table.to_csv(

    FINAL_SUMMARY_TABLE_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    final_summary_table
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 218
# Save FINAL malignant-state object
# ============================================================

FINAL_H5AD = (

    DATA_DIR /
    "GSE149614_STEP23_FINAL_MALIGNANT_STATES.h5ad"
)


adata.uns[
    "Step23_final_state_lock"
] = {

    "Primary_partition":
        "lineage-clean Harmony(patient), Leiden r=0.4",

    "Final_consensus_states":
        8,

    "Merged_source_clusters":
        "3|4",

    "Retained_related_but_distinct_clusters":
        "7|8",

    "Bulk_risk_used_for_state_definition":
        False,

    "Status":
        "FROZEN"
}


adata.write_h5ad(

    FINAL_H5AD,

    compression="gzip"
)


print("=" * 110)
print("FINAL STEP23 OBJECT SAVED")
print("=" * 110)

print(
    FINAL_H5AD
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 219
# Final decision record
# ============================================================

final_decision = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step23_Final_Malignant_State_Lock",

    "Status":
        "FROZEN",

    "Lineage_clean_cells":
        int(
            adata.n_obs
        ),

    "Source_partition":
        "state_clean_leiden_r0_4",

    "Source_clusters":
        9,

    "Final_consensus_states":
        8,

    "Consensus_mapping": {

        "0":
            "MS1_Hypoxia_TGFb_Inflammatory",

        "1":
            "MS2_S100P_SPINK1_Epithelial",

        "2":
            "MS3_Cycling_G2M",

        "3|4":
            "MS4_HCC08_Replication_Fetal_Progenitor",

        "5":
            "MS5_Stress_Inflammatory_Progenitor",

        "6":
            "MS6_Hepatic_Differentiated",

        "7":
            "MS7_HCC04_Glycolytic_Hepatic",

        "8":
            "MS8_HCC04_Hepatic_BileAcid_Metabolic"
    },

    "Merge_decision": {

        "Clusters_3_and_4":
            (
                "Merged because both were HCC08-specific and "
                "collapsed to the same corresponding clusters under "
                "no-PC4 and patient-capped sensitivity analyses, "
                "with moderate patient-adjusted transcriptomic similarity."
            ),

        "Clusters_7_and_8":
            (
                "Retained separately despite high transcriptomic "
                "similarity because both remained structurally "
                "distinct and reproducible across sensitivity analyses."
            )
    },

    "Mixed_lineage_populations":
        {

            "Historical_candidate_5":
                "Excluded before final reconstruction",

            "Historical_candidate_6":
                "Excluded before final reconstruction"
        },

    "Important_interpretation":
        (
            "MS4, MS7 and MS8 are patient-specific programs and "
            "must not be presented as universally recurrent HCC states "
            "without external validation."
        ),

    "Bulk_risk_features_used_for_state_definition":
        False,

    "Next_step":
        (
            "Step24 bulk-to-single-cell projection using frozen "
            "malignant states. Original Final Top10, Original RiskScore, "
            "StableBio7 and Hub18 remain independent bulk anchors."
        ),

    "Final_object":
        str(
            FINAL_H5AD
        )
}


FINAL_DECISION_PATH = (

    LOGS_DIR /
    "Step23_FINAL_StateLock.json"
)


with open(

    FINAL_DECISION_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        final_decision,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    FINAL_DECISION_PATH
)

In [ ]:
# ============================================================
# STEP23 FINAL LOCK - CELL 220
# Final manifest and review
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest = pd.DataFrame(
    manifest_rows
)


FINAL_MANIFEST = (

    RESULTS_DIR /
    "Step23_FINAL_Output_Manifest.csv"
)


manifest.to_csv(

    FINAL_MANIFEST,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 125)
print("STEP23 FINAL MALIGNANT-STATE REVIEW")
print("=" * 125)


print("\n[1] Final state mapping")
print("-" * 125)

display(
    final_mapping_table
)


print("\n[2] Final patient support")
print("-" * 125)

display(
    final_support
)


print("\n[3] Final state × patient counts")
print("-" * 125)

display(
    final_state_counts
)


print("\n[4] Final biological programs")
print("-" * 125)

display(
    final_program_summary
    .round(3)
)


print("\n[5] Final top markers")
print("-" * 125)

for state_id in final_state_ids:

    print(
        f"\n{state_id}"
    )

    display(

        final_top20_markers[
            final_top20_markers[
                "Final_state_id"
            ]
            ==
            state_id
        ][
            [
                "names",
                "logfoldchanges",
                "pvals_adj"
            ]
        ]
        .head(10)
        .round(4)
    )


print("\n[6] Final files")
print("-" * 125)


for path in [

    FINAL_H5AD,

    MAPPING_PATH,

    FINAL_ALL_MARKERS_PATH,

    FINAL_TOP20_PATH,

    FINAL_PB_COUNTS_PATH,

    FINAL_SUPPORT_PATH,

    FINAL_PROGRAM_PATH,

    FINAL_SUMMARY_TABLE_PATH,

    FINAL_DECISION_PATH,

    FINAL_MANIFEST

]:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP23 STATUS = FROZEN"
)

print(
    "FINAL CONSENSUS MALIGNANT STATES = 8"
)

print(
    "BULK RISK INFORMATION USED FOR STATE DEFINITION = NO"
)

print(
    "NEXT = STEP24 BULK-TO-SINGLE-CELL RISK PROJECTION"
)

print("=" * 125)